# Tokyo Ward Recommender: Inside Airbnb pipeline (raw data → ward profiles)
**MSBA Capstone, Rakim.** Inside Airbnb Tokyo, snapshot 2025-09-29 (scraped 29–30 Sep 2025), 11 wards: Shinjuku, Shibuya, Taito, Chuo, Chiyoda, Minato, Sumida, Koto, Toshima, Setagaya, Itabashi.

This single notebook runs Stages 0–5: schema check → cleaning and unified corpus → sentiment bake-off → BERTopic → aspect-based sentiment → sentence-level validation → one JSON profile per ward. Each stage keeps its design notes as markdown. The saved outputs below come from a full run, so the results can be read without running anything.

## How to run
**Google Colab.** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rakimreid/tokyo-ward/blob/main/Tokyo_Ward_Recommender_Capstone.ipynb) Open the notebook in Colab and run all cells. The setup cell clones the repository (https://github.com/rakimreid/tokyo-ward: code, data and precomputed GPU outputs) into the Colab session. A CPU runtime is enough.

**Locally.** Clone the repository, create an environment with Python 3.12 or 3.13, run `pip install -r requirements.txt`, start Jupyter in the repository folder and run all cells.

**Data.** `raw/` holds the Inside Airbnb Tokyo files (CC BY 4.0), reduced to the columns this pipeline uses (reviewer names and other unused columns removed), compressed and split so each file stays under 25 MB. The original full files from Inside Airbnb also work: put `listings.csv(.gz)` and `reviews.csv(.gz)` in `raw/`.

**GPU steps.** Three steps use transformer models: Stage 2b (nlptown and cardiffnlp scoring), Stage 3b (mpnet embeddings) and Stage 4b (nlptown aspect scoring). Their outputs are included in `data/colab_out/`, so these steps are skipped and the notebook reproduces the reported results. To recompute them, delete those files and run on a Colab GPU (T4); results may then differ slightly because of GPU numerics.

**Time.** About 25–30 minutes on a 2-CPU machine, mostly language detection on 736k reviews. Outputs go to `data/interim/` and the ward JSONs to `data/ward_profiles/`.

## Labels used (embedded below, so nothing else needs uploading)
- 300 review labels (1–5), **LLM-assisted (Claude) with a human audit** (Rakim, 50 items; English-only kappa computed).
- 108 aspect-assignment precision judgments (LLM).
- 150 sentence-level sentiment and aspect labels (LLM) with a 30-item human audit (Rakim).
- A check of the 120 negative quotes selected in Stage 5 (LLM, read one by one); only the 36 judged correct are published.

**Random seed: 42 throughout.** Tested end to end with Colab's package versions (Python 3.13, pandas 2.2.3, numpy 2.4.6, scikit-learn 1.6.1, umap-learn 0.5.12) and with a newer local stack (pandas 3.0, numpy 2.5, scikit-learn 1.9, umap-learn 0.6). All downstream results are identical; only the tuning-grid rows for `n_neighbors` 10 and 30 differ slightly, because UMAP output depends on package versions. The saved outputs come from the Colab-version run.


## Setup: packages, paths, source modules, labels

In [1]:
# Install the packages Colab does not ship, at the versions this notebook was tested with.
# Pinned because they change results: the language detector decides which reviews enter the label set,
# and BERTopic / gensim define the topics and coherence scores.
import importlib.util, importlib.metadata as md, subprocess, sys
PINNED = {'lingua': 'lingua-language-detector==2.2.0', 'datasketch': 'datasketch==2.0.0',
          'vaderSentiment': 'vaderSentiment==3.3.2', 'bertopic': 'bertopic==0.17.4', 'gensim': 'gensim==4.4.0'}
OTHER = {'umap': 'umap-learn', 'hdbscan': 'hdbscan', 'statsmodels': 'statsmodels', 'openpyxl': 'openpyxl',
         'pyarrow': 'pyarrow', 'sklearn': 'scikit-learn', 'scipy': 'scipy', 'matplotlib': 'matplotlib'}
def _installed(pip_name):
    try: return md.version(pip_name.split('==')[0])
    except md.PackageNotFoundError: return None
todo = [p for p in PINNED.values() if _installed(p) != p.split('==')[1]]
todo += [p for m, p in OTHER.items() if importlib.util.find_spec(m) is None]
if todo:
    print('installing:', todo)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *todo], check=True)
    print('done. If pip changed numpy or pandas, restart the runtime once and run all cells again.')
else:
    print('all packages present at the tested versions')


all packages present at the tested versions


In [2]:
import os, sys, json, warnings, subprocess, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd
SEED = 42
REPO_URL = 'https://github.com/rakimreid/tokyo-ward.git'   # used on Colab to fetch code and data

# Quiet, harmless library warnings (kept out of the saved outputs)
warnings.filterwarnings('ignore', message='IProgress not found')
warnings.filterwarnings('ignore', category=FutureWarning, module='umap')
warnings.filterwarnings('ignore', message='n_jobs value')

IN_COLAB = 'google.colab' in sys.modules
has_data = lambda p: (p/'raw').is_dir() and any((p/'raw').glob('listings*'))
if 'CAPSTONE_ROOT' in os.environ:
    ROOT = Path(os.environ['CAPSTONE_ROOT'])
elif has_data(Path.cwd()):
    ROOT = Path.cwd()                                   # local clone, or Colab after %cd into the repo
elif IN_COLAB and REPO_URL:
    ROOT = Path('/content')/Path(REPO_URL).stem
    if not ROOT.exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, str(ROOT)], check=True)
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/tokyo_capstone')
else:
    ROOT = Path.cwd()
ROOT = ROOT.resolve()
for d in ['raw', 'src', 'data/interim', 'data/labels', 'data/colab_in', 'data/colab_out', 'data/ward_profiles']:
    (ROOT/d).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
RAW_DIR = ROOT/'raw'

def raw_paths(stem):
    """Raw file(s) for 'listings' or 'reviews': one full file, or numbered parts (e.g. reviews_part1.csv.xz ...)."""
    for ext in ['.csv.gz', '.csv', '.csv.xz']:
        if (RAW_DIR/f'{stem}{ext}').exists():
            return [RAW_DIR/f'{stem}{ext}']
    parts = sorted(RAW_DIR.glob(f'{stem}_part*.csv*'), key=lambda p: int(p.name.split('_part')[1].split('.')[0]))
    if parts:
        return parts
    raise FileNotFoundError(f'put {stem}.csv (or .csv.gz) or {stem}_part*.csv.xz in {RAW_DIR}')

def read_raw(stem, **kw):
    return pd.concat([pd.read_csv(p, **kw) for p in raw_paths(stem)], ignore_index=True)

EMB_STEM = ROOT/'data/colab_out/topic_embeddings_mpnet'
def emb_available():
    return Path(f'{EMB_STEM}.npy').exists() or Path(f'{EMB_STEM}_part1.npy').exists()
def load_embeddings():
    if Path(f'{EMB_STEM}.npy').exists():
        return np.load(f'{EMB_STEM}.npy')
    parts = sorted(EMB_STEM.parent.glob(EMB_STEM.name + '_part*.npy'), key=lambda p: int(p.stem.split('_part')[1]))
    return np.concatenate([np.load(p) for p in parts])
def save_embeddings(E, n_parts=3):                    # parts keep each file under GitHub's 25 MB web-upload limit
    for i, part in enumerate(np.array_split(E, n_parts), 1):
        np.save(f'{EMB_STEM}_part{i}.npy', part)

print('project folder:', ROOT.name, '| Colab:', IN_COLAB)
print('raw files:', [p.name for p in raw_paths('listings') + raw_paths('reviews')])
pre = ['sent_nlptown_labelset.parquet','sent_nlptown_sample.parquet','sent_cardiff_labelset.parquet','sent_cardiff_sample.parquet',
       'topic_embeddings_ids.parquet','absa_scores_nlptown.parquet','absa_extension_units_v2.parquet']
status = {f: (ROOT/'data/colab_out'/f).exists() for f in pre}
status['topic_embeddings_mpnet (parts)'] = emb_available()
print('precomputed GPU outputs present:', status)


project folder: tokyo-ward-recommender | Colab: False
raw files: ['listings.csv.gz', 'reviews_part1.csv.xz', 'reviews_part2.csv.xz', 'reviews_part3.csv.xz', 'reviews_part4.csv.xz', 'reviews_part5.csv.xz']
precomputed GPU outputs present: {'sent_nlptown_labelset.parquet': True, 'sent_nlptown_sample.parquet': True, 'sent_cardiff_labelset.parquet': True, 'sent_cardiff_sample.parquet': True, 'topic_embeddings_ids.parquet': True, 'absa_scores_nlptown.parquet': True, 'absa_extension_units_v2.parquet': True, 'topic_embeddings_mpnet (parts)': True}


### Source modules
Written to `<root>/src/` and imported by the stages below.

In [3]:
%%writefile src/textclean.py
"""Text cleaning and language helpers for the Tokyo ward recommender (Stage 1)."""
import html
import re
import unicodedata

_BR = re.compile(r'<br\s*/?>', re.I)
_TAG = re.compile(r'<[^>]+>')
_URL = re.compile(r'https?://\S+|www\.\S+', re.I)
_WS = re.compile(r'\s+')

# Languages covered by nlptown/bert-base-multilingual-uncased-sentiment (ISO 639-1)
NLPTOWN_LANGS = {'EN', 'NL', 'DE', 'FR', 'IT', 'ES'}
CJK_LANGS = {'JA', 'ZH', 'KO'}


def clean_text(s):
    """Normalize one raw text. Returns '' for missing input."""
    if s is None or (isinstance(s, float) and s != s):
        return ''
    s = html.unescape(str(s))
    s = _BR.sub(' ', s)
    s = _TAG.sub(' ', s)
    s = _URL.sub(' ', s)
    s = unicodedata.normalize('NFKC', s)   # full-width -> half-width; common in Japanese text
    return _WS.sub(' ', s).strip()


def has_content(s):
    """True if the text has at least one letter in any script (drops '.', '!!', '5/5')."""
    return any(ch.isalpha() for ch in s)


def approx_tokens(s, lang):
    """Rough token estimate used only to flag long texts.
    Not a real tokenizer: CJK ~1 token per char, others ~1.3 per whitespace word.
    Real 512-token splitting happens on Colab with the model's own tokenizer."""
    if lang in CJK_LANGS:
        return len(s)
    return int(len(s.split()) * 1.3)


def shingles(s, k=5):
    """Character k-gram shingles on lowercased, whitespace-collapsed text (script-agnostic)."""
    s = _WS.sub(' ', s.lower())
    return {s[i:i + k] for i in range(max(1, len(s) - k + 1))}


def jaccard(a, b):
    return len(a & b) / len(a | b) if a or b else 0.0


Writing src/textclean.py


In [4]:
%%writefile src/labeling.py
"""Stratified sampling helpers for the labeled set and the Colab subsample (Stage 2)."""
import numpy as np
import pandas as pd


def round_robin(df, n, strata, seed):
    """Take n rows spread as evenly as possible across strata.
    Shuffle with a fixed seed, rank rows within each stratum, then take rank 0 of every
    stratum, then rank 1, and so on. Small strata simply run out; others fill the gap."""
    if n >= len(df):
        return df.copy()
    d = df.sample(frac=1, random_state=seed)
    d = d.assign(_rk=d.groupby(strata).cumcount())
    d = d.sort_values('_rk', kind='stable').head(n)
    return d.drop(columns='_rk')


def vader_bucket(compound):
    """Standard VADER 3-way thresholds (+/-0.05)."""
    return np.where(compound <= -0.05, 'neg', np.where(compound >= 0.05, 'pos', 'neu'))


def length_bin(n_words, edges):
    return pd.cut(n_words, bins=[-np.inf, *edges, np.inf], labels=['short', 'medium', 'long'])


Writing src/labeling.py


In [5]:
%%writefile src/metrics.py
"""Evaluation metrics for the sentiment bake-off (Stage 2)."""
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score, mean_absolute_error
from statsmodels.stats.contingency_tables import mcnemar

# VADER compound -> 1..5 with FIXED equal-width cut points on [-1, 1], chosen before seeing labels (not tuned).
VADER5_EDGES = [-0.6, -0.2, 0.2, 0.6]


def vader_to_5(compound):
    return np.digitize(np.asarray(compound), VADER5_EDGES) + 1


def vader_to_3(compound):
    """0=neg, 1=neu, 2=pos using the standard VADER +/-0.05 thresholds."""
    c = np.asarray(compound)
    return np.where(c <= -0.05, 0, np.where(c >= 0.05, 2, 1))


def collapse3(stars):
    """1-2 -> 0 (neg), 3 -> 1 (neu), 4-5 -> 2 (pos)."""
    s = np.asarray(stars)
    return np.where(s <= 2, 0, np.where(s == 3, 1, 2))


def eval_5(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return {'n': len(y), 'accuracy': accuracy_score(y, p),
            'weighted_f1': f1_score(y, p, average='weighted', zero_division=0),
            'mae': mean_absolute_error(y, p),
            'qwk': cohen_kappa_score(y, p, weights='quadratic', labels=[1, 2, 3, 4, 5])}


def eval_3(y3, p3):
    y3, p3 = np.asarray(y3), np.asarray(p3)
    return {'n': len(y3), 'accuracy': accuracy_score(y3, p3),
            'weighted_f1': f1_score(y3, p3, average='weighted', zero_division=0),
            'macro_f1': f1_score(y3, p3, average='macro', zero_division=0),
            'qwk': cohen_kappa_score(y3, p3, weights='quadratic', labels=[0, 1, 2])}


def confusion_df(y, p, labels):
    return pd.DataFrame(confusion_matrix(y, p, labels=labels),
                        index=[f'true_{l}' for l in labels], columns=[f'pred_{l}' for l in labels])


def paired_bootstrap(y, pa, pb, metric, n_boot=2000, seed=42):
    """Bootstrap the difference metric(A) - metric(B) on the same resampled items.
    Returns observed diff, 95% percentile CI, and a two-sided p-value (share of resamples
    on the other side of zero, doubled, capped at 1)."""
    y, pa, pb = map(np.asarray, (y, pa, pb))
    rng = np.random.default_rng(seed)
    obs = metric(y, pa) - metric(y, pb)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        diffs[i] = metric(y[idx], pa[idx]) - metric(y[idx], pb[idx])
    p = min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))
    return {'diff': obs, 'ci_low': np.percentile(diffs, 2.5), 'ci_high': np.percentile(diffs, 97.5), 'p_value': p}


def mcnemar_test(y, pa, pb):
    """McNemar on correct/incorrect. Exact binomial when discordant pairs < 25."""
    y, pa, pb = map(np.asarray, (y, pa, pb))
    a, b = pa == y, pb == y
    table = [[np.sum(a & b), np.sum(a & ~b)], [np.sum(~a & b), np.sum(~a & ~b)]]
    exact = (table[0][1] + table[1][0]) < 25
    res = mcnemar(table, exact=exact, correction=not exact)
    return {'a_only_correct': int(table[0][1]), 'b_only_correct': int(table[1][0]),
            'exact': exact, 'statistic': float(res.statistic), 'p_value': float(res.pvalue)}


# ---- Sampling-weighted (design-based) estimates ----
def kish_ess(w):
    """Kish effective sample size: how many equally weighted items the weighted sample is 'worth'."""
    w = np.asarray(w, float)
    return w.sum() ** 2 / (w ** 2).sum()


def weighted_metrics(y, p, w, kind):
    """Population-weighted metrics. kind='5' (1-5 ordinal) or '3' (0/1/2)."""
    y, p, w = np.asarray(y), np.asarray(p), np.asarray(w, float)
    labels = [1, 2, 3, 4, 5] if kind == '5' else [0, 1, 2]
    out = {'accuracy': accuracy_score(y, p, sample_weight=w),
           'weighted_f1': f1_score(y, p, average='weighted', sample_weight=w, zero_division=0),
           'qwk': cohen_kappa_score(y, p, weights='quadratic', labels=labels, sample_weight=w)}
    if kind == '5':
        out['mae'] = mean_absolute_error(y, p, sample_weight=w)
    else:
        out['macro_f1'] = f1_score(y, p, average='macro', sample_weight=w, zero_division=0)
    return out


def stratified_boot_idx(strata, n_boot, seed):
    """Yield bootstrap index arrays that resample with replacement WITHIN each sampling stratum,
    so every replicate keeps the original design (e.g. 80 / 40 / 120 per VADER bucket)."""
    strata = np.asarray(strata)
    groups = [np.flatnonzero(strata == s) for s in np.unique(strata)]
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        yield np.concatenate([rng.choice(g, size=len(g), replace=True) for g in groups])


def weighted_boot_ci(y, p, w, strata, kind, n_boot=2000, seed=42):
    """Point estimates plus 95% percentile CIs from a stratified bootstrap."""
    y, p, w = map(np.asarray, (y, p, w))
    point = weighted_metrics(y, p, w, kind)
    reps = {k: [] for k in point}
    for idx in stratified_boot_idx(strata, n_boot, seed):
        m = weighted_metrics(y[idx], p[idx], w[idx], kind)
        for k in reps:
            reps[k].append(m[k])
    return {k: (point[k], np.nanpercentile(reps[k], 2.5), np.nanpercentile(reps[k], 97.5)) for k in point}


def weighted_paired_diff(y, pa, pb, w, strata, kind, metric='weighted_f1', n_boot=2000, seed=42):
    """Weighted metric(A) - metric(B), stratified paired bootstrap; same resamples for both models."""
    y, pa, pb, w = map(np.asarray, (y, pa, pb, w))
    obs = weighted_metrics(y, pa, w, kind)[metric] - weighted_metrics(y, pb, w, kind)[metric]
    d = np.array([weighted_metrics(y[i], pa[i], w[i], kind)[metric] - weighted_metrics(y[i], pb[i], w[i], kind)[metric]
                  for i in stratified_boot_idx(strata, n_boot, seed)])
    p = min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))
    return {'diff': obs, 'ci_low': np.percentile(d, 2.5), 'ci_high': np.percentile(d, 97.5), 'p_value': p}


Writing src/metrics.py


In [6]:
%%writefile src/colab_sentiment_core.py
# ---- Core scoring logic (embedded verbatim in the Colab notebook; logic-tested in the sandbox with stubs) ----
import re
import numpy as np
import pandas as pd

_SENT = re.compile(r'(?<=[.!?])\s+|(?<=[。！？])')


def split_sentences(text):
    parts = [s.strip() for s in _SENT.split(text) if s and s.strip()]
    return parts or [text]


def chunk_text(text, n_tokens, count_tokens, max_tokens=510):
    """Return [text] if it fits; otherwise greedy sentence packing into chunks of <= max_tokens.
    A single sentence longer than max_tokens becomes its own chunk and is truncated by the tokenizer."""
    if n_tokens <= max_tokens:
        return [text]
    chunks, cur = [], ''
    for s in split_sentences(text):
        cand = f'{cur} {s}'.strip() if cur else s
        if count_tokens([cand])[0] <= max_tokens:
            cur = cand
        else:
            if cur:
                chunks.append(cur)
            cur = s
    if cur:
        chunks.append(cur)
    return chunks


def score_corpus(df, predict_proba, count_tokens, labels, batch_size=64, max_tokens=510):
    """df needs text_id, text. predict_proba(list[str]) -> (n, k) array. count_tokens(list[str]) -> list[int].
    Long texts are sentence-chunked; chunk probabilities are averaged per text, weighted by chunk token count.
    Chunks are sorted by length before batching (less padding = faster on GPU)."""
    texts = df.text.tolist()
    ids = df.text_id.tolist()
    lens = []
    for i in range(0, len(texts), 1000):
        lens.extend(count_tokens(texts[i:i + 1000]))
    rows = []
    for tid, t, n in zip(ids, texts, lens):
        for ch in chunk_text(t, n, count_tokens, max_tokens):
            rows.append((tid, ch, n if n <= max_tokens else count_tokens([ch])[0]))
    ch = pd.DataFrame(rows, columns=['text_id', 'chunk', 'w'])
    order = np.argsort(ch.w.values, kind='stable')
    probs = np.zeros((len(ch), len(labels)))
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        probs[idx] = predict_proba(ch.chunk.values[idx].tolist())
    w = np.maximum(ch.w.values, 1)[:, None]
    agg = pd.DataFrame(probs * w, columns=labels)
    agg['w'] = w[:, 0]
    agg['text_id'] = ch.text_id.values
    g = agg.groupby('text_id', sort=False)
    out = g[labels].sum().div(g.w.sum(), axis=0)
    out['n_chunks'] = g.size()
    out['n_tokens'] = pd.Series(lens, index=ids).groupby(level=0).first()
    return out.reindex(ids).reset_index()


Writing src/colab_sentiment_core.py


In [7]:
%%writefile src/topics.py
"""Topic-modeling helpers for Stage 3 (BERTopic with precomputed embeddings)."""
import re
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import HDBSCAN

WARDS = ['shinjuku', 'shibuya', 'taito', 'chuo', 'chiyoda', 'minato', 'sumida', 'koto', 'toshima', 'setagaya', 'itabashi']
# Removed from topic REPRESENTATIONS only (embeddings still see the full sentence).
DOMAIN_STOP = set(WARDS) | {
    'ku', 'tokyo', 'japan',
    # generic accommodation words
    'stay', 'stayed', 'staying', 'place', 'apartment', 'apt', 'room', 'rooms', 'house', 'home', 'accommodation',
    'airbnb', 'property', 'unit', 'flat', 'hotel', 'location', 'located', 'area', 'trip', 'experience', 'time',
    # generic praise / filler (topics should say WHAT, sentiment is Stage 4's job)
    'great', 'good', 'nice', 'lovely', 'perfect', 'amazing', 'excellent', 'wonderful', 'awesome', 'best', 'fantastic',
    'recommend', 'recommended', 'highly', 'definitely', 'really', 'super', 'overall', 'thank', 'thanks', 'also',
    'just', 'like', 'would', 'will', 'us', 'got', 'get', 'everything', 'need', 'needed', 'lot', 'lots'}
STOPWORDS = sorted(ENGLISH_STOP_WORDS | DOMAIN_STOP)

_UNIT_SPLIT = re.compile(r'(?<=[.!?])\s+|\s*[・■●◆★※•]\s*|\s+-\s+|\s*\|\s*')


def split_units(text, min_words=5):
    """Sentence-like units. Also splits host bullet lists (・ ■ - |). Drops units under min_words."""
    return [u.strip(' -') for u in _UNIT_SPLIT.split(text) if u and len(u.split()) >= min_words]


def make_vectorizer(min_df=5):
    return CountVectorizer(stop_words=STOPWORDS, ngram_range=(1, 1), min_df=min_df,
                           token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b')


def tfidf_svd_embed(docs, dim=100, seed=42):
    """Sandbox fallback embedding: TF-IDF (1-2 grams) -> truncated SVD -> L2 normalize."""
    X = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=3, sublinear_tf=True).fit_transform(docs)
    Z = TruncatedSVD(n_components=dim, random_state=seed).fit_transform(X)
    return normalize(Z).astype(np.float32)


def topic_words(model, k=10):
    return {t: [w for w, _ in model.get_topic(t)][:k] for t in model.get_topics() if t != -1}


def topic_diversity(words_by_topic, k=10):
    tops = [w[:k] for w in words_by_topic.values()]
    return len({w for ws in tops for w in ws}) / max(1, k * len(tops))


def coherence(words_by_topic, tokens, dictionary, measure):
    from gensim.models.coherencemodel import CoherenceModel
    tw = [[w for w in ws if w in dictionary.token2id] for ws in words_by_topic.values()]
    tw = [ws for ws in tw if len(ws) >= 2]
    if not tw:
        return np.nan
    return CoherenceModel(topics=tw, texts=tokens, dictionary=dictionary, coherence=measure, processes=1).get_coherence()


def fit_bertopic(docs, reduced, min_cluster_size, min_samples, seed=42):
    """BERTopic on already-UMAP-reduced embeddings (UMAP is cached per n_neighbors outside)."""
    from bertopic import BERTopic
    from bertopic.dimensionality import BaseDimensionalityReduction
    from bertopic.vectorizers import ClassTfidfTransformer
    tm = BERTopic(embedding_model=None, umap_model=BaseDimensionalityReduction(),
                  hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples),
                  vectorizer_model=make_vectorizer(), ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
                  calculate_probabilities=False, verbose=False)
    topics, _ = tm.fit_transform(docs, embeddings=reduced)
    return tm, np.asarray(topics)


def run_grid(docs, emb, n_neighbors_list, mcs_list, min_samples, tokens, dictionary, seed=42, log=print):
    import umap
    rows, cache = [], {}
    for nn in n_neighbors_list:
        t0 = time.time()
        red = umap.UMAP(n_neighbors=nn, n_components=5, min_dist=0.0, metric='cosine', random_state=seed).fit_transform(emb)
        cache[nn] = red
        log(f'UMAP n_neighbors={nn}: {time.time() - t0:.0f}s')
        for mcs in mcs_list:
            t0 = time.time()
            try:
                tm, topics = fit_bertopic(docs, red, mcs, min_samples, seed)
            except ValueError as e:            # e.g. too few clusters for the c-TF-IDF vectorizer
                rows.append({'n_neighbors': nn, 'min_topic_size': mcs, 'n_topics': 0, 'outlier_rate': np.nan,
                             'largest_topic_share': np.nan, 'c_v': np.nan, 'c_npmi': np.nan, 'diversity': np.nan,
                             'seconds': round(time.time() - t0, 1), 'error': str(e)[:80]})
                log(rows[-1]); continue
            words = topic_words(tm)
            counts = pd.Series(topics[topics != -1]).value_counts()
            rows.append({'n_neighbors': nn, 'min_topic_size': mcs, 'n_topics': len(words),
                         'outlier_rate': float((topics == -1).mean()),
                         'largest_topic_share': float(counts.iloc[0] / len(topics)) if len(counts) else np.nan,
                         'c_v': coherence(words, tokens, dictionary, 'c_v'),
                         'c_npmi': coherence(words, tokens, dictionary, 'c_npmi'),
                         'diversity': topic_diversity(words), 'seconds': round(time.time() - t0, 1)})
            log({k: (round(float(v), 3) if isinstance(v, float) else v) for k, v in rows[-1].items()})
    return pd.DataFrame(rows), cache


def select_config(grid, max_largest=0.40, max_outlier=0.40, k_range=(8, 80)):
    """Pre-specified rule: among valid configs, highest c_npmi; tie-break by diversity."""
    g = grid.copy()
    g['valid'] = (g.largest_topic_share < max_largest) & (g.outlier_rate <= max_outlier) & g.n_topics.between(*k_range)
    v = g[g.valid].sort_values(['c_npmi', 'diversity'], ascending=False)
    return g, (v.iloc[0] if len(v) else None)


Writing src/topics.py


In [8]:
%%writefile src/absa.py
"""Aspect assignment helpers for Stage 4 (hybrid: keywords + embedding centroids + topic map)."""
import re
import numpy as np
import pandas as pd

ASPECTS = ['food_dining', 'nightlife', 'shopping', 'traditional_cultural', 'family_friendliness',
           'nature_parks', 'walkability_transit', 'safety', 'accommodation_value', 'crowd_level']

# English seed keywords (regex, matched case-insensitively on word boundaries).
KEYWORDS = {
    'food_dining': r"restaurants?|food|foodie|ramen|sushi|izakayas?|caf[eé]s?|coffee shops?|eater(?:y|ies)|dining|dinner|lunch|breakfast"
                   r"|baker(?:y|ies)|yakitori|udon|soba|tempura|gyoza|cuisine|street food|places to eat|eat(?:ing)? out",
    'nightlife': r"nightlife|night life|bars?|pubs?|clubs?|clubbing|karaoke|golden gai|live music|night out",
    'shopping': r"shopping|shops|boutiques?|malls?|department stores?|souvenirs?|don quijote|donki|electronics|fashion"
                r"|shopping streets?|shotengai|shopping arcades?|markets?",
    'traditional_cultural': r"temples?|shrines?|senso-?ji|traditional|cultur(?:e|al)|historic(?:al)?|history|museums?|kimono"
                            r"|tatami|festivals?|sumo|edo|old town|shitamachi|kaminarimon|nakamise|tea ceremony|onsen|sento",
    'family_friendliness': r"kids?|child(?:ren)?|famil(?:y|ies)(?!\s?-?mart)|bab(?:y|ies)|toddlers?|strollers?|pushchair|playgrounds?|cribs?|high chair|family-friendly",
    'nature_parks': r"parks?|gardens?|riverside|river|nature|greenery|cherry blossoms?|sakura|gyoen|canal|trees",
    'walkability_transit': r"stations?|trains?|subway|metro|jr|yamanote|toei|bus(?:es)?|walkable|walking distance|transport(?:ation)?|transit"
                           r"|commut(?:e|ing)|airport|narita|haneda|access(?:ible)?",
    'safety': r"safe(?:ty)?|secure|security|unsafe|dangerous|sketchy|crime|police|felt safe",
    'accommodation_value': r"price|value|money|worth|affordable|expensive|cheap|budget|deal|reasonabl[ey]|overpriced|cost"
                           r"|clean(?:liness)?|dirty|spotless|amenities|beds?|bathroom|shower|kitchen|laundry|washing machine|spacious|cramped|towels",
    'crowd_level': r"crowds?|crowded|busy|bustl(?:e|ing)|hustle|packed|touristy|tourists|quiet(?:er)?|peaceful|calm|noisy|noise|lively|rush hour|deserted",
}
KW_RE = {a: re.compile(rf"\b(?:{p})\b", re.I) for a, p in KEYWORDS.items()}
# Word-sense exclusions, stripped from the text before that aspect's keywords are matched (v3 prune).
EXCLUDE = {
    'nature_parks': re.compile(r"sky\s?tree|amusement parks?|park (?:the|our|my|a|your) cars?|car parks?", re.I),
    'nightlife': re.compile(r"mini-?bars?|bar stools?|chocolate bars?|snack bars?", re.I),
    'safety': re.compile(r"microwave[- ]safe|safe(?:ty)?[- ](?:deposit|box)", re.I),
}
_SKYTREE = EXCLUDE['nature_parks']   # backward-compatible name

# Stage 3 topic -> aspect map (topic ids from the final mpnet model; checked against top words at load time).
TOPIC_MAP = {
    0: ['food_dining', 'shopping'], 2: ['walkability_transit'], 3: ['walkability_transit'], 4: ['accommodation_value'],
    6: ['family_friendliness', 'accommodation_value'], 8: ['food_dining'], 9: ['safety', 'crowd_level'],
    11: ['food_dining', 'shopping'], 12: ['food_dining', 'shopping', 'walkability_transit'], 13: ['accommodation_value'],
    14: ['accommodation_value'], 16: ['accommodation_value'], 18: ['accommodation_value'], 19: ['traditional_cultural'],
    20: ['accommodation_value'], 21: ['nightlife'], 22: ['traditional_cultural'], 23: ['shopping', 'crowd_level'],
    24: ['walkability_transit'], 26: ['nature_parks'], 28: ['walkability_transit'], 29: ['family_friendliness', 'nature_parks'],
    31: ['walkability_transit'], 32: ['accommodation_value'], 33: ['accommodation_value'], 34: ['shopping'],
    35: ['accommodation_value'], 36: ['traditional_cultural'], 38: ['accommodation_value'], 39: ['food_dining', 'shopping'],
    40: ['crowd_level', 'accommodation_value'],
}
TOPIC_ANCHOR = {0: 'convenience', 8: 'ramen', 21: 'kabukicho', 26: 'garden', 40: 'noise', 13: 'stairs'}

_CONTRAST = re.compile(r",?\s+\b(?:but|however|although|though|except|unfortunately|yet|whereas)\b\s*", re.I)


def keyword_hits(text):
    return {a: [m.group(0) for m in r.finditer(EXCLUDE[a].sub(' ', text) if a in EXCLUDE else text)] for a, r in KW_RE.items()}


def split_clauses(text):
    """Split on contrastive connectives. Returns [] when there is no contrast (sentence is one clause)."""
    parts = [p.strip(' ,;') for p in _CONTRAST.split(text) if p and len(p.split()) >= 2]
    return parts if len(parts) >= 2 else []


def to_score(exp_star):
    """nlptown expected star (1..5) -> sentiment score in [-1, 1]."""
    return (np.asarray(exp_star) - 3.0) / 2.0


# ---- assignment ----
TAU_TOPIC = 0.40      # topic-mapped units need this centred-cosine similarity to the aspect centroid
TAU_EMB = 0.55        # embedding-only path (no keyword, no mapped topic)
EMB_ONLY_OFF = {'safety', 'family_friendliness'}   # v1 inspection (kept for the record)
# v2, after the 108-item precision audit: a non-keyword path is kept only where audit precision was >= 0.75
TOPIC_OK = {'accommodation_value', 'walkability_transit', 'nature_parks', 'traditional_cultural'}
EMB_OK = {'accommodation_value', 'walkability_transit', 'crowd_level'}


def centroids(emb, kw_hits):
    """Centre embeddings, then average units whose keywords hit exactly one aspect ('pure' examples)."""
    Ec = emb - emb.mean(axis=0)
    Ec /= np.linalg.norm(Ec, axis=1, keepdims=True)
    pure = kw_hits.sum(axis=1) == 1
    C = np.stack([Ec[(kw_hits[a] & pure).values].mean(axis=0) for a in ASPECTS])
    C /= np.linalg.norm(C, axis=1, keepdims=True)
    return Ec, C


# Fallback when topic ids differ from the run TOPIC_MAP was written for: map by each topic's top-5 words.
# Only aspects in TOPIC_OK use the topic path, so only those need rules.
TOPIC_RULES = {
    'walkability_transit': r"station|stations|subway|metro|line|lines|train|trains|airport|transport|access|transit|jr",
    'accommodation_value': r"bathroom|shower|toilet|bed|beds|clean|tidy|kitchen|laundry|washing|price|value|budget|stairs|elevator|renovated|newly|spacious",
    'nature_parks': r"garden|gardens|park|parks|playground|nature|green",
    'traditional_cultural': r"temple|temples|shrine|sensoji|museum|asakusa|kaminarimon",
}


def resolve_topic_map(info, log=print):
    """Use the hand-written TOPIC_MAP if every anchor word still sits in its topic; otherwise rule-based."""
    ok = all(t in info.index and w in info.loc[t, 'top_words'] for t, w in TOPIC_ANCHOR.items())
    if ok:
        log('topic ids match the Stage 3 run the map was written for -> using hand-written TOPIC_MAP')
        return dict(TOPIC_MAP)
    log('WARNING: topic ids changed -> using rule-based topic map (top-5 words); results will differ slightly from the reported run')
    m = {}
    for t, words in info.top_words.items():
        top5 = ' '.join(words.split(', ')[:5])
        hits = [a for a, rx in TOPIC_RULES.items() if re.search(rf"\b(?:{rx})\b", top5)]
        if hits:
            m[int(t)] = hits
    return m


def assign(units, kw_lists, sims, topic_col='topic', topic_map=None):
    """One row per (unit, aspect). path: keyword > topic+embedding > embedding."""
    topic_map = TOPIC_MAP if topic_map is None else topic_map
    rows = []
    for i, (uid, topic) in enumerate(zip(units.unit_id, units[topic_col])):
        mapped = set(topic_map.get(int(topic), []))
        for j, a in enumerate(ASPECTS):
            kw, sim = kw_lists[i][a], float(sims[i, j])
            if kw:
                path = 'keyword'
            elif a in mapped and sim >= TAU_TOPIC and a in TOPIC_OK:
                path = 'topic+embedding'
            elif sim >= TAU_EMB and a in EMB_OK:
                path = 'embedding'
            else:
                continue
            rows.append((uid, a, path, sim, ', '.join(dict.fromkeys(k.lower() for k in kw))))
    return pd.DataFrame(rows, columns=['unit_id', 'aspect', 'path', 'sim', 'keywords'])


def pick_clause(clauses, aspect):
    """Index of the clause that carries the aspect keyword, or None (then the whole sentence is used)."""
    for k, c in enumerate(clauses):
        if keyword_hits(c)[aspect]:
            return k
    return None


Writing src/absa.py


### Embedded labels
- `llm_labels.csv`: 300 reviews, 1–5 star, LLM pre-labels (Stage 2).
- `audit_completed_stage2.csv`: Rakim's blind audit of 50 of them; the 12 non-English rows are blank or marked unreadable and excluded.
- `aspect_audit_*`: the frozen 108-item aspect-assignment sample and its LLM precision judgments (Stage 4).
- `sentence_*`: the 150-span sentence-level validation set, its LLM labels, and Rakim's 30-item blind audit (Stage 4c).

In [9]:
import pandas as pd
LABEL_DATA = {'llm_labels.csv': {'columns': ['text_id', 'llm_label'], 'data': [['rev_1429473266791308502', 5], ['rev_1387453310181444179', 5], ['rev_1448262032681435961', 5], ['rev_965571036637552106', 3], ['rev_1495394358795019939', 4], ['rev_1333011500123144850', 5], ['rev_830035203123716245', 4], ['rev_1472145225030786581', 3], ['rev_832185334673677110', 1], ['rev_901034772417040874', 5], ['rev_1509857359102373418', 5], ['rev_1359867070125223999', 5], ['rev_1485221376181845400', 4], ['rev_1272852640922476587', 2], ['rev_1007666840140070063', 5], ['rev_1319222046673152692', 5], ['rev_1327258575098907891', 5], ['rev_1226534741086546914', 4], ['rev_1312012572205045504', 5], ['rev_1064910430687620654', 5], ['rev_1059861080218801271', 1], ['rev_576243351', 4], ['rev_1280119957707532994', 5], ['rev_1359181506490833113', 4], ['rev_945300575718847318', 3], ['rev_592448004', 4], ['rev_1282324013874070291', 4], ['rev_1140257342620476265', 3], ['rev_1433805120509044357', 3], ['rev_1224348838941725836', 5], ['rev_1293144460499478447', 4], ['rev_285882190', 5], ['rev_1507027317023765584', 2], ['rev_509673951', 4], ['rev_914880844889218235', 5], ['rev_1134472875673267074', 3], ['rev_1356967687870636730', 2], ['rev_1324286134590078416', 2], ['rev_250672134', 2], ['rev_784439119232170475', 3], ['rev_1069165378098819936', 4], ['rev_911171087899491017', 1], ['rev_877209054967380759', 1], ['rev_361901298', 2], ['rev_832975618292857350', 2], ['rev_1483767115036113552', 3], ['rev_478402048', 5], ['rev_1512804435440604544', 5], ['rev_764077652264016649', 3], ['rev_1504123766337180688', 5], ['rev_1479433960787461072', 5], ['rev_769876279548066366', 4], ['rev_1072088668097572157', 5], ['rev_919241125832137082', 3], ['rev_1263430584524376355', 4], ['rev_959080350951827216', 4], ['rev_248193674', 4], ['rev_581147094', 3], ['rev_245363268', 5], ['rev_880798782949216224', 3], ['rev_1019200171854663731', 5], ['rev_1038051504343802824', 1], ['rev_1280100457496192448', 1], ['rev_940967091058750016', 2], ['rev_430188640', 4], ['rev_945293379104742479', 3], ['rev_213767749', 3], ['rev_874927900273512402', 4], ['rev_763365535233263226', 2], ['rev_599122962', 5], ['rev_959832506047548680', 4], ['rev_1014846913744799122', 5], ['rev_1437394085483802026', 5], ['rev_526463245', 5], ['rev_885152248107705180', 5], ['rev_1248212977620322619', 5], ['rev_1206241019432712785', 4], ['rev_1164217866433159153', 3], ['rev_1419351767379669266', 1], ['rev_281929510', 2], ['rev_997505387143633713', 4], ['rev_1314249807099695533', 5], ['rev_900295551399227462', 1], ['rev_726362659275655635', 5], ['rev_1098231763048309947', 2], ['rev_859799359932449884', 3], ['rev_1320664221854670100', 3], ['rev_1114135809859986150', 2], ['rev_887265940999661288', 1], ['rev_1433060226998600121', 3], ['rev_1046731780300682874', 5], ['rev_1193914640694510035', 1], ['rev_1272127621916199876', 4], ['rev_1127905408222405914', 4], ['rev_812664609041163538', 2], ['rev_1511347227184016075', 2], ['rev_430567650', 5], ['rev_116844695', 5], ['rev_906850239656437945', 4], ['rev_1127243250038109367', 2], ['rev_1388121648368464213', 4], ['rev_354156816', 3], ['rev_1285892356206409787', 3], ['rev_921399347064812900', 5], ['rev_964851016291133705', 3], ['rev_1265663906516028177', 4], ['rev_1199016396710733274', 3], ['rev_1488085595880084222', 1], ['rev_577032302', 5], ['rev_825737954238206680', 3], ['rev_1219254151831440828', 5], ['rev_1254770813724062692', 5], ['rev_1077884207792128766', 4], ['rev_532511577', 5], ['rev_1505509233404209610', 4], ['rev_860473025907662025', 3], ['rev_1159045976270081655', 4], ['rev_1499007335743103848', 5], ['rev_637256915012393259', 5], ['rev_843101687495201724', 5], ['rev_903999093305198299', 3], ['rev_1253277870505508738', 5], ['rev_1429471521104256003', 4], ['rev_820646957540413100', 4], ['rev_941684291050112274', 5], ['rev_851798277821841545', 5], ['rev_1049689062482662756', 2], ['rev_1501204309830184134', 5], ['rev_663488509', 5], ['rev_1294655530950541633', 5], ['rev_1342475490330496237', 5], ['rev_972065033959016043', 3], ['rev_1219232332128555758', 5], ['rev_1357713901849812259', 2], ['rev_547373904', 5], ['rev_1453380780687012969', 5], ['rev_1092392199058813593', 5], ['rev_1343132068869163823', 5], ['rev_575672336', 5], ['rev_856152977381180084', 2], ['rev_1043836497271619091', 3], ['rev_1399054748770164207', 5], ['rev_1389519579723373928', 5], ['rev_1175793829844508814', 5], ['rev_1014108124202055097', 3], ['rev_348948357', 5], ['rev_1093864617594075089', 5], ['rev_1212686246099990362', 2], ['rev_1138821989485575275', 4], ['rev_1061922233334708922', 4], ['rev_1401915822932674177', 4], ['rev_1422185665518155075', 3], ['rev_1163472549243296031', 5], ['rev_900321599230999530', 4], ['rev_1201890277081798794', 5], ['rev_301866716', 4], ['rev_1270773233061367435', 5], ['rev_589415210', 5], ['rev_1104789262540891215', 4], ['rev_1166299954357851404', 4], ['rev_197257320', 5], ['rev_268387734', 5], ['rev_1359891428761851735', 3], ['rev_640330351', 1], ['rev_821343007007481845', 3], ['rev_1274314033452316235', 4], ['rev_922065412044532336', 5], ['rev_1254016433208657072', 5], ['rev_1124288568108357320', 5], ['rev_362239765', 4], ['rev_867691850872670446', 3], ['rev_1309107752795352400', 5], ['rev_550590309', 4], ['rev_1372942839640491983', 4], ['rev_1098267715077665526', 4], ['rev_897419125072974272', 5], ['rev_1315627821342632660', 4], ['rev_714477929', 1], ['rev_1405479705896214666', 5], ['rev_203040330', 5], ['rev_203340971', 5], ['rev_1330135671029691195', 2], ['rev_1279362658904188975', 5], ['rev_1125772803425665406', 4], ['rev_582415188', 4], ['rev_175874488', 4], ['rev_1009780389939701932', 5], ['rev_1003956293492486850', 4], ['rev_185123629', 4], ['rev_1333796864561921326', 4], ['rev_432763377', 4], ['rev_1421446423117439361', 5], ['rev_948877993539361656', 5], ['rev_1427285069477928458', 5], ['rev_1512733773884508852', 4], ['rev_1422146746720055502', 2], ['rev_1143931122580120750', 3], ['rev_300336567', 4], ['rev_1165612202389007527', 3], ['rev_977152308606267635', 5], ['rev_1509845578064442536', 2], ['rev_1135199974005934467', 5], ['rev_966365763671462574', 1], ['rev_1106187287317832260', 4], ['rev_1514280883356957802', 3], ['rev_792329821863027178', 5], ['rev_1146822505125227711', 5], ['rev_1153250807498734058', 5], ['rev_781531502795568263', 1], ['rev_1261325002095305166', 5], ['rev_1045331855432107501', 5], ['rev_1228680853667393969', 5], ['rev_1273647043582207053', 2], ['rev_1127248741603675409', 2], ['rev_1501151330442154074', 5], ['rev_774203877722756225', 3], ['rev_851120896747016212', 2], ['rev_948239315261260687', 5], ['rev_559874591', 5], ['rev_1297479987146544545', 2], ['rev_505397060203035305', 5], ['rev_1191005173852151567', 4], ['rev_412554312', 5], ['rev_159375464', 4], ['rev_822793222069495516', 2], ['rev_1251122836056957322', 4], ['rev_858311244721139270', 4], ['rev_1186687706303236491', 4], ['rev_1389582797677468048', 3], ['rev_1335237246332354679', 3], ['rev_1254036152890945244', 2], ['rev_893805304627631322', 4], ['rev_1496140993631139249', 5], ['rev_1243904690152696527', 3], ['rev_1264207943721504208', 5], ['rev_1043141172638989899', 2], ['rev_1271478172528901764', 4], ['rev_1319972164182702289', 4], ['rev_1245400643482280391', 4], ['rev_489470433', 4], ['rev_1364148766392436782', 5], ['rev_1427931994806465026', 5], ['rev_971431479060507733', 5], ['rev_552991262', 3], ['rev_1264869382874756182', 5], ['rev_140117240', 5], ['rev_1218551070837421606', 4], ['rev_793055260723507245', 5], ['rev_1159797860044223091', 4], ['rev_1332328260010907493', 5], ['rev_620815212', 3], ['rev_1517803183769985137', 3], ['rev_1066321799411578207', 5], ['rev_1401117185115331352', 3], ['rev_918417868422594349', 5], ['rev_1478773421753396284', 4], ['rev_575668676', 3], ['rev_790152894610968893', 4], ['rev_1347541566885077604', 5], ['rev_1382321496029351103', 2], ['rev_1350471012480046670', 5], ['rev_829297121216897453', 5], ['rev_1494656699655397859', 4], ['rev_1009127579614789075', 2], ['rev_1067108119474751951', 3], ['rev_1208401704813357871', 5], ['rev_1134416297828810554', 4], ['rev_1166391001485642259', 4], ['rev_972147031287505673', 4], ['rev_1222834073213078325', 4], ['rev_1396053784294733070', 2], ['rev_1057648407526245743', 3], ['rev_748789914', 5], ['rev_439885843', 4], ['rev_1082304102869818230', 4], ['rev_850363899576158986', 4], ['rev_1305492497097836105', 4], ['rev_536085617', 5], ['rev_1381653713504494752', 3], ['rev_1299699291336300448', 5], ['rev_1234507499056318928', 4], ['rev_1102601859175813910', 2], ['rev_576624617', 5], ['rev_1498271616809863337', 5], ['rev_1499718257699972910', 5], ['rev_1390975886556804468', 5], ['rev_621894147', 5], ['rev_1156245743912689833', 5], ['rev_407716909', 5], ['rev_494218116', 3], ['rev_1346787622847839718', 4], ['rev_1172868638621461143', 4], ['rev_999700358474680899', 5], ['rev_1469244046529886801', 3], ['rev_858992554430900616', 3], ['rev_195344369', 4], ['rev_167789453', 3], ['rev_896000065623840247', 4], ['rev_1319247704371649832', 4], ['rev_872038659461276880', 4]]}, 'audit_completed_stage2.csv': {'columns': ['audit_id', 'text_id', 'language', 'rakim_label'], 'data': [[1, 'rev_640330351', 'EN', 1.0], [2, 'rev_964851016291133705', 'ES', None], [3, 'rev_281929510', 'IT', None], [4, 'rev_900295551399227462', 'EN', 1.0], [5, 'rev_559874591', 'EN', 5.0], [6, 'rev_589415210', 'EN', 5.0], [7, 'rev_911171087899491017', 'EN', 2.0], [8, 'rev_901034772417040874', 'EN', 5.0], [9, 'rev_1049689062482662756', 'EN', 2.0], [10, 'rev_1038051504343802824', 'EN', 1.0], [11, 'rev_1254036152890945244', 'EN', 2.0], [12, 'rev_1488085595880084222', 'EN', 1.0], [13, 'rev_1419351767379669266', 'EN', 1.0], [14, 'rev_195344369', 'EN', 4.0], [15, 'rev_1134416297828810554', 'NL', 3.0], [16, 'rev_877209054967380759', 'DE', 3.0], [17, 'rev_494218116', 'EN', 2.0], [18, 'rev_1143931122580120750', 'NL', 3.0], [19, 'rev_1191005173852151567', 'DE', 3.0], [20, 'rev_1274314033452316235', 'EN', 4.0], [21, 'rev_1245400643482280391', 'ES', 3.0], [22, 'rev_763365535233263226', 'EN', 2.0], [23, 'rev_940967091058750016', 'EN', 2.0], [24, 'rev_821343007007481845', 'EN', 3.0], [25, 'rev_858311244721139270', 'EN', 4.0], [26, 'rev_1332328260010907493', 'EN', 4.0], [27, 'rev_897419125072974272', 'FR', 3.0], [28, 'rev_714477929', 'EN', 1.0], [29, 'rev_1333011500123144850', 'EN', 5.0], [30, 'rev_945300575718847318', 'EN', 3.0], [31, 'rev_1226534741086546914', 'EN', 4.0], [32, 'rev_620815212', 'EN', 3.0], [33, 'rev_966365763671462574', 'EN', 1.0], [34, 'rev_1324286134590078416', 'EN', 2.0], [35, 'rev_822793222069495516', 'EN', 2.0], [36, 'rev_781531502795568263', 'EN', 1.0], [37, 'rev_1163472549243296031', 'EN', 5.0], [38, 'rev_1279362658904188975', 'ES', 3.0], [39, 'rev_825737954238206680', 'EN', 3.0], [40, 'rev_532511577', 'FR', 3.0], [41, 'rev_1212686246099990362', 'EN', 2.0], [42, 'rev_1098231763048309947', 'EN', 2.0], [43, 'rev_1228680853667393969', 'IT', 3.0], [44, 'rev_1243904690152696527', 'FR', 3.0], [45, 'rev_1138821989485575275', 'EN', 4.0], [46, 'rev_581147094', 'EN', 3.0], [47, 'rev_1319972164182702289', 'EN', 4.0], [48, 'rev_1483767115036113552', 'EN', 3.0], [49, 'rev_1106187287317832260', 'EN', 4.0], [50, 'rev_1102601859175813910', 'EN', 2.0]]}, 'aspect_audit_sample.csv': {'columns': ['audit_id', 'unit_id', 'aspect', 'path', 'sim', 'keywords', 'unit_text'], 'data': [[1, 'rev_1369232332629495911_u1', 'accommodation_value', 'embedding', 0.6194630861, None, 'The house itself is comfortable with almost everything you will need for a short stay.'], [2, 'rev_1127919270850715885_u0', 'accommodation_value', 'embedding', 0.56711936, None, 'It was comfortable space at a convenient residential location.'], [3, 'rev_1481613833644508386_u4', 'accommodation_value', 'embedding', 0.5660777688, None, 'The apartment was minimal (which was what I loved!) yet so thoughtfully furnished, with everything I needed.'], [4, 'rev_1454854737663051753_u1', 'accommodation_value', 'embedding', 0.6830852032, None, 'The room had all the appliances that I needed and was very comfortable.'], [5, 'rev_936610343562918230_u0', 'accommodation_value', 'keyword', 0.265280962, 'clean', 'Incredibly wise charming lady with her clean, cozy, well-organized house and 2 lovely cats!!'], [6, 'rev_1464188876356313642_u2', 'accommodation_value', 'keyword', 0.0900307149, 'worth', 'Definitely a place worth staying at!'], [7, 'rev_1404753087818247819_u0', 'accommodation_value', 'keyword', 0.3094339371, 'clean', 'Very well located ; very easy instruction for the arrival ; large and very clean place.'], [8, 'rev_328014717_u2', 'accommodation_value', 'keyword', 0.4433554113, 'bathroom, kitchen', 'The bathroom has been squeezed into a tiny space and the kitchen is basic, but you probably wouldn’t be cooking here.'], [9, 'rev_1076470418421887143_u7', 'accommodation_value', 'topic+embedding', 0.4726132154, None, 'The apartment was equilped with everything you need such as kettle, microwave, stove, cooker, iron etc.'], [10, 'rev_1203265382464455768_u1', 'accommodation_value', 'topic+embedding', 0.4356540442, None, 'there was only 1 uncomfortable pillow.'], [11, 'rev_449560937_u0', 'accommodation_value', 'topic+embedding', 0.5228923559, None, 'Apartment was small but comfortable, great for a couple.'], [12, 'rev_920637970928895464_u0', 'accommodation_value', 'topic+embedding', 0.6909155846, None, 'nice room, tidy environment host is very helpful'], [13, 'rev_442177174_u12', 'crowd_level', 'embedding', 0.5968993902, None, 'The area is quite so if you are tired you are not disturbed by trains or noises outside the house.'], [14, 'rev_424086763_u3', 'crowd_level', 'embedding', 0.5823799968, None, 'The place is clean and neat, and it is at quite residential area.'], [15, 'rev_524903823_u0', 'crowd_level', 'embedding', 0.5793037415, None, 'A very clean and nice place in a good neighborhood.'], [16, 'ov_1254759970017467345_u2', 'crowd_level', 'embedding', 0.5790713429, None, 'There is also a lot of greenery in the surrounding area, giving the area a relaxed atmosphere even in the city.'], [17, 'rev_1047510851064765563_u0', 'crowd_level', 'keyword', 0.3185017407, 'noisy', 'It is a reasonable price, user-friendly facility, however, it is too close to the highway,a bit of noisy.'], [18, 'rev_1164125977685768472_u3', 'crowd_level', 'keyword', 0.1667039394, 'noise', 'Only problem with the place is that the walls and floor is thin so if the other guest make a bit of noise, you will hear it.'], [19, 'rev_1454121160945337670_u5', 'crowd_level', 'keyword', 0.616861701, 'quiet', 'It’s in a quiet neighbourhood and the beds are really comfortable.'], [20, 'rev_67885720_u3', 'crowd_level', 'keyword', 0.3962512612, 'quiet', 'She showed me her quiet and quaint neighborhood and shared many good tips for the cool places to go.'], [21, 'ov_1452963182804068154_u4', 'crowd_level', 'topic+embedding', 0.4705193043, None, 'The neighborhood is a safe and easily accessible.'], [22, 'rev_333896314_u3', 'crowd_level', 'topic+embedding', 0.7008440495, None, 'The neighborhood is very local and cozy, far from the busiest life of the city.'], [23, 'rev_112217263_u5', 'crowd_level', 'topic+embedding', 0.4605626166, None, 'It is more of a business neighborhood and much less residential.'], [24, 'rev_858279196596328984_u1', 'crowd_level', 'topic+embedding', 0.5113312006, None, 'The location is great and you get a lot of privacy.'], [25, 'rev_1238097692416267517_u1', 'family_friendliness', 'keyword', 0.4880105555, 'family', 'S.Apartment went above and beyond for our family.'], [26, 'rev_902495662413212675_u5', 'family_friendliness', 'keyword', 0.0349281915, 'kids', 'In the other direction also 2 min away, Maruetsu Petit (a supermarket) is where we would grab food and snacks for our kids.'], [27, 'rev_1431561770949837370_u2', 'family_friendliness', 'keyword', 0.4949063063, 'toddler, baby', 'We have a toddler who co sleep with us so we do not need the baby cot.'], [28, 'rev_1435274459393343742_u6', 'family_friendliness', 'keyword', 0.6205254197, 'family', 'It is convenient for the a large family.'], [29, 'rev_1403388529194191566_u0', 'family_friendliness', 'topic+embedding', 0.4793284833, None, 'We stayed 4 nights and for a group of 8, we all slept comfortable.'], [30, 'rev_1155462140574050087_u0', 'family_friendliness', 'topic+embedding', 0.4079040289, None, 'Spacious (especially when with 2 people).'], [31, 'rev_63869458_u0', 'family_friendliness', 'topic+embedding', 0.5146082044, None, 'My father & my son all like it very much.'], [32, 'ov_50778637_u0', 'family_friendliness', 'topic+embedding', 0.4836534262, None, 'Spacious one room for a couple, accommodates max 4 people.'], [33, 'ov_1050385723428727811_u5', 'food_dining', 'keyword', 0.4583215415, 'dining', 'Shopping and Dining Shopping: The area is close to major shopping districts such as Shinjuku and Okubo.'], [34, 'rev_723793117_u2', 'food_dining', 'keyword', 0.8265057802, 'restaurants', 'Plenty shops/restaurants around the area.'], [35, 'rev_1448953431643331464_u2', 'food_dining', 'keyword', 0.3528097272, 'restaurants', 'Walking from the house to the train station was a joy due to all the restaurants and convenience stores on the way.'], [36, 'ov_38111853_u1', 'food_dining', 'keyword', 0.4719561934, 'restaurants', 'There are convenience stores, restaurants, supermarkets, and public baths near my house as well.'], [37, 'rev_1351846825726003343_u0', 'food_dining', 'topic+embedding', 0.4021554589, None, 'Near convenience stores and subway.'], [38, 'ov_1010395870676802741_u4', 'food_dining', 'topic+embedding', 0.4132385254, None, '◉Convenience stores and supermarkets near Peacock Store Sangenjaya no Mori'], [39, 'rev_1252613855787564482_u7', 'food_dining', 'topic+embedding', 0.5583224297, None, "don't miss the street eats on Hoppy Street!"], [40, 'rev_121323891_u1', 'food_dining', 'topic+embedding', 0.5512574315, None, 'Convenience stores are around the place.'], [41, 'ov_1139637342174356482_u1', 'nature_parks', 'embedding', 0.5586922169, None, 'Recommended sightseeing spots ·Sky tree'], [42, 'ov_21103119_u5', 'nature_parks', 'embedding', 0.586359024, None, 'You can also enjoy the beautiful scenery through the seasons along the Sumida and Arakawa rivers that run through the Hikifune area.'], [43, 'ov_53517793_u1', 'nature_parks', 'embedding', 0.5658148527, None, 'Skytree, Sensoji Temple, Ueno Zoo, and many more are located in the area, so please enjoy your trip♪'], [44, 'rev_1227919968963510649_u4', 'nature_parks', 'embedding', 0.5598151684, None, 'The view of Shinjuku is pretty.'], [45, 'ov_18691692_u6', 'nature_parks', 'keyword', 0.4146439135, 'park', 'Families and couples alike will enjoy the nostalgic Hanayashiki amusement park, Japan’s oldest, filled with retro attractions.'], [46, 'ov_1420819457062444316_u1', 'nature_parks', 'keyword', 0.6181144714, 'cherry blossoms, trees', "It's also famous as the birthplace of Somei-Yoshino cherry blossoms, so in spring the cherry trees are in full bloom and the area is bustling with people who come to view the flowers."], [47, 'ov_1384565852982200552_u4', 'nature_parks', 'keyword', 0.6361476183, 'park', 'And when you want a breather, a 15-minute stroll takes you to Arisugawa Park —lush, hilly, and quiet, with winding paths and peaceful ponds.'], [48, 'ov_1136020236548211335_u1', 'nature_parks', 'keyword', 0.2838150263, 'gyoen', 'shinjuku gyoen The Shinjuku Gyoen was established in 1906 on the site of the former residence of the Naito family, lords of the Takato domain in Shinshu during the Edo period (for a detailed history, click here).'], [49, 'ov_9892431_u0', 'nature_parks', 'topic+embedding', 0.4118008018, None, 'Sighseeing spots in this area, all within walking distance!'], [50, 'ov_3264676_u4', 'nature_parks', 'topic+embedding', 0.4250107408, None, 'Famous as a wedding venue, it offers beautiful seasonal scenery.'], [51, 'ov_1145934270830934322_u8', 'nature_parks', 'topic+embedding', 0.4670764208, None, 'It is worth visiting as a place of relaxation for walks, picnics, events, etc.'], [52, 'ov_10999290_u0', 'nature_parks', 'topic+embedding', 0.4194416702, None, 'With fashionable tall commercial buildings standing on both sides of the road, there are some green places like Akasaka Imperial Palace around Aoyama Itchome station.'], [53, 'ov_27879147_u2', 'nightlife', 'embedding', 0.5561169982, None, 'walk, you will be wandering the streets of Shimokitazawa, the bohemian town with countless local shops and restaurants.'], [54, 'ov_50158940_u3', 'nightlife', 'embedding', 0.6208781004, None, 'Traditional restaurants in Ueno and various Japanese taverns are lively place in the night.'], [55, 'rev_1483079989591485361_u6', 'nightlife', 'embedding', 0.5800981522, None, 'Quiet evenings in the heart of Shibuya action with short walks to amazing shopping, attractions, restaurants, etc etc.'], [56, 'ov_29555575_u3', 'nightlife', 'embedding', 0.5834835172, None, 'Just 1-minute walk from the famous gourmet street in Shin-okubo area, 10 minutes walking distance to lively Kabuki-cho.'], [57, 'ov_1370156986158008083_u2', 'nightlife', 'keyword', 0.5179255605, 'kabukicho', 'There are various shops nearby such as Don Quijote, Bic Camera, and shopping malls, and there are various tourist attractions such as the recently opened Tokyu Kabukicho Tower where you can experience fun attractions.'], [58, 'ov_1064759091156204105_u4', 'nightlife', 'keyword', 0.7219905853, 'nightlife, bars', 'For nightlife, there are bars and izakayas near the property, but Shibuya and Shinjuku are also nearby, so you can fully enjoy the nightlife.'], [59, 'ov_35779580_u1', 'nightlife', 'keyword', 0.6229453087, 'kabukicho', 'It is also close to Kabukicho and Golden Street.'], [60, 'ov_29024905_u0', 'nightlife', 'keyword', 0.5222984552, 'night life', 'About Shinjuku (3 mins by train / 10 mins on foot from the apartment) Shinjuku is one of the biggest spots to go shopping, sightseeing, and experience Tokyo’s night life.'], [61, 'ov_41297165_u0', 'nightlife', 'topic+embedding', 0.4904285073, None, 'In the center of Shinjuku.'], [62, 'ov_30783354_u0', 'nightlife', 'topic+embedding', 0.4058927894, None, 'Located close to central Shinjuku area.'], [63, 'ov_32875538_u0', 'nightlife', 'topic+embedding', 0.4307277799, None, 'The location of the hotel is located in the residential area of Shinjuku District, it is very lively after exiting the alley to the main road, transportation, shopping, and eating are convenient.'], [64, 'ov_1097446230178330204_u1', 'nightlife', 'topic+embedding', 0.553769052, None, "It is also close to Shinjuku, one of Tokyo's largest downtown areas, where you can enjoy a variety of gourmet food, shopping, and entertainment."], [65, 'rev_566898635_u0', 'safety', 'keyword', 0.2359736562, 'safe', 'Great and safe place to stay if you’re spending a couple of nights in Tokyo.'], [66, 'rev_1078649026431471702_u0', 'safety', 'keyword', 0.2613722384, 'safe', 'Good place to stay, for the price you pay is a really nice accomodation, clean, safe and warm.'], [67, 'rev_1197473049229536134_u3', 'safety', 'keyword', 0.6668774486, 'safe', 'Even as a solo female traveller I felt extremely safe even late in the night.'], [68, 'rev_1224271817957366849_u1', 'safety', 'keyword', 0.2740520835, 'safety', 'We would email host (their preferred method of communication) about specific issues related to safety of building, garbage disposal and we would receive AI messages that didn’t address the issues.'], [69, 'rev_1414252697094209731_u1', 'safety', 'topic+embedding', 0.4523882568, None, 'The neighborhood was quiet and peaceful.'], [70, 'ov_41992108_u0', 'safety', 'topic+embedding', 0.4153720737, None, '[Atmosphere of the surrounding area] The house is in a quiet residential area so you can rest in a very quiet environment at night.'], [71, 'rev_1001792351468874225_u0', 'safety', 'topic+embedding', 0.4253663421, None, 'very quiet and cosy place.'], [72, 'rev_987296062064292737_u2', 'safety', 'topic+embedding', 0.4494318962, None, 'Neighbor is quiet and clean.'], [73, 'ov_12161159_u0', 'shopping', 'embedding', 0.5511174798, None, 'Quiet residential area in busy Shinjuku area 3 min walk to the convenience store.'], [74, 'ov_1450457966225777452_u3', 'shopping', 'embedding', 0.6052196026, None, 'In the Omotesando and Daikanyama areas, you can enjoy strolling around stylish cafes☕ and unique stores.'], [75, 'ov_32456280_u1', 'shopping', 'embedding', 0.5999633074, None, 'There’re parks, shrines and temples, supermarkets, convenience stores, charming restaurants, and Izakaya(Japanese style bar) within walking distance.'], [76, 'ov_9096540_u0', 'shopping', 'embedding', 0.6126237512, None, 'Restaurants.Izakaya, family-friendly, supermarket.There are many convenience stores'], [77, 'ov_35068494_u1', 'shopping', 'keyword', 0.3626397848, 'department store', 'If you take the south exit out, you can find LOFT and UNIQLO and a department store "LUMINE", very convenient!!'], [78, 'rev_1233018152696085210_u5', 'shopping', 'keyword', 0.5199377537, 'market, shops', 'close to the station and a market, and many shops and restaurant options.'], [79, 'ov_842308330178246538_u7', 'shopping', 'keyword', 0.3660111427, 'ginza, akihabara', 'Roppongi, Harajuku, Shibuya, Ginza, and Akihabara are also nearby, and there is great access to tourist destinations as well.'], [80, 'rev_1372848215190379764_u0', 'shopping', 'keyword', 0.0360911526, 'akihabara', 'This hostel is a really good place to stay next to Akihabara.'], [81, 'ov_26959352_u12', 'shopping', 'topic+embedding', 0.4302895963, None, 'Convinience store (7-Eleven / LAWSON) : 2 minutes walk'], [82, 'ov_949667782779849523_u0', 'shopping', 'topic+embedding', 0.5506875515, None, '◉Nearby convenience stores and supermarkets 7-Eleven Sumida Higashi Mukojima 3-chome 2 minutes on foot Family Mart Tsubota Riverside 3-minute walk Life Higashimukojima 3-minute walk 4-minute walk from the supermarket Sumida Restaurants ◉Nearby American Bath School of Block 20-second walk 1-minute walk to the restaurant Katayama Chinese Cuisine Hiroko House 2 minutes on foot Japanese food and Sumida Tidori store 3-minute walk There are also many other restaurants around the property.'], [83, 'ov_738680413287783221_u11', 'shopping', 'topic+embedding', 0.5294860601, None, '100 yen store "Seria" : 20m 1 minute walk'], [84, 'ov_738680413287783221_u10', 'shopping', 'topic+embedding', 0.4412323833, None, 'Supermarket "Maruetsu" : 20m, 1 min.'], [85, 'ov_1244908208132213609_u0', 'traditional_cultural', 'embedding', 0.5616529584, None, '✨ Experience the Charm of Okushibu Area: Shibuya’s Hidden Gem!'], [86, 'ov_40975971_u6', 'traditional_cultural', 'embedding', 0.5500049591, None, 'From Nishisugamo Station to Famous Sightseeing Spots 1 ️'], [87, 'ov_1019413976349644343_u0', 'traditional_cultural', 'embedding', 0.5989388227, None, 'Shinjuku Kabukicho Sightseeing Walking Area'], [88, 'ov_1508397793067652748_u0', 'traditional_cultural', 'embedding', 0.5783262253, None, '✨ Experience the Charm of Yoyogi-Uehara — Shibuya’s Peaceful Hideaway!'], [89, 'ov_909344860610734420_u1', 'traditional_cultural', 'keyword', 0.4292472005, 'museum', 'There are convenience stores, restaurants, Yayoi Kusama Museum of Art, etc.'], [90, 'ov_22349368_u0', 'traditional_cultural', 'keyword', 0.2524693608, 'traditional', 'Traditional Village Tourist Area Ueno Zoo Cafeteria Natural Hot Springs Walk'], [91, 'ov_979330577299649915_u6', 'traditional_cultural', 'keyword', 0.4695155025, 'culture', 'Ikebukuro also includes many animation-related attractions, nurturing the unique culture of the place.'], [92, 'ov_48142320_u0', 'traditional_cultural', 'keyword', 0.4511226714, 'history, temple', 'The area around Akabane-bashi Station is a quiet and calm residential area, but it is a well-balanced area with spots where you can experience history and nature, such as Shiba Park, Tokyo Tower, and Zojo-ji Temple.'], [93, 'ov_971620486337173403_u10', 'traditional_cultural', 'topic+embedding', 0.6694966555, None, 'Tourist attractions Located in the heart of the Ueno and Asaku'], [94, 'ov_36527719_u0', 'traditional_cultural', 'topic+embedding', 0.5886833072, None, 'The streets of Japanese art and humanities.'], [95, 'ov_35659060_u12', 'traditional_cultural', 'topic+embedding', 0.4056269825, None, '14 min to Suidobashi Station(Tokyo Dome) by train Of course, you can easily access famous sightseeing spots by train!'], [96, 'ov_1152741116573264065_u6', 'traditional_cultural', 'topic+embedding', 0.4006046057, None, 'Boro Ichi in Setagaya is a popular event with nearly 700 stalls lined up along the street from Setagaya Station to the neighboring Kamimachi Station, and is crowded with visitors every year during the year-end and New Year holidays.'], [97, 'ov_1076498786076146531_u6', 'walkability_transit', 'embedding', 0.5791682005, None, '15 minutes to Tokyo Tower'], [98, 'rev_1248288626783776257_u2', 'walkability_transit', 'embedding', 0.5704833269, None, '15 minutes walk to Tokyo Tower.'], [99, 'ov_1038326124016042379_u4', 'walkability_transit', 'embedding', 0.5596219301, None, '15-minute drive to Tokyo Tower.'], [100, 'ov_1192553452203225647_u1', 'walkability_transit', 'embedding', 0.575922966, None, '3 min by walk Tokyo Skytree L'], [101, 'ov_1117088525078292065_u0', 'walkability_transit', 'keyword', 0.5516331196, 'station, jr, yamanote', 'Otsuka Station is a stop on the JR Yamanote Line, a major line in central Tokyo, and is conveniently located next to Ikebukuro Station, a terminal station!'], [102, 'rev_872763534886545890_u0', 'walkability_transit', 'keyword', 0.1493580043, 'lines to', 'If u want to beat the lines to some of the more popular stores/restaurants at Tsukiji outer market, then this is the place for u.'], [103, 'ov_1031452261840366934_u1', 'walkability_transit', 'keyword', 0.2927592993, 'station', 'In addition to supermarkets and drug stores around the station, there are also lively shopping streets, making it convenient for everyday shopping.A short walk away, there is the lush Kineuta Park and the Setagaya Art Museum, where you can easily enjoy nature and culture on your days off.'], [104, 'ov_38676850_u2', 'walkability_transit', 'keyword', 0.7260518074, 'station, metro', 'Ochiai Station is about a 9-minute on foot from my house, so you can also use the Tokyo Metro Tozai Line'], [105, 'ov_43385421_u4', 'walkability_transit', 'topic+embedding', 0.5976700187, None, 'You can easily take the tram to reach the main bustling areas such as Shinjuku, Roppongi, Harajuku, Shibuya, etc.!'], [106, 'ov_1306230906627448130_u4', 'walkability_transit', 'topic+embedding', 0.4479343891, None, 'Tokyo Disneyland: 60 minutes (2 transfers)'], [107, 'rev_1509173666147111474_u0', 'walkability_transit', 'topic+embedding', 0.4688521624, None, 'Good location, easy walk to SHIBUYA.'], [108, 'ov_44939084_u1', 'walkability_transit', 'topic+embedding', 0.5865728855, None, 'It takes 20 minutes to tram to Shinjuku, Shibuya, Harajuku, Shimokitazawa, Kichijoji and other business districts.']]}, 'aspect_audit_llm.csv': {'columns': ['audit_id', 'correct'], 'data': [[1, 1], [2, 1], [3, 1], [4, 1], [5, 1], [6, 1], [7, 1], [8, 1], [9, 1], [10, 1], [11, 1], [12, 1], [13, 1], [14, 1], [15, 0], [16, 1], [17, 1], [18, 0], [19, 1], [20, 1], [21, 0], [22, 1], [23, 1], [24, 0], [25, 1], [26, 1], [27, 1], [28, 1], [29, 0], [30, 0], [31, 1], [32, 0], [33, 1], [34, 1], [35, 1], [36, 1], [37, 0], [38, 0], [39, 1], [40, 0], [41, 0], [42, 1], [43, 0], [44, 0], [45, 0], [46, 1], [47, 1], [48, 1], [49, 0], [50, 1], [51, 1], [52, 1], [53, 0], [54, 1], [55, 0], [56, 1], [57, 0], [58, 1], [59, 1], [60, 1], [61, 0], [62, 0], [63, 0], [64, 1], [65, 1], [66, 1], [67, 1], [68, 1], [69, 0], [70, 0], [71, 0], [72, 0], [73, 0], [74, 1], [75, 0], [76, 0], [77, 1], [78, 1], [79, 0], [80, 0], [81, 0], [82, 0], [83, 0], [84, 0], [85, 0], [86, 0], [87, 0], [88, 0], [89, 1], [90, 1], [91, 1], [92, 1], [93, 1], [94, 1], [95, 0], [96, 1], [97, 1], [98, 1], [99, 1], [100, 1], [101, 1], [102, 0], [103, 1], [104, 1], [105, 1], [106, 1], [107, 1], [108, 1]]}, 'sentence_val_candidates.csv': {'columns': ['item', 'text_id', 'unit_id', 'ward', 'aspect', 'band', 'sentiment_score', 'exp_star', 'pred_star', 'supporting_span'], 'data': [[1, 'rev_1445412511797024861', 'rev_1445412511797024861_u8', 'Minato', 'nature_parks', 'pos', 0.785389211, 4.5707784221, 5, 'When we arrived, he greeted us warmly, helped direct traffic so we could park our car on a narrow street, carried our luggage, and patiently showed us how to use the amenities.'], [2, 'rev_424543807', 'rev_424543807_u1', 'Setagaya', 'crowd_level', 'mid', 0.4283081246, 3.8566162492, 4, 'Nice quiet neighbourhood with some noises from the street in the daytime'], [3, 'rev_1327932107130465257', 'rev_1327932107130465257_u0', 'Itabashi', 'shopping', 'pos', 0.7641188957, 4.5282377914, 5, 'Great place to stay when planning on shopping around tokyo area, very close to train station.'], [4, 'rev_851046309010178868', 'rev_851046309010178868_u2', 'Shinjuku', 'nightlife', 'mid', 0.4770754264, 3.9541508527, 4, 'You have a private room with a balcony all to yourself, while you are also able to socialize with the other people staying if you like, and maybe experience the great nightlife of Tokyo!'], [5, 'rev_408916493708608065', 'rev_408916493708608065_u2', 'Minato', 'nightlife', 'mid', 0.1016374752, 3.2032749504, 5, 'The refrigerator was like a “Mini-bar” and unfriendly for cooking by ourselves'], [6, 'rev_1001125037615310888', 'rev_1001125037615310888_u4', 'Setagaya', 'family_friendliness', 'mid', 0.3402559664, 3.6805119328, 4, 'Many young families seemed to live in the area which was perfect for us as we are travelling with our 10month old baby.'], [7, 'rev_1140961843419158123', 'rev_1140961843419158123_u6', 'Sumida', 'nature_parks', 'mid', 0.4676130087, 3.9352260175, 4, 'Overall, I enjoyed the ambiance and convenience of staying at Sakura Inn!'], [8, 'rev_258896118', 'rev_258896118_u3', 'Minato', 'walkability_transit', 'neg', -0.3979595229, 2.2040809542, 1, 'Only about 5 mins away from Shimbashi station, a major JR station, and spotlessly clean and tidy.'], [9, 'rev_1391747427457310410', 'rev_1391747427457310410_u1', 'Minato', 'nature_parks', 'neg', -0.4771242011, 2.0457515977, 2, 'Two of us stayed for 2 weeks, admittedly it was sakura season'], [10, 'rev_1277287456640526419', 'rev_1277287456640526419_u5', 'Toshima', 'nightlife', 'neg', -0.3000968164, 2.3998063672, 2, 'Not particularly for nightlife fans (first metro goes at 4:30'], [11, 'rev_1195282689740467512', 'rev_1195282689740467512_u5', 'Shinjuku', 'shopping', 'neg', -0.5530270678, 1.8939458644, 2, 'Under the TV the electrical outlet usage felt dangerous (surge protector plugged into another protector).'], [12, 'rev_1454083080626832636', 'rev_1454083080626832636_u2', 'Sumida', 'accommodation_value', 'pos', 0.676444042, 4.352888084, 4, 'Spacious main area with a double, sofa bed and a futon as well.'], [13, 'rev_1071349361407685311', 'rev_1071349361407685311_u4', 'Toshima', 'family_friendliness', 'neg', -0.5243454175, 1.951309165, 1, 'Also there is no portable wifi and they don’t you to make any noise which can be hard with two kids.'], [14, 'rev_1462719028990693486', 'rev_1462719028990693486_u3', 'Minato', 'accommodation_value', 'neg', -0.6767509882, 1.6464980235, 1, 'The apartment was spotless and'], [15, 'rev_411903047', 'rev_411903047_u1', 'Koto', 'crowd_level', 'mid', 0.2212870959, 3.4425741918, 4, 'A few even 24hrs which is great if you are not quiet on the right time zone and wake up hunger at 3am LOL.'], [16, 'rev_610696209', 'rev_610696209_u2', 'Minato', 'food_dining', 'pos', 0.8412363484, 4.6824726969, 5, 'The surrounding area is very nice and close to the market and awesome restaurants!'], [17, 'rev_1200454934957976148', 'rev_1200454934957976148_u2', 'Toshima', 'safety', 'pos', 0.5290174913, 4.0580349825, 4, 'It is in a quiet residential neighbourhood of Tokyo, felt very safe and relaxing'], [18, 'rev_1127881790732066128', 'rev_1127881790732066128_u2', 'Shibuya', 'nature_parks', 'neg', -0.3160716686, 2.3678566627, 2, 'Only minutes from Yoyogi station.'], [19, 'rev_1496785907946261117', 'rev_1496785907946261117_u0', 'Minato', 'food_dining', 'pos', 0.8816221436, 4.7632442871, 5, 'Thank you very much to the host, this was an excellent stay with easy access to public transportation and lots of local activities/restaurants'], [20, 'rev_1264239712784296799', 'rev_1264239712784296799_u1', 'Taito', 'traditional_cultural', 'pos', 0.7105573993, 4.4211147986, 5, 'Transportation was also very easy with all the bus stops close by and not being far from Asakusa Station.'], [21, 'rev_1269255602716143816', 'rev_1269255602716143816_u1', 'Sumida', 'crowd_level', 'pos', 0.6872180005, 4.3744360011, 4, 'It is also on a quiet street which allowed us to get great rest after long days of exploring.'], [22, 'rev_928611293126674642', 'rev_928611293126674642_u1', 'Taito', 'traditional_cultural', 'neg', -0.2699622139, 2.4600755721, 3, 'Asasuka temple area is only minutes away.'], [23, 'rev_1420794581404316649', 'rev_1420794581404316649_u3', 'Toshima', 'crowd_level', 'pos', 0.6343856386, 4.2687712773, 4, 'The place is nice and peaceful.'], [24, 'rev_1280863120173763217', 'rev_1280863120173763217_u2', 'Sumida', 'traditional_cultural', 'neg', -0.3052855544, 2.3894288912, 3, 'at least it is only about 5min walk away from Morishita station and pretty near Akihabara / Asakusa.'], [25, 'rev_1154776177334351603', 'rev_1154776177334351603_u0', 'Taito', 'traditional_cultural', 'pos', 0.7709790558, 4.5419581117, 5, "This home is well located in Asakusa near Senso-ji and is very convenient for getting around Tokyo's subway system."], [26, 'rev_927145812488213119', 'rev_927145812488213119_u1', 'Setagaya', 'walkability_transit', 'pos', 0.8230516323, 4.6461032645, 5, 'The place was really clean, and the location was perfect with everything we wanted to see in Tokyo 2 trains away at most.'], [27, 'rev_1169999054607570278', 'rev_1169999054607570278_u0', 'Chuo', 'shopping', 'mid', 0.4550712188, 3.9101424376, 4, 'The location was good, within 8-15 minutes walk from subway lines and near major areas such as Ginza and Tsukiji Market.'], [28, 'rev_1426554323534034844', 'rev_1426554323534034844_u0', 'Chiyoda', 'walkability_transit', 'neg', -0.2900918433, 2.4198163133, 3, 'Notes: -Access is via 2 levels of stairs only.'], [29, 'rev_1027905743710674593', 'rev_1027905743710674593_u2', 'Minato', 'food_dining', 'mid', -0.1236656718, 2.7526686564, 2, 'Short walking distance from multiple grocery stores, restaurants, and to Azabu Juban station.'], [30, 'rev_1510570319195436483', 'rev_1510570319195436483_u3', 'Taito', 'family_friendliness', 'mid', 0.4035900403, 3.8071800806, 5, 'Would definitely rebook this place for any future family outings.'], [31, 'rev_920619628020481928', 'rev_920619628020481928_u0', 'Toshima', 'crowd_level', 'neg', -0.5982237714, 1.8035524571, 2, 'it’s too noisy!'], [32, 'rev_1004040733153659615', 'rev_1004040733153659615_u5', 'Shinjuku', 'crowd_level', 'mid', 0.0124145821, 3.0248291641, 3, 'And while we found the neighborhood to be quiet, there is a path/road right next to the bedroom that definitely experienced some noise through the night.'], [33, 'rev_1359895280987134993', 'rev_1359895280987134993_u1', 'Koto', 'nature_parks', 'neg', -0.2901733145, 2.419653371, 1, 'Saw a sakura tree 5 minutes walk away the day before the flight!'], [34, 'rev_1298940154588535309', 'rev_1298940154588535309_u0', 'Toshima', 'accommodation_value', 'pos', 0.6952215145, 4.390443029, 4, 'Affordable price, quiet neighborhood, spacious room, the toilet is great with bidet, big refrigerator, it’s located on the first floor on the side of the road.'], [35, 'rev_435931005', 'rev_435931005_u5', 'Minato', 'family_friendliness', 'pos', 0.7631274536, 4.5262549073, 5, 'Also I had to wash my baby clothing and I used the washer which was perfect and convenient !!!'], [36, 'rev_1375133700561334129', 'rev_1375133700561334129_u0', 'Koto', 'walkability_transit', 'pos', 0.79274155, 4.5854831, 5, 'We had a lovely time at this clean and spacious apartment near Kameido station.'], [37, 'rev_1017059320510849684', 'rev_1017059320510849684_u1', 'Minato', 'food_dining', 'neg', -0.5618259367, 1.8763481267, 1, 'carpet stains in lobby and rooms, dust on bed frame, breakfast consists of small bread and coffee/tea (not actual food)'], [38, 'rev_1320766676131074966', 'rev_1320766676131074966_u0', 'Itabashi', 'crowd_level', 'pos', 0.7536296726, 4.5072593451, 5, 'It’s a really nice home in a very private and quiet area.'], [39, 'rev_875010452556970061', 'rev_875010452556970061_u3', 'Koto', 'accommodation_value', 'mid', 0.1252342034, 3.2504684068, 3, 'But is good for the price.'], [40, 'rev_1466425943337150609', 'rev_1466425943337150609_u0', 'Shinjuku', 'family_friendliness', 'mid', 0.3921979032, 3.7843958065, 5, 'We are a family of 4 with 2 boys aged 14 and 11.'], [41, 'rev_1170021235498627106', 'rev_1170021235498627106_u0', 'Shinjuku', 'walkability_transit', 'neg', -0.6342977937, 1.7314044125, 1, 'Less than a minute walk to the nearest metro.'], [42, 'rev_1301165994857073801', 'rev_1301165994857073801_u5', 'Sumida', 'nightlife', 'pos', 0.8354272132, 4.6708544265, 5, 'While in this area I’d highly recommend checking out a bar called Kaeruhonpo, very friendly barman who speaks perfect English.'], [43, 'rev_1320679279843647239', 'rev_1320679279843647239_u1', 'Setagaya', 'nightlife', 'pos', 0.9457216481, 4.8914432961, 5, 'Amazing apartment, clean, located in a beautiful area full of bars, restaraunts and vintage stores.'], [44, 'rev_1243199768915338115', 'rev_1243199768915338115_u0', 'Minato', 'crowd_level', 'mid', 0.3759834794, 3.7519669589, 4, 'It was really cozy and quiet.'], [45, 'rev_405376854487786237', 'rev_405376854487786237_u4', 'Shibuya', 'accommodation_value', 'pos', 0.5350094363, 4.0700188726, 4, 'The bathroom had clothes dryer which we used and was convenient.'], [46, 'rev_524926640', 'rev_524926640_u4', 'Koto', 'nightlife', 'neg', -0.3896204084, 2.2207591832, 2, 'only 10 minutes from a station and with lots of shops/supermarket, bars etc.'], [47, 'rev_1310550993950489142', 'rev_1310550993950489142_u6', 'Sumida', 'traditional_cultural', 'pos', 0.8956818584, 4.7913637168, 5, 'It’s one of the most affordable Airbnb options in Asakusa, with a clean and spacious interior and almost complete amenities.'], [48, 'rev_1504066050515532729', 'rev_1504066050515532729_u0', 'Toshima', 'walkability_transit', 'mid', 0.3512687103, 3.7025374207, 4, 'The price is reasonable, the location is about a 10-minute walk from the subway, and the room is clean.'], [49, 'rev_995274734967458583', 'rev_995274734967458583_u3', 'Taito', 'nightlife', 'mid', 0.4666455593, 3.9332911186, 4, 'There was accessible drinking water and a washing machine.'], [50, 'rev_1515697653130306003', 'rev_1515697653130306003_u0', 'Taito', 'shopping', 'mid', 0.4819705458, 3.9639410917, 4, 'Close to the shops of Asakusa.'], [51, 'rev_761250666998437330', 'rev_761250666998437330_u4', 'Chiyoda', 'safety', 'neg', -0.7621326442, 1.4757347116, 1, 'Dangerous sharp corner of wood next to bed at eye level.'], [52, 'rev_1211267622403076126', 'rev_1211267622403076126_u4', 'Taito', 'shopping', 'mid', 0.0642630197, 3.1285260394, 3, 'I did not mind it because it’s at a great location not too far from Ginza where we did lots of shopping.'], [53, 'rev_1267865037471470984', 'rev_1267865037471470984_u7', 'Koto', 'nature_parks', 'neg', -0.7527747005, 1.494450599, 1, 'One word of caution, do not exit the metro on the wrong side (as we did one time) you end up on the wrong side of a river with no easy way to cross.'], [54, 'rev_1325025195186697938', 'rev_1325025195186697938_u5', 'Sumida', 'accommodation_value', 'mid', 0.0153304297, 3.0306608593, 3, 'Not too much clean, but ok.'], [55, 'rev_1108349269001840682', 'rev_1108349269001840682_u12', 'Sumida', 'traditional_cultural', 'mid', 0.0322567159, 3.0645134319, 3, 'However, the house is Japanese traditional wooden house, It is best to keep in mind you will hear footsteps, the sound of the door closing.'], [56, 'rev_1325017912182793925', 'rev_1325017912182793925_u1', 'Shibuya', 'nature_parks', 'neg', -0.531226946, 1.9375461079, 1, 'The walk to Yoyogi station was about 5 minutes and there is a McDonald’s and 7-11 within a 2 minute walk.'], [57, 'rev_1451139736212101078', 'rev_1451139736212101078_u6', 'Taito', 'crowd_level', 'pos', 0.5727594784, 4.1455189567, 5, 'Calm area and never bothered by any sound in the apt as it’s off main streets.'], [58, 'rev_365921762', 'rev_365921762_u1', 'Setagaya', 'family_friendliness', 'pos', 0.7785343493, 4.5570686986, 5, 'Yui and the family has been very helpful and patient right from the start.'], [59, 'rev_1293123304138252406', 'rev_1293123304138252406_u2', 'Taito', 'family_friendliness', 'pos', 0.8193645757, 4.6387291513, 5, 'It made my first Japan experience taking my family around even better than it already was.'], [60, 'rev_146405510', 'rev_146405510_u9', 'Setagaya', 'nature_parks', 'mid', 0.3926393576, 3.7852787152, 4, 'Although the distance to the nearest station (Sakurashimmachi) was quite a walk, our daily hikes to and from the station easily became pleasant strolls along sidewalks lined with cherry blossom trees.'], [61, 'rev_985826814722483992', 'rev_985826814722483992_u2', 'Taito', 'traditional_cultural', 'mid', 0.3659073189, 3.7318146378, 5, 'From Ginza Line Asakusa station it takes about 15-17mins walking.'], [62, 'rev_1388185037337847482', 'rev_1388185037337847482_u4', 'Sumida', 'crowd_level', 'neg', -0.4038673863, 2.1922652274, 2, 'very strict on noise and there is a camera there (kind of creepy) we basically didn’t use it.'], [63, 'rev_459923089', 'rev_459923089_u0', 'Setagaya', 'crowd_level', 'pos', 0.8591847854, 4.7183695708, 5, 'Accommodation 15 minutes from Shibuya, in a super quiet area, with all the necessary services just a few minutes, and one of the friendliest hosts I have ever encountered.'], [64, 'rev_355829807', 'rev_355829807_u2', 'Sumida', 'crowd_level', 'neg', -0.4986659046, 2.0026681907, 1, 'it says it can sleep 7 people we were only 3 people and thought having 7 people in this apartment would be crowded.'], [65, 'rev_1125069156899634321', 'rev_1125069156899634321_u3', 'Toshima', 'safety', 'mid', 0.4295069713, 3.8590139425, 4, 'overall it feels safe.'], [66, 'rev_1365621700305684109', 'rev_1365621700305684109_u1', 'Taito', 'family_friendliness', 'neg', -0.2954153469, 2.4091693062, 2, 'not as suitable for the family accommodation we were looking for.'], [67, 'rev_1430116897977182922', 'rev_1430116897977182922_u5', 'Shibuya', 'safety', 'neg', -0.4519518716, 2.0960962567, 2, 'We felt pretty unsafe with a key to our apartment just out there in the world.'], [68, 'rev_1163459247247861355', 'rev_1163459247247861355_u0', 'Chiyoda', 'shopping', 'pos', 0.8608775852, 4.7217551705, 5, 'Great location, close to everything ( shopping, local testing, the park, art galleries, museums).'], [69, 'rev_733607048481699172', 'rev_733607048481699172_u7', 'Minato', 'food_dining', 'mid', -0.1121802852, 2.7756394297, 3, 'The bathroom had extensive amount of mould between the tiles and on the wall as well as other areas in the living room still had food and stains from former guests.'], [70, 'rev_1278654135090364803', 'rev_1278654135090364803_u6', 'Setagaya', 'safety', 'neg', -0.4363108773, 2.1273782453, 2, 'We did not like the following things so much: The security camera on the ceiling inside the flat in the entrance area felt creepy.'], [71, 'rev_1125785738679384604', 'rev_1125785738679384604_u1', 'Shinjuku', 'shopping', 'neg', -0.628451529, 1.7430969421, 1, 'only 10 mins walk from the bright lights, shopping and chaos.'], [72, 'rev_1297507668840582916', 'rev_1297507668840582916_u0', 'Shibuya', 'family_friendliness', 'mid', 0.1419629604, 3.2839259207, 5, 'We were as family (3 persons) and first time in Tokyo.'], [73, 'rev_846022837929348964', 'rev_846022837929348964_u1', 'Minato', 'walkability_transit', 'mid', 0.3715770715, 3.743154143, 4, 'The best part was that the apartment is easily accessible from the subway.'], [74, 'rev_1400419537737492385', 'rev_1400419537737492385_u3', 'Setagaya', 'safety', 'pos', 0.7406751143, 4.4813502287, 5, 'I added layers of picnic mats to the table to make it usable and safe.'], [75, 'rev_1325104355024170046', 'rev_1325104355024170046_u5', 'Setagaya', 'nature_parks', 'mid', 0.3606991312, 3.7213982623, 4, '(bonus point is the house has a very pretty garden)'], [76, 'rev_1116340555329723506', 'rev_1116340555329723506_u5', 'Shibuya', 'nightlife', 'neg', -0.3918989468, 2.2162021063, 2, 'When paying over $500 USD PER NIGHT, we expected more than 4 disposable drinking cups.'], [77, 'rev_1406294007212721813', 'rev_1406294007212721813_u0', 'Taito', 'walkability_transit', 'pos', 0.8315502671, 4.6631005341, 5, 'Great place to stay for solo travellers, easily accessible from JR lines.'], [78, 'rev_1138790443538468230', 'rev_1138790443538468230_u4', 'Setagaya', 'food_dining', 'neg', -0.5123368576, 1.9753262848, 2, 'we didn’t cook as we eat out everyday.'], [79, 'rev_915532807553287696', 'rev_915532807553287696_u7', 'Chuo', 'accommodation_value', 'neg', -0.3430763176, 2.3138473649, 2, 'Only very small amounts of items in the kitchen for cooking/eating, absolutely bare minimum'], [80, 'rev_476299609', 'rev_476299609_u4', 'Sumida', 'safety', 'mid', 0.4399230015, 3.8798460029, 4, "You have to read the notice from the host (really do), they are using electron chains to keep rooms safe, you won't know the code to enter to the building unless you read the notice."], [81, 'rev_347825317', 'rev_347825317_u4', 'Toshima', 'shopping', 'neg', -0.3881437927, 2.2237124145, 1, 'She even got us diapers when we weren’t able to find any in any of the convenience stores, pharmacies, grocery stores or department stores in Tokyo.'], [82, 'rev_1146730717065423206', 'rev_1146730717065423206_u3', 'Toshima', 'food_dining', 'mid', 0.2402393539, 3.4804787077, 4, 'There are also many restaurants that are a walkable distance and a convenience store about two minutes away.'], [83, 'rev_1276537256749072171', 'rev_1276537256749072171_u2', 'Taito', 'family_friendliness', 'neg', -0.3267144953, 2.3465710094, 2, 'the toilet is small and water tap is abit hard to use as mixture function, especially for kids and old folk if any.'], [84, 'rev_532488194', 'rev_532488194_u1', 'Sumida', 'nightlife', 'neg', -0.5472602171, 1.9054795657, 2, 'The location abounds with places to shop, eat, and drink (I recommend trying a Sake bar called "No.'], [85, 'rev_168668779', 'rev_168668779_u2', 'Minato', 'nightlife', 'mid', 0.3640180789, 3.7280361578, 5, '10 minutes from Roppongi Hills with a cinema, 15 from Modern Art Centre and Midtown.'], [86, 'rev_851101449304519196', 'rev_851101449304519196_u0', 'Taito', 'family_friendliness', 'mid', 0.3829754828, 3.7659509657, 4, "Tashi's place in Ueno is a good place for a large or multi-family stay in Tokyo."], [87, 'rev_1030846516087900699', 'rev_1030846516087900699_u2', 'Shinjuku', 'shopping', 'neg', -0.2833640794, 2.4332718411, 3, 'While the location was great (near a train station and several shops) the address in the AirBnb write-up is not correct and is approx.'], [88, 'rev_1132330329919495661', 'rev_1132330329919495661_u1', 'Setagaya', 'accommodation_value', 'neg', -0.4604319736, 2.0791360529, 2, 'The space felt too cramped.'], [89, 'rev_1443203413951580709', 'rev_1443203413951580709_u0', 'Sumida', 'family_friendliness', 'pos', 0.7278397786, 4.4556795573, 5, 'We stayed here with our 10 and 12-year-old kids.'], [90, 'rev_1296044942341524172', 'rev_1296044942341524172_u0', 'Shibuya', 'nature_parks', 'pos', 0.7205297245, 4.441059449, 5, 'We loved the clean neighborhood and being able to walk to a park and the Shibuya area.'], [91, 'rev_1183798341527361485', 'rev_1183798341527361485_u2', 'Taito', 'safety', 'neg', -0.5617289784, 1.8765420433, 2, 'Frustratingly, the apartment is missing all sorts of commonly expected amenities including a clothes iron, conditioner, microwave safe dishes, cups (there are mugs'], [92, 'rev_1494636174738919112', 'rev_1494636174738919112_u1', 'Taito', 'traditional_cultural', 'mid', 0.353038637, 3.7060772739, 4, 'The location being close to Ueno, Asakusa and Akihabara by way of a quick train ride was sweet'], [93, 'rev_1074286153506374363', 'rev_1074286153506374363_u3', 'Minato', 'nightlife', 'neg', -0.2952639153, 2.4094721694, 2, 'Most embassies are in a short walking distance and Roppongi is only a 25 minutes walk away.'], [94, 'rev_472062642', 'rev_472062642_u0', 'Toshima', 'traditional_cultural', 'pos', 0.8713013488, 4.7426026977, 5, 'Great place to stay, a unique experience for me because of the traditional feel, really close to station and very responsive and nice host!'], [95, 'rev_861955699006086768', 'rev_861955699006086768_u1', 'Chiyoda', 'walkability_transit', 'mid', 0.3627993464, 3.7255986929, 5, '10 minutes from Ueno station after my arrival by train from Narita Airport.'], [96, 'rev_1390309060734801083', 'rev_1390309060734801083_u0', 'Minato', 'nightlife', 'pos', 0.8619372519, 4.7238745037, 5, 'Our stay in Roppongi was wonderful!'], [97, 'rev_804685091873857822', 'rev_804685091873857822_u6', 'Minato', 'accommodation_value', 'neg', -0.588145157, 1.8237096861, 2, 'Also, the sponge that was left for washing dishes was dirty and had hair on it.'], [98, 'rev_1513536602847253397', 'rev_1513536602847253397_u1', 'Setagaya', 'accommodation_value', 'pos', 0.6353639099, 4.2707278198, 5, 'Room was also clean and cozy.'], [99, 'rev_1402580285456833501', 'rev_1402580285456833501_u0', 'Taito', 'shopping', 'pos', 0.6791605891, 4.3583211782, 4, 'Good value stay close to Senso-Ji (popular temple), Asakusa shopping, and the station.'], [100, 'rev_1355568006630800560', 'rev_1355568006630800560_u1', 'Toshima', 'traditional_cultural', 'neg', -0.570594595, 1.8588108099, 2, 'bathroom became clogged and flooding occurred after one person showered so we used the nearby sento for most nights due to this.'], [101, 'rev_1319971355872818534', 'rev_1319971355872818534_u2', 'Shibuya', 'accommodation_value', 'mid', 0.3042444845, 3.608488969, 4, 'Overall, the place was worth the price I paid.'], [102, 'rev_1134421461812416620', 'rev_1134421461812416620_u2', 'Minato', 'crowd_level', 'mid', 0.475828547, 3.9516570941, 4, 'The other bedroom is very quiet with a nice view on Mori Tower and a small cemetery.'], [103, 'rev_864141947790593823', 'rev_864141947790593823_u3', 'Minato', 'family_friendliness', 'neg', -0.2733065151, 2.4533869699, 2, 'One of the 2 beds is only 120 cm large, and it worked for us with 2 children.'], [104, 'rev_1159059116319495790', 'rev_1159059116319495790_u2', 'Shibuya', 'nature_parks', 'pos', 0.724215925, 4.4484318499, 5, 'The garden space is beautiful.'], [105, 'rev_1394672604917385142', 'rev_1394672604917385142_u1', 'Setagaya', 'walkability_transit', 'mid', 0.4618433155, 3.923686631, 4, 'Walking distance to Metro and bus stop.'], [106, 'rev_906124838307476655', 'rev_906124838307476655_u6', 'Setagaya', 'accommodation_value', 'neg', -0.5297492783, 1.9405014433, 2, 'The bathroom has a gross smell, like one of the drains has gunk in it or something.'], [107, 'rev_1282282654630380971', 'rev_1282282654630380971_u2', 'Setagaya', 'shopping', 'neg', -0.3079679087, 2.3840641826, 2, "There's a laundromat, convenience store, market, and a mediocre lengthed walk to the nearby station, or a short taxi to Shibuya!"], [108, 'rev_1138043232449957067', 'rev_1138043232449957067_u9', 'Shibuya', 'safety', 'pos', 0.5020618528, 4.0041237056, 4, 'Pretty good safe and quiet stay for maximum of 2 people.'], [109, 'rev_1206022417907885327', 'rev_1206022417907885327_u2', 'Chuo', 'accommodation_value', 'mid', 0.3824829303, 3.7649658606, 4, "The bed is very comfortable and the price is cheap, so I recommend it if you're a heavy sleeper or have ear plugs and a mask."], [110, 'rev_1132950924450976671', 'rev_1132950924450976671_u2', 'Shibuya', 'nature_parks', 'mid', 0.2249284033, 3.4498568065, 3, 'It is short walking distance to the JR line or the metro and you can also walk to Yoyogi park.'], [111, 'rev_1167119530502349974', 'rev_1167119530502349974_u2', 'Shibuya', 'food_dining', 'neg', -0.3442662545, 2.3114674911, 1, 'Lenny of restaurants and convenience store, there is actually a Lawson right at the first block which is 1 mins walk.'], [112, 'rev_1436649791571052369', 'rev_1436649791571052369_u2', 'Shinjuku', 'traditional_cultural', 'neg', -0.5434444062, 1.9131111875, 2, 'There was a noticeable smell, particularly on the stairs, and we found dust on all surfaces, hairs (even on towels), stains, and dirt under the tatami.'], [113, 'rev_1402610837807362217', 'rev_1402610837807362217_u0', 'Toshima', 'food_dining', 'neg', -0.2991790734, 2.4016418532, 1, '2 mins from Station and all restaurants, shops.'], [114, 'rev_1462828889182675244', 'rev_1462828889182675244_u2', 'Toshima', 'shopping', 'pos', 0.6552526383, 4.3105052765, 5, 'Ikebukuro connects a few of the lines and also offer stores like Uniqlo and Don Quijote.'], [115, 'rev_1451226178417104715', 'rev_1451226178417104715_u5', 'Shibuya', 'food_dining', 'mid', -0.0179634653, 2.9640730694, 3, 'Sidenote: if you want to go to the bakery (called Dough-ist), plan accordingly.'], [116, 'rev_1110505772357571491', 'rev_1110505772357571491_u0', 'Taito', 'walkability_transit', 'mid', 0.1390649807, 3.2781299613, 3, 'little bit far from public transport.'], [117, 'rev_42061056', 'rev_42061056_u10', 'Setagaya', 'nature_parks', 'mid', 0.3127301838, 3.6254603676, 4, 'There is a park around the corner from their house which'], [118, 'rev_1185177289139995398', 'rev_1185177289139995398_u2', 'Toshima', 'walkability_transit', 'pos', 0.7940818254, 4.5881636508, 5, 'Both Cherry & Gerry were extremely proactive & responsive, before & during our stay, giving suggestions on weather, airport transportation, etc.'], [119, 'rev_911221925362752082', 'rev_911221925362752082_u1', 'Shinjuku', 'nightlife', 'mid', 0.3711572886, 3.7423145771, 5, 'Next to Korean town and 15 minutes walk to Kabukicho.'], [120, 'rev_1387372991190186116', 'rev_1387372991190186116_u1', 'Setagaya', 'food_dining', 'pos', 0.7961066813, 4.5922133627, 5, 'lots of karaoke places around and great food.'], [121, 'rev_68088447', 'rev_68088447_u1', 'Minato', 'safety', 'mid', 0.3707380537, 3.7414761074, 4, "When I encountered a problem pressing the wrong button of the intercom (as I was responding to a visitor), Akari's husband helped iron out the trouble with security."], [122, 'rev_569254994', 'rev_569254994_u0', 'Toshima', 'family_friendliness', 'pos', 0.7622702119, 4.5245404239, 5, 'We were very impressed by the house, it was a great amount of space for our family to hang out together which is rare to find in Tokyo.'], [123, 'rev_1358391572760349916', 'rev_1358391572760349916_u0', 'Taito', 'food_dining', 'pos', 0.7890755439, 4.5781510877, 5, 'Great location with heaps of restaurants and activities near.'], [124, 'rev_1425826473801325225', 'rev_1425826473801325225_u4', 'Shinjuku', 'safety', 'mid', -0.1690445112, 2.6619109777, 3, 'The stairs leading to the lower level are steep and a bit dangerous.'], [125, 'rev_417361930', 'rev_417361930_u4', 'Shibuya', 'food_dining', 'mid', 0.3475012463, 3.6950024925, 5, 'Last days we realized there are plenty of really nice restaurants in the very same blocks.'], [126, 'rev_1290271375695187996', 'rev_1290271375695187996_u5', 'Itabashi', 'nightlife', 'pos', 0.8217943797, 4.6435887595, 5, 'Shops and Ikebukuro night life was so easy reached.'], [127, 'rev_1457021059054304470', 'rev_1457021059054304470_u3', 'Setagaya', 'shopping', 'mid', -0.0219890289, 2.9560219422, 3, 'It’s only a couple of minute walk to the nearest train station and there are lots of shops nearby.'], [128, 'rev_1155519367013006091', 'rev_1155519367013006091_u2', 'Shibuya', 'traditional_cultural', 'mid', 0.099174805, 3.19834961, 5, 'Meiji shrine a 15-20 minute walk away.'], [129, 'rev_756160443674977013', 'rev_756160443674977013_u9', 'Chiyoda', 'crowd_level', 'neg', -0.2912474498, 2.4175051004, 2, 'far enough from the hustle and bustle.'], [130, 'rev_436895052', 'rev_436895052_u2', 'Itabashi', 'food_dining', 'pos', 0.5033579916, 4.0067159832, 4, 'The food there was great, and I also bought a cheap pair of Nike sweats that I needed (I forgot to pack sweats).'], [131, 'rev_1222825787855295331', 'rev_1222825787855295331_u0', 'Itabashi', 'walkability_transit', 'neg', -0.3320659287, 2.3358681425, 2, 'Very short distance walk to the train station.'], [132, 'rev_556957436', 'rev_556957436_u2', 'Setagaya', 'traditional_cultural', 'neg', -0.4291547691, 2.1416904619, 2, "Unfortunately we didn't get to see her Kimono store"], [133, 'rev_798135497785306492', 'rev_798135497785306492_u1', 'Shibuya', 'family_friendliness', 'neg', -0.4836442892, 2.0327114215, 2, 'Not recommended if you have older family members because the stairs to other rooms are very steep and narrow, it is difficult to move large luggages upstairs.'], [134, 'rev_1343223485589733096', 'rev_1343223485589733096_u0', 'Toshima', 'walkability_transit', 'neg', -0.3019495234, 2.3961009532, 1, 'location is 20 min walking from ikebukuro station or 14 min walking from the next station.'], [135, 'rev_1427258769315839919', 'rev_1427258769315839919_u4', 'Sumida', 'accommodation_value', 'mid', 0.0147314128, 3.0294628255, 3, 'If you’re traveling solo and find it at a discount, maybe it’s worth considering.'], [136, 'rev_1178725259271866408', 'rev_1178725259271866408_u0', 'Shibuya', 'food_dining', 'neg', -0.4311845708, 2.1376308585, 2, 'not far to the train station and restaurants.'], [137, 'rev_59991878', 'rev_59991878_u2', 'Koto', 'traditional_cultural', 'pos', 0.883093045, 4.7661860901, 5, 'They are a wonderful couple, full of life and eager to show us Japanese culture.'], [138, 'rev_1509215956791282434', 'rev_1509215956791282434_u1', 'Shibuya', 'safety', 'pos', 0.6796236155, 4.3592472309, 5, 'Building and room is secure.'], [139, 'rev_889510960021112625', 'rev_889510960021112625_u0', 'Shinjuku', 'safety', 'neg', -0.3149131183, 2.3701737635, 3, 'may be unsafe for girls staying alone'], [140, 'rev_288047506', 'rev_288047506_u1', 'Chiyoda', 'shopping', 'pos', 0.7562198925, 4.5124397851, 5, "Also has a lot of space for you to spread out your suitcases and pack all the souvenirs you've bought!!!"], [141, 'rev_65955114', 'rev_65955114_u2', 'Setagaya', 'nightlife', 'pos', 0.5562539692, 4.1125079384, 4, 'He had lots of snacks and drinks available for us, guidebooks/local recommendations, and tour books we could use.'], [142, 'rev_582929849522766616', 'rev_582929849522766616_u2', 'Setagaya', 'nature_parks', 'pos', 0.7594391853, 4.5188783705, 5, 'Location is great, just right between Komazawa Olympic park and Jiyugaoka station area.'], [143, 'rev_1518643998081320808', 'rev_1518643998081320808_u0', 'Taito', 'traditional_cultural', 'mid', 0.1541531002, 3.3083062004, 3, 'it is located in a nice neighborhood called Asakusa this place is kind of close to the hot area of the neighborhood'], [144, 'rev_1085844351085859059', 'rev_1085844351085859059_u0', 'Minato', 'walkability_transit', 'pos', 0.843977615, 4.68795523, 5, 'a clean and great apartment, immediate replies of the host and the team, and a subway station in walking distance.'], [145, 'rev_1146061264182290917', 'rev_1146061264182290917_u3', 'Setagaya', 'crowd_level', 'neg', -0.5835856156, 1.8328287688, 2, 'Unfortunately we stayed during a construction work outside, which was very (!) noisy in the morning.'], [146, 'rev_1294678158375091790', 'rev_1294678158375091790_u3', 'Shibuya', 'nature_parks', 'pos', 0.5235754494, 4.0471508987, 5, 'Yoyogi station and Shinjuku station are easily'], [147, 'rev_415555411', 'rev_415555411_u5', 'Toshima', 'shopping', 'mid', 0.3905223664, 3.7810447328, 4, 'The closest convenience store and Don Quijote is only a few minutes walk away which was a plus.'], [148, 'rev_1433755138877037140', 'rev_1433755138877037140_u1', 'Minato', 'accommodation_value', 'pos', 0.8060337125, 4.6120674249, 5, 'Perfect for 2-4 people because there’s basically 2 beds in two separate areas.'], [149, 'rev_1313498116644177933', 'rev_1313498116644177933_u5', 'Taito', 'safety', 'mid', 0.1653246768, 3.3306493536, 3, 'neighbourhood feels very safe to walk at night though.'], [150, 'rev_366604246', 'rev_366604246_u3', 'Setagaya', 'safety', 'pos', 0.5038433685, 4.007686737, 4, 'there is a circuit breaker at the door so it’s safe .']]}, 'sentence_llm_labels.csv': {'columns': ['text_id', 'unit_id', 'aspect', 'llm_star', 'aspect_ok', 'item'], 'data': [['rev_1445412511797024861', 'rev_1445412511797024861_u8', 'nature_parks', 5, 0, 1], ['rev_424543807', 'rev_424543807_u1', 'crowd_level', 4, 1, 2], ['rev_1327932107130465257', 'rev_1327932107130465257_u0', 'shopping', 5, 1, 3], ['rev_851046309010178868', 'rev_851046309010178868_u2', 'nightlife', 5, 1, 4], ['rev_408916493708608065', 'rev_408916493708608065_u2', 'nightlife', 2, 0, 5], ['rev_1001125037615310888', 'rev_1001125037615310888_u4', 'family_friendliness', 5, 1, 6], ['rev_1140961843419158123', 'rev_1140961843419158123_u6', 'nature_parks', 5, 0, 7], ['rev_258896118', 'rev_258896118_u3', 'walkability_transit', 5, 1, 8], ['rev_1391747427457310410', 'rev_1391747427457310410_u1', 'nature_parks', 3, 1, 9], ['rev_1277287456640526419', 'rev_1277287456640526419_u5', 'nightlife', 2, 1, 10], ['rev_1195282689740467512', 'rev_1195282689740467512_u5', 'shopping', 2, 0, 11], ['rev_1454083080626832636', 'rev_1454083080626832636_u2', 'accommodation_value', 4, 1, 12], ['rev_1071349361407685311', 'rev_1071349361407685311_u4', 'family_friendliness', 2, 1, 13], ['rev_1462719028990693486', 'rev_1462719028990693486_u3', 'accommodation_value', 5, 1, 14], ['rev_411903047', 'rev_411903047_u1', 'crowd_level', 4, 0, 15], ['rev_610696209', 'rev_610696209_u2', 'food_dining', 5, 1, 16], ['rev_1200454934957976148', 'rev_1200454934957976148_u2', 'safety', 5, 1, 17], ['rev_1127881790732066128', 'rev_1127881790732066128_u2', 'nature_parks', 4, 0, 18], ['rev_1496785907946261117', 'rev_1496785907946261117_u0', 'food_dining', 5, 1, 19], ['rev_1264239712784296799', 'rev_1264239712784296799_u1', 'traditional_cultural', 4, 0, 20], ['rev_1269255602716143816', 'rev_1269255602716143816_u1', 'crowd_level', 5, 1, 21], ['rev_928611293126674642', 'rev_928611293126674642_u1', 'traditional_cultural', 4, 1, 22], ['rev_1420794581404316649', 'rev_1420794581404316649_u3', 'crowd_level', 4, 1, 23], ['rev_1280863120173763217', 'rev_1280863120173763217_u2', 'traditional_cultural', 3, 0, 24], ['rev_1154776177334351603', 'rev_1154776177334351603_u0', 'traditional_cultural', 4, 1, 25], ['rev_927145812488213119', 'rev_927145812488213119_u1', 'walkability_transit', 5, 1, 26], ['rev_1169999054607570278', 'rev_1169999054607570278_u0', 'shopping', 4, 0, 27], ['rev_1426554323534034844', 'rev_1426554323534034844_u0', 'walkability_transit', 3, 0, 28], ['rev_1027905743710674593', 'rev_1027905743710674593_u2', 'food_dining', 4, 1, 29], ['rev_1510570319195436483', 'rev_1510570319195436483_u3', 'family_friendliness', 5, 1, 30], ['rev_920619628020481928', 'rev_920619628020481928_u0', 'crowd_level', 1, 1, 31], ['rev_1004040733153659615', 'rev_1004040733153659615_u5', 'crowd_level', 2, 1, 32], ['rev_1359895280987134993', 'rev_1359895280987134993_u1', 'nature_parks', 4, 1, 33], ['rev_1298940154588535309', 'rev_1298940154588535309_u0', 'accommodation_value', 5, 1, 34], ['rev_435931005', 'rev_435931005_u5', 'family_friendliness', 5, 1, 35], ['rev_1375133700561334129', 'rev_1375133700561334129_u0', 'walkability_transit', 5, 1, 36], ['rev_1017059320510849684', 'rev_1017059320510849684_u1', 'food_dining', 1, 1, 37], ['rev_1320766676131074966', 'rev_1320766676131074966_u0', 'crowd_level', 5, 1, 38], ['rev_875010452556970061', 'rev_875010452556970061_u3', 'accommodation_value', 4, 1, 39], ['rev_1466425943337150609', 'rev_1466425943337150609_u0', 'family_friendliness', 3, 1, 40], ['rev_1170021235498627106', 'rev_1170021235498627106_u0', 'walkability_transit', 4, 1, 41], ['rev_1301165994857073801', 'rev_1301165994857073801_u5', 'nightlife', 5, 1, 42], ['rev_1320679279843647239', 'rev_1320679279843647239_u1', 'nightlife', 5, 1, 43], ['rev_1243199768915338115', 'rev_1243199768915338115_u0', 'crowd_level', 5, 1, 44], ['rev_405376854487786237', 'rev_405376854487786237_u4', 'accommodation_value', 4, 1, 45], ['rev_524926640', 'rev_524926640_u4', 'nightlife', 4, 1, 46], ['rev_1310550993950489142', 'rev_1310550993950489142_u6', 'traditional_cultural', 4, 0, 47], ['rev_1504066050515532729', 'rev_1504066050515532729_u0', 'walkability_transit', 4, 1, 48], ['rev_995274734967458583', 'rev_995274734967458583_u3', 'nightlife', 4, 0, 49], ['rev_1515697653130306003', 'rev_1515697653130306003_u0', 'shopping', 4, 1, 50], ['rev_761250666998437330', 'rev_761250666998437330_u4', 'safety', 2, 1, 51], ['rev_1211267622403076126', 'rev_1211267622403076126_u4', 'shopping', 4, 1, 52], ['rev_1267865037471470984', 'rev_1267865037471470984_u7', 'nature_parks', 2, 0, 53], ['rev_1325025195186697938', 'rev_1325025195186697938_u5', 'accommodation_value', 2, 1, 54], ['rev_1108349269001840682', 'rev_1108349269001840682_u12', 'traditional_cultural', 3, 1, 55], ['rev_1325017912182793925', 'rev_1325017912182793925_u1', 'nature_parks', 4, 0, 56], ['rev_1451139736212101078', 'rev_1451139736212101078_u6', 'crowd_level', 5, 1, 57], ['rev_365921762', 'rev_365921762_u1', 'family_friendliness', 5, 0, 58], ['rev_1293123304138252406', 'rev_1293123304138252406_u2', 'family_friendliness', 5, 1, 59], ['rev_146405510', 'rev_146405510_u9', 'nature_parks', 4, 1, 60], ['rev_985826814722483992', 'rev_985826814722483992_u2', 'traditional_cultural', 3, 0, 61], ['rev_1388185037337847482', 'rev_1388185037337847482_u4', 'crowd_level', 2, 0, 62], ['rev_459923089', 'rev_459923089_u0', 'crowd_level', 5, 1, 63], ['rev_355829807', 'rev_355829807_u2', 'crowd_level', 2, 0, 64], ['rev_1125069156899634321', 'rev_1125069156899634321_u3', 'safety', 4, 1, 65], ['rev_1365621700305684109', 'rev_1365621700305684109_u1', 'family_friendliness', 2, 1, 66], ['rev_1430116897977182922', 'rev_1430116897977182922_u5', 'safety', 2, 1, 67], ['rev_1163459247247861355', 'rev_1163459247247861355_u0', 'shopping', 5, 1, 68], ['rev_733607048481699172', 'rev_733607048481699172_u7', 'food_dining', 1, 0, 69], ['rev_1278654135090364803', 'rev_1278654135090364803_u6', 'safety', 2, 0, 70], ['rev_1125785738679384604', 'rev_1125785738679384604_u1', 'shopping', 4, 1, 71], ['rev_1297507668840582916', 'rev_1297507668840582916_u0', 'family_friendliness', 3, 1, 72], ['rev_846022837929348964', 'rev_846022837929348964_u1', 'walkability_transit', 5, 1, 73], ['rev_1400419537737492385', 'rev_1400419537737492385_u3', 'safety', 2, 0, 74], ['rev_1325104355024170046', 'rev_1325104355024170046_u5', 'nature_parks', 5, 1, 75], ['rev_1116340555329723506', 'rev_1116340555329723506_u5', 'nightlife', 2, 0, 76], ['rev_1406294007212721813', 'rev_1406294007212721813_u0', 'walkability_transit', 5, 1, 77], ['rev_1138790443538468230', 'rev_1138790443538468230_u4', 'food_dining', 3, 1, 78], ['rev_915532807553287696', 'rev_915532807553287696_u7', 'accommodation_value', 2, 1, 79], ['rev_476299609', 'rev_476299609_u4', 'safety', 3, 1, 80], ['rev_347825317', 'rev_347825317_u4', 'shopping', 5, 0, 81], ['rev_1146730717065423206', 'rev_1146730717065423206_u3', 'food_dining', 4, 1, 82], ['rev_1276537256749072171', 'rev_1276537256749072171_u2', 'family_friendliness', 2, 1, 83], ['rev_532488194', 'rev_532488194_u1', 'nightlife', 5, 1, 84], ['rev_168668779', 'rev_168668779_u2', 'nightlife', 3, 0, 85], ['rev_851101449304519196', 'rev_851101449304519196_u0', 'family_friendliness', 4, 1, 86], ['rev_1030846516087900699', 'rev_1030846516087900699_u2', 'shopping', 3, 0, 87], ['rev_1132330329919495661', 'rev_1132330329919495661_u1', 'accommodation_value', 2, 1, 88], ['rev_1443203413951580709', 'rev_1443203413951580709_u0', 'family_friendliness', 3, 1, 89], ['rev_1296044942341524172', 'rev_1296044942341524172_u0', 'nature_parks', 5, 1, 90], ['rev_1183798341527361485', 'rev_1183798341527361485_u2', 'safety', 2, 0, 91], ['rev_1494636174738919112', 'rev_1494636174738919112_u1', 'traditional_cultural', 4, 0, 92], ['rev_1074286153506374363', 'rev_1074286153506374363_u3', 'nightlife', 3, 0, 93], ['rev_472062642', 'rev_472062642_u0', 'traditional_cultural', 5, 1, 94], ['rev_861955699006086768', 'rev_861955699006086768_u1', 'walkability_transit', 3, 1, 95], ['rev_1390309060734801083', 'rev_1390309060734801083_u0', 'nightlife', 5, 0, 96], ['rev_804685091873857822', 'rev_804685091873857822_u6', 'accommodation_value', 2, 1, 97], ['rev_1513536602847253397', 'rev_1513536602847253397_u1', 'accommodation_value', 4, 1, 98], ['rev_1402580285456833501', 'rev_1402580285456833501_u0', 'shopping', 4, 1, 99], ['rev_1355568006630800560', 'rev_1355568006630800560_u1', 'traditional_cultural', 2, 0, 100], ['rev_1319971355872818534', 'rev_1319971355872818534_u2', 'accommodation_value', 4, 1, 101], ['rev_1134421461812416620', 'rev_1134421461812416620_u2', 'crowd_level', 4, 1, 102], ['rev_864141947790593823', 'rev_864141947790593823_u3', 'family_friendliness', 3, 1, 103], ['rev_1159059116319495790', 'rev_1159059116319495790_u2', 'nature_parks', 5, 1, 104], ['rev_1394672604917385142', 'rev_1394672604917385142_u1', 'walkability_transit', 4, 1, 105], ['rev_906124838307476655', 'rev_906124838307476655_u6', 'accommodation_value', 2, 1, 106], ['rev_1282282654630380971', 'rev_1282282654630380971_u2', 'shopping', 3, 0, 107], ['rev_1138043232449957067', 'rev_1138043232449957067_u9', 'safety', 4, 1, 108], ['rev_1206022417907885327', 'rev_1206022417907885327_u2', 'accommodation_value', 3, 1, 109], ['rev_1132950924450976671', 'rev_1132950924450976671_u2', 'nature_parks', 4, 1, 110], ['rev_1167119530502349974', 'rev_1167119530502349974_u2', 'food_dining', 4, 1, 111], ['rev_1436649791571052369', 'rev_1436649791571052369_u2', 'traditional_cultural', 1, 0, 112], ['rev_1402610837807362217', 'rev_1402610837807362217_u0', 'food_dining', 4, 1, 113], ['rev_1462828889182675244', 'rev_1462828889182675244_u2', 'shopping', 4, 1, 114], ['rev_1451226178417104715', 'rev_1451226178417104715_u5', 'food_dining', 3, 1, 115], ['rev_1110505772357571491', 'rev_1110505772357571491_u0', 'walkability_transit', 2, 1, 116], ['rev_42061056', 'rev_42061056_u10', 'nature_parks', 4, 1, 117], ['rev_1185177289139995398', 'rev_1185177289139995398_u2', 'walkability_transit', 5, 0, 118], ['rev_911221925362752082', 'rev_911221925362752082_u1', 'nightlife', 3, 1, 119], ['rev_1387372991190186116', 'rev_1387372991190186116_u1', 'food_dining', 5, 1, 120], ['rev_68088447', 'rev_68088447_u1', 'safety', 4, 0, 121], ['rev_569254994', 'rev_569254994_u0', 'family_friendliness', 5, 1, 122], ['rev_1358391572760349916', 'rev_1358391572760349916_u0', 'food_dining', 5, 1, 123], ['rev_1425826473801325225', 'rev_1425826473801325225_u4', 'safety', 2, 1, 124], ['rev_417361930', 'rev_417361930_u4', 'food_dining', 5, 1, 125], ['rev_1290271375695187996', 'rev_1290271375695187996_u5', 'nightlife', 4, 1, 126], ['rev_1457021059054304470', 'rev_1457021059054304470_u3', 'shopping', 4, 1, 127], ['rev_1155519367013006091', 'rev_1155519367013006091_u2', 'traditional_cultural', 3, 1, 128], ['rev_756160443674977013', 'rev_756160443674977013_u9', 'crowd_level', 4, 1, 129], ['rev_436895052', 'rev_436895052_u2', 'food_dining', 4, 1, 130], ['rev_1222825787855295331', 'rev_1222825787855295331_u0', 'walkability_transit', 4, 1, 131], ['rev_556957436', 'rev_556957436_u2', 'traditional_cultural', 3, 1, 132], ['rev_798135497785306492', 'rev_798135497785306492_u1', 'family_friendliness', 2, 1, 133], ['rev_1343223485589733096', 'rev_1343223485589733096_u0', 'walkability_transit', 3, 1, 134], ['rev_1427258769315839919', 'rev_1427258769315839919_u4', 'accommodation_value', 2, 1, 135], ['rev_1178725259271866408', 'rev_1178725259271866408_u0', 'food_dining', 4, 1, 136], ['rev_59991878', 'rev_59991878_u2', 'traditional_cultural', 5, 1, 137], ['rev_1509215956791282434', 'rev_1509215956791282434_u1', 'safety', 4, 1, 138], ['rev_889510960021112625', 'rev_889510960021112625_u0', 'safety', 2, 1, 139], ['rev_288047506', 'rev_288047506_u1', 'shopping', 4, 0, 140], ['rev_65955114', 'rev_65955114_u2', 'nightlife', 5, 0, 141], ['rev_582929849522766616', 'rev_582929849522766616_u2', 'nature_parks', 4, 1, 142], ['rev_1518643998081320808', 'rev_1518643998081320808_u0', 'traditional_cultural', 4, 0, 143], ['rev_1085844351085859059', 'rev_1085844351085859059_u0', 'walkability_transit', 5, 1, 144], ['rev_1146061264182290917', 'rev_1146061264182290917_u3', 'crowd_level', 2, 1, 145], ['rev_1294678158375091790', 'rev_1294678158375091790_u3', 'nature_parks', 4, 0, 146], ['rev_415555411', 'rev_415555411_u5', 'shopping', 4, 1, 147], ['rev_1433755138877037140', 'rev_1433755138877037140_u1', 'accommodation_value', 4, 1, 148], ['rev_1313498116644177933', 'rev_1313498116644177933_u5', 'safety', 5, 1, 149], ['rev_366604246', 'rev_366604246_u3', 'safety', 4, 0, 150]]}, 'sentence_audit_completed.csv': {'columns': ['audit_id', 'item', 'rakim_star', 'rakim_aspect_ok'], 'data': [[1, 142, 5, 1], [2, 88, 2, 1], [3, 96, 5, 1], [4, 124, 1, 1], [5, 140, 5, 1], [6, 123, 5, 1], [7, 36, 5, 1], [8, 19, 5, 1], [9, 47, 5, 1], [10, 82, 4, 1], [11, 138, 4, 1], [12, 133, 1, 1], [13, 24, 5, 1], [14, 87, 3, 1], [15, 63, 5, 1], [16, 30, 5, 1], [17, 148, 5, 1], [18, 54, 2, 1], [19, 2, 4, 1], [20, 126, 5, 1], [21, 145, 2, 1], [22, 33, 5, 1], [23, 18, 5, 1], [24, 118, 5, 1], [25, 49, 4, 0], [26, 105, 4, 1], [27, 3, 4, 1], [28, 74, 2, 1], [29, 86, 5, 1], [30, 143, 5, 1]]}, 'sentence_audit_key.csv': {'columns': ['audit_id', 'item', 'llm_star', 'aspect_ok'], 'data': [[1, 142, 4, 1], [2, 88, 2, 1], [3, 96, 5, 0], [4, 124, 2, 1], [5, 140, 4, 0], [6, 123, 5, 1], [7, 36, 5, 1], [8, 19, 5, 1], [9, 47, 4, 0], [10, 82, 4, 1], [11, 138, 4, 1], [12, 133, 2, 1], [13, 24, 3, 0], [14, 87, 3, 0], [15, 63, 5, 1], [16, 30, 5, 1], [17, 148, 4, 1], [18, 54, 2, 1], [19, 2, 4, 1], [20, 126, 4, 1], [21, 145, 2, 1], [22, 33, 4, 1], [23, 18, 4, 0], [24, 118, 5, 0], [25, 49, 4, 0], [26, 105, 4, 1], [27, 3, 5, 1], [28, 74, 2, 0], [29, 86, 4, 1], [30, 143, 4, 0]]}}
LABEL_DATA['labelset_design.csv'] = {'columns': ['item', 'text_id', 'language', 'stratum_group'], 'data': [[1, 'rev_1429473266791308502', 'EN', 'EN_pos'], [2, 'rev_1387453310181444179', 'ES', 'ES'], [3, 'rev_1448262032681435961', 'EN', 'EN_pos'], [4, 'rev_965571036637552106', 'EN', 'EN_neg'], [5, 'rev_1495394358795019939', 'EN', 'EN_pos'], [6, 'rev_1333011500123144850', 'EN', 'EN_pos'], [7, 'rev_830035203123716245', 'EN', 'EN_pos'], [8, 'rev_1472145225030786581', 'EN', 'EN_neu'], [9, 'rev_832185334673677110', 'EN', 'EN_neg'], [10, 'rev_901034772417040874', 'EN', 'EN_pos'], [11, 'rev_1509857359102373418', 'EN', 'EN_pos'], [12, 'rev_1359867070125223999', 'EN', 'EN_neg'], [13, 'rev_1485221376181845400', 'EN', 'EN_pos'], [14, 'rev_1272852640922476587', 'EN', 'EN_neg'], [15, 'rev_1007666840140070063', 'EN', 'EN_pos'], [16, 'rev_1319222046673152692', 'EN', 'EN_pos'], [17, 'rev_1327258575098907891', 'IT', 'IT'], [18, 'rev_1226534741086546914', 'EN', 'EN_pos'], [19, 'rev_1312012572205045504', 'EN', 'EN_pos'], [20, 'rev_1064910430687620654', 'EN', 'EN_pos'], [21, 'rev_1059861080218801271', 'EN', 'EN_neg'], [22, 'rev_576243351', 'EN', 'EN_neu'], [23, 'rev_1280119957707532994', 'FR', 'FR'], [24, 'rev_1359181506490833113', 'FR', 'FR'], [25, 'rev_945300575718847318', 'EN', 'EN_neu'], [26, 'rev_592448004', 'EN', 'EN_neg'], [27, 'rev_1282324013874070291', 'DE', 'DE'], [28, 'rev_1140257342620476265', 'NL', 'NL'], [29, 'rev_1433805120509044357', 'EN', 'EN_pos'], [30, 'rev_1224348838941725836', 'EN', 'EN_pos'], [31, 'rev_1293144460499478447', 'EN', 'EN_neg'], [32, 'rev_285882190', 'EN', 'EN_pos'], [33, 'rev_1507027317023765584', 'EN', 'EN_neg'], [34, 'rev_509673951', 'EN', 'EN_neg'], [35, 'rev_914880844889218235', 'EN', 'EN_pos'], [36, 'rev_1134472875673267074', 'EN', 'EN_neu'], [37, 'rev_1356967687870636730', 'EN', 'EN_neg'], [38, 'rev_1324286134590078416', 'EN', 'EN_neu'], [39, 'rev_250672134', 'EN', 'EN_neg'], [40, 'rev_784439119232170475', 'EN', 'EN_neg'], [41, 'rev_1069165378098819936', 'EN', 'EN_neu'], [42, 'rev_911171087899491017', 'EN', 'EN_neg'], [43, 'rev_877209054967380759', 'DE', 'DE'], [44, 'rev_361901298', 'EN', 'EN_neg'], [45, 'rev_832975618292857350', 'EN', 'EN_neg'], [46, 'rev_1483767115036113552', 'EN', 'EN_neg'], [47, 'rev_478402048', 'EN', 'EN_pos'], [48, 'rev_1512804435440604544', 'EN', 'EN_pos'], [49, 'rev_764077652264016649', 'EN', 'EN_neu'], [50, 'rev_1504123766337180688', 'EN', 'EN_pos'], [51, 'rev_1479433960787461072', 'EN', 'EN_pos'], [52, 'rev_769876279548066366', 'EN', 'EN_pos'], [53, 'rev_1072088668097572157', 'EN', 'EN_pos'], [54, 'rev_919241125832137082', 'EN', 'EN_neg'], [55, 'rev_1263430584524376355', 'NL', 'NL'], [56, 'rev_959080350951827216', 'DE', 'DE'], [57, 'rev_248193674', 'EN', 'EN_neu'], [58, 'rev_581147094', 'EN', 'EN_neu'], [59, 'rev_245363268', 'EN', 'EN_pos'], [60, 'rev_880798782949216224', 'EN', 'EN_neg'], [61, 'rev_1019200171854663731', 'EN', 'EN_pos'], [62, 'rev_1038051504343802824', 'EN', 'EN_neg'], [63, 'rev_1280100457496192448', 'EN', 'EN_neg'], [64, 'rev_940967091058750016', 'EN', 'EN_pos'], [65, 'rev_430188640', 'EN', 'EN_pos'], [66, 'rev_945293379104742479', 'EN', 'EN_neg'], [67, 'rev_213767749', 'EN', 'EN_neg'], [68, 'rev_874927900273512402', 'FR', 'FR'], [69, 'rev_763365535233263226', 'EN', 'EN_neg'], [70, 'rev_599122962', 'EN', 'EN_pos'], [71, 'rev_959832506047548680', 'EN', 'EN_neu'], [72, 'rev_1014846913744799122', 'EN', 'EN_pos'], [73, 'rev_1437394085483802026', 'EN', 'EN_pos'], [74, 'rev_526463245', 'EN', 'EN_pos'], [75, 'rev_885152248107705180', 'EN', 'EN_pos'], [76, 'rev_1248212977620322619', 'DE', 'DE'], [77, 'rev_1206241019432712785', 'EN', 'EN_neu'], [78, 'rev_1164217866433159153', 'EN', 'EN_neu'], [79, 'rev_1419351767379669266', 'EN', 'EN_neg'], [80, 'rev_281929510', 'IT', 'IT'], [81, 'rev_997505387143633713', 'EN', 'EN_pos'], [82, 'rev_1314249807099695533', 'EN', 'EN_pos'], [83, 'rev_900295551399227462', 'EN', 'EN_neg'], [84, 'rev_726362659275655635', 'EN', 'EN_pos'], [85, 'rev_1098231763048309947', 'EN', 'EN_neg'], [86, 'rev_859799359932449884', 'EN', 'EN_neu'], [87, 'rev_1320664221854670100', 'EN', 'EN_neu'], [88, 'rev_1114135809859986150', 'EN', 'EN_neg'], [89, 'rev_887265940999661288', 'EN', 'EN_neg'], [90, 'rev_1433060226998600121', 'EN', 'EN_neg'], [91, 'rev_1046731780300682874', 'EN', 'EN_pos'], [92, 'rev_1193914640694510035', 'EN', 'EN_neu'], [93, 'rev_1272127621916199876', 'EN', 'EN_neg'], [94, 'rev_1127905408222405914', 'EN', 'EN_pos'], [95, 'rev_812664609041163538', 'EN', 'EN_neg'], [96, 'rev_1511347227184016075', 'EN', 'EN_neg'], [97, 'rev_430567650', 'EN', 'EN_pos'], [98, 'rev_116844695', 'EN', 'EN_pos'], [99, 'rev_906850239656437945', 'DE', 'DE'], [100, 'rev_1127243250038109367', 'EN', 'EN_neg'], [101, 'rev_1388121648368464213', 'EN', 'EN_pos'], [102, 'rev_354156816', 'EN', 'EN_neg'], [103, 'rev_1285892356206409787', 'EN', 'EN_neg'], [104, 'rev_921399347064812900', 'EN', 'EN_pos'], [105, 'rev_964851016291133705', 'ES', 'ES'], [106, 'rev_1265663906516028177', 'EN', 'EN_pos'], [107, 'rev_1199016396710733274', 'EN', 'EN_pos'], [108, 'rev_1488085595880084222', 'EN', 'EN_neg'], [109, 'rev_577032302', 'EN', 'EN_pos'], [110, 'rev_825737954238206680', 'EN', 'EN_neu'], [111, 'rev_1219254151831440828', 'EN', 'EN_pos'], [112, 'rev_1254770813724062692', 'ES', 'ES'], [113, 'rev_1077884207792128766', 'EN', 'EN_neu'], [114, 'rev_532511577', 'FR', 'FR'], [115, 'rev_1505509233404209610', 'FR', 'FR'], [116, 'rev_860473025907662025', 'EN', 'EN_neu'], [117, 'rev_1159045976270081655', 'EN', 'EN_pos'], [118, 'rev_1499007335743103848', 'IT', 'IT'], [119, 'rev_637256915012393259', 'EN', 'EN_pos'], [120, 'rev_843101687495201724', 'EN', 'EN_pos'], [121, 'rev_903999093305198299', 'EN', 'EN_neg'], [122, 'rev_1253277870505508738', 'EN', 'EN_pos'], [123, 'rev_1429471521104256003', 'EN', 'EN_pos'], [124, 'rev_820646957540413100', 'EN', 'EN_pos'], [125, 'rev_941684291050112274', 'EN', 'EN_pos'], [126, 'rev_851798277821841545', 'FR', 'FR'], [127, 'rev_1049689062482662756', 'EN', 'EN_neg'], [128, 'rev_1501204309830184134', 'EN', 'EN_pos'], [129, 'rev_663488509', 'EN', 'EN_pos'], [130, 'rev_1294655530950541633', 'DE', 'DE'], [131, 'rev_1342475490330496237', 'EN', 'EN_pos'], [132, 'rev_972065033959016043', 'EN', 'EN_neu'], [133, 'rev_1219232332128555758', 'EN', 'EN_pos'], [134, 'rev_1357713901849812259', 'EN', 'EN_neu'], [135, 'rev_547373904', 'DE', 'DE'], [136, 'rev_1453380780687012969', 'ES', 'ES'], [137, 'rev_1092392199058813593', 'EN', 'EN_pos'], [138, 'rev_1343132068869163823', 'EN', 'EN_pos'], [139, 'rev_575672336', 'EN', 'EN_pos'], [140, 'rev_856152977381180084', 'EN', 'EN_neg'], [141, 'rev_1043836497271619091', 'EN', 'EN_neu'], [142, 'rev_1399054748770164207', 'EN', 'EN_pos'], [143, 'rev_1389519579723373928', 'EN', 'EN_pos'], [144, 'rev_1175793829844508814', 'EN', 'EN_pos'], [145, 'rev_1014108124202055097', 'EN', 'EN_pos'], [146, 'rev_348948357', 'ES', 'ES'], [147, 'rev_1093864617594075089', 'EN', 'EN_pos'], [148, 'rev_1212686246099990362', 'EN', 'EN_neg'], [149, 'rev_1138821989485575275', 'EN', 'EN_neg'], [150, 'rev_1061922233334708922', 'EN', 'EN_neu'], [151, 'rev_1401915822932674177', 'ES', 'ES'], [152, 'rev_1422185665518155075', 'EN', 'EN_neu'], [153, 'rev_1163472549243296031', 'EN', 'EN_pos'], [154, 'rev_900321599230999530', 'EN', 'EN_pos'], [155, 'rev_1201890277081798794', 'DE', 'DE'], [156, 'rev_301866716', 'FR', 'FR'], [157, 'rev_1270773233061367435', 'EN', 'EN_pos'], [158, 'rev_589415210', 'EN', 'EN_neg'], [159, 'rev_1104789262540891215', 'EN', 'EN_pos'], [160, 'rev_1166299954357851404', 'EN', 'EN_neu'], [161, 'rev_197257320', 'EN', 'EN_pos'], [162, 'rev_268387734', 'EN', 'EN_pos'], [163, 'rev_1359891428761851735', 'EN', 'EN_neg'], [164, 'rev_640330351', 'EN', 'EN_neg'], [165, 'rev_821343007007481845', 'EN', 'EN_neg'], [166, 'rev_1274314033452316235', 'EN', 'EN_pos'], [167, 'rev_922065412044532336', 'DE', 'DE'], [168, 'rev_1254016433208657072', 'FR', 'FR'], [169, 'rev_1124288568108357320', 'EN', 'EN_pos'], [170, 'rev_362239765', 'EN', 'EN_neg'], [171, 'rev_867691850872670446', 'EN', 'EN_neu'], [172, 'rev_1309107752795352400', 'EN', 'EN_pos'], [173, 'rev_550590309', 'FR', 'FR'], [174, 'rev_1372942839640491983', 'EN', 'EN_neg'], [175, 'rev_1098267715077665526', 'EN', 'EN_neu'], [176, 'rev_897419125072974272', 'FR', 'FR'], [177, 'rev_1315627821342632660', 'EN', 'EN_pos'], [178, 'rev_714477929', 'EN', 'EN_neg'], [179, 'rev_1405479705896214666', 'IT', 'IT'], [180, 'rev_203040330', 'EN', 'EN_pos'], [181, 'rev_203340971', 'EN', 'EN_pos'], [182, 'rev_1330135671029691195', 'EN', 'EN_neg'], [183, 'rev_1279362658904188975', 'ES', 'ES'], [184, 'rev_1125772803425665406', 'ES', 'ES'], [185, 'rev_582415188', 'EN', 'EN_neu'], [186, 'rev_175874488', 'FR', 'FR'], [187, 'rev_1009780389939701932', 'FR', 'FR'], [188, 'rev_1003956293492486850', 'EN', 'EN_pos'], [189, 'rev_185123629', 'EN', 'EN_neu'], [190, 'rev_1333796864561921326', 'EN', 'EN_neg'], [191, 'rev_432763377', 'EN', 'EN_neg'], [192, 'rev_1421446423117439361', 'EN', 'EN_pos'], [193, 'rev_948877993539361656', 'EN', 'EN_pos'], [194, 'rev_1427285069477928458', 'EN', 'EN_pos'], [195, 'rev_1512733773884508852', 'FR', 'FR'], [196, 'rev_1422146746720055502', 'EN', 'EN_neg'], [197, 'rev_1143931122580120750', 'NL', 'NL'], [198, 'rev_300336567', 'ES', 'ES'], [199, 'rev_1165612202389007527', 'EN', 'EN_neg'], [200, 'rev_977152308606267635', 'EN', 'EN_pos'], [201, 'rev_1509845578064442536', 'EN', 'EN_neg'], [202, 'rev_1135199974005934467', 'EN', 'EN_pos'], [203, 'rev_966365763671462574', 'EN', 'EN_neg'], [204, 'rev_1106187287317832260', 'EN', 'EN_pos'], [205, 'rev_1514280883356957802', 'EN', 'EN_pos'], [206, 'rev_792329821863027178', 'EN', 'EN_pos'], [207, 'rev_1146822505125227711', 'DE', 'DE'], [208, 'rev_1153250807498734058', 'EN', 'EN_pos'], [209, 'rev_781531502795568263', 'EN', 'EN_neg'], [210, 'rev_1261325002095305166', 'EN', 'EN_pos'], [211, 'rev_1045331855432107501', 'EN', 'EN_pos'], [212, 'rev_1228680853667393969', 'IT', 'IT'], [213, 'rev_1273647043582207053', 'EN', 'EN_neg'], [214, 'rev_1127248741603675409', 'EN', 'EN_neg'], [215, 'rev_1501151330442154074', 'ES', 'ES'], [216, 'rev_774203877722756225', 'EN', 'EN_neg'], [217, 'rev_851120896747016212', 'EN', 'EN_neg'], [218, 'rev_948239315261260687', 'DE', 'DE'], [219, 'rev_559874591', 'EN', 'EN_pos'], [220, 'rev_1297479987146544545', 'EN', 'EN_neg'], [221, 'rev_505397060203035305', 'EN', 'EN_pos'], [222, 'rev_1191005173852151567', 'DE', 'DE'], [223, 'rev_412554312', 'EN', 'EN_pos'], [224, 'rev_159375464', 'EN', 'EN_pos'], [225, 'rev_822793222069495516', 'EN', 'EN_neg'], [226, 'rev_1251122836056957322', 'ES', 'ES'], [227, 'rev_858311244721139270', 'EN', 'EN_neu'], [228, 'rev_1186687706303236491', 'EN', 'EN_neu'], [229, 'rev_1389582797677468048', 'EN', 'EN_pos'], [230, 'rev_1335237246332354679', 'EN', 'EN_pos'], [231, 'rev_1254036152890945244', 'EN', 'EN_neg'], [232, 'rev_893805304627631322', 'FR', 'FR'], [233, 'rev_1496140993631139249', 'IT', 'IT'], [234, 'rev_1243904690152696527', 'FR', 'FR'], [235, 'rev_1264207943721504208', 'EN', 'EN_pos'], [236, 'rev_1043141172638989899', 'EN', 'EN_neg'], [237, 'rev_1271478172528901764', 'EN', 'EN_neg'], [238, 'rev_1319972164182702289', 'EN', 'EN_neu'], [239, 'rev_1245400643482280391', 'ES', 'ES'], [240, 'rev_489470433', 'EN', 'EN_pos'], [241, 'rev_1364148766392436782', 'ES', 'ES'], [242, 'rev_1427931994806465026', 'EN', 'EN_pos'], [243, 'rev_971431479060507733', 'EN', 'EN_pos'], [244, 'rev_552991262', 'EN', 'EN_neg'], [245, 'rev_1264869382874756182', 'EN', 'EN_pos'], [246, 'rev_140117240', 'EN', 'EN_pos'], [247, 'rev_1218551070837421606', 'EN', 'EN_pos'], [248, 'rev_793055260723507245', 'EN', 'EN_pos'], [249, 'rev_1159797860044223091', 'FR', 'FR'], [250, 'rev_1332328260010907493', 'EN', 'EN_neg'], [251, 'rev_620815212', 'EN', 'EN_neu'], [252, 'rev_1517803183769985137', 'EN', 'EN_neg'], [253, 'rev_1066321799411578207', 'EN', 'EN_pos'], [254, 'rev_1401117185115331352', 'EN', 'EN_neg'], [255, 'rev_918417868422594349', 'EN', 'EN_pos'], [256, 'rev_1478773421753396284', 'EN', 'EN_neu'], [257, 'rev_575668676', 'EN', 'EN_neg'], [258, 'rev_790152894610968893', 'EN', 'EN_neu'], [259, 'rev_1347541566885077604', 'EN', 'EN_pos'], [260, 'rev_1382321496029351103', 'EN', 'EN_neg'], [261, 'rev_1350471012480046670', 'EN', 'EN_neu'], [262, 'rev_829297121216897453', 'EN', 'EN_neg'], [263, 'rev_1494656699655397859', 'FR', 'FR'], [264, 'rev_1009127579614789075', 'EN', 'EN_neg'], [265, 'rev_1067108119474751951', 'EN', 'EN_neu'], [266, 'rev_1208401704813357871', 'EN', 'EN_pos'], [267, 'rev_1134416297828810554', 'NL', 'NL'], [268, 'rev_1166391001485642259', 'EN', 'EN_pos'], [269, 'rev_972147031287505673', 'DE', 'DE'], [270, 'rev_1222834073213078325', 'FR', 'FR'], [271, 'rev_1396053784294733070', 'EN', 'EN_neg'], [272, 'rev_1057648407526245743', 'EN', 'EN_neg'], [273, 'rev_748789914', 'EN', 'EN_pos'], [274, 'rev_439885843', 'EN', 'EN_pos'], [275, 'rev_1082304102869818230', 'NL', 'NL'], [276, 'rev_850363899576158986', 'EN', 'EN_pos'], [277, 'rev_1305492497097836105', 'EN', 'EN_pos'], [278, 'rev_536085617', 'EN', 'EN_neg'], [279, 'rev_1381653713504494752', 'EN', 'EN_neg'], [280, 'rev_1299699291336300448', 'EN', 'EN_pos'], [281, 'rev_1234507499056318928', 'FR', 'FR'], [282, 'rev_1102601859175813910', 'EN', 'EN_neg'], [283, 'rev_576624617', 'EN', 'EN_pos'], [284, 'rev_1498271616809863337', 'DE', 'DE'], [285, 'rev_1499718257699972910', 'EN', 'EN_pos'], [286, 'rev_1390975886556804468', 'FR', 'FR'], [287, 'rev_621894147', 'EN', 'EN_pos'], [288, 'rev_1156245743912689833', 'EN', 'EN_pos'], [289, 'rev_407716909', 'EN', 'EN_pos'], [290, 'rev_494218116', 'EN', 'EN_neu'], [291, 'rev_1346787622847839718', 'EN', 'EN_neu'], [292, 'rev_1172868638621461143', 'EN', 'EN_pos'], [293, 'rev_999700358474680899', 'EN', 'EN_pos'], [294, 'rev_1469244046529886801', 'NL', 'NL'], [295, 'rev_858992554430900616', 'EN', 'EN_neg'], [296, 'rev_195344369', 'EN', 'EN_pos'], [297, 'rev_167789453', 'EN', 'EN_neg'], [298, 'rev_896000065623840247', 'EN', 'EN_neu'], [299, 'rev_1319247704371649832', 'ES', 'ES'], [300, 'rev_872038659461276880', 'EN', 'EN_neu']]}
LABEL_DATA['negative_quote_check.csv'] = {'columns': ['ward', 'aspect', 'text_id', 'text', 'verdict', 'note'], 'data': [['Shinjuku', 'food_dining', 'rev_354156816', "Do not try the Italian Restaurant downstairs, both the salad and the pizza was horrible, and the servers' attitude was pretty bad.", 'keep', ''], ['Shinjuku', 'family_friendliness', 'rev_559881723', 'They also took our garbage because we asked about where to dispose of our children’s diapers.', 'not_negative', 'positive or neutral text misread as negative'], ['Shinjuku', 'walkability_transit', 'rev_695999426542390270', 'One train stop away from Shinjuku.', 'not_negative', 'positive or neutral text misread as negative'], ['Shinjuku', 'walkability_transit', 'rev_1267755674790683207', 'Really bad beds, not close to stations, not worthy.', 'keep', ''], ['Shinjuku', 'safety', 'rev_1457702044414135661', 'We had a very disappointing and unsafe experience at this property.', 'wrong_aspect_or_not_about_area', 'safety of the property, not the area'], ['Shinjuku', 'accommodation_value', 'rev_1069239575425742661', '1 bathroom with both shower and toilet in the same room is a joke.', 'keep', ''], ['Shinjuku', 'accommodation_value', 'rev_984441679314460611', 'The apartment was absolutely spotless and every detail was looked after.', 'not_negative', 'positive or neutral text misread as negative'], ['Shinjuku', 'crowd_level', 'rev_1437148465894796958', 'Stay away from this place they have BEDBUGS BADLY AND THE BEDS ARE OLD CREAKING AND NOISY.', 'wrong_aspect_or_not_about_area', 'bed noise'], ['Shinjuku', 'crowd_level', 'rev_329036984', 'I saw one reviewer said the streets were busy in a negative way, and I completely disagree!', 'not_negative', 'positive or neutral text misread as negative'], ['Shibuya', 'food_dining', 'rev_864826588180798833', 'Fall out the front door and countless places to eat, drink and shop.', 'not_negative', 'positive or neutral text misread as negative'], ['Shibuya', 'nightlife', 'rev_1156181665402981276', 'The building has a bad smell, the toilet is not functioning, the neighborhood is noisy (expected as it’s a very central location surrounded with bars, etc).', 'keep', ''], ['Shibuya', 'shopping', 'rev_1458460538372327498', 'Shopping for another place on Air Bnb is a waste of time.', 'wrong_aspect_or_not_about_area', '"shopping for another place"'], ['Shibuya', 'family_friendliness', 'rev_1407736140729960097', 'Extremely noisy place even in midnight (not recommended for families with elderly or kids) .', 'keep', ''], ['Shibuya', 'family_friendliness', 'rev_1021361590797008256', 'Stayed here for 6 days with my family, total of 8 people including one child.', 'not_negative', 'positive or neutral text misread as negative'], ['Shibuya', 'walkability_transit', 'rev_329929229', 'I spent no money on any form of transportation service besides getting to and from the airport.', 'not_negative', 'positive or neutral text misread as negative'], ['Shibuya', 'walkability_transit', 'rev_1380164087556061163', 'One train stop away from the Shibuya stop.', 'not_negative', 'positive or neutral text misread as negative'], ['Shibuya', 'safety', 'rev_1314963768374987154', 'I am beyond shocked and feel unsafe.', 'wrong_aspect_or_not_about_area', 'unclear what is unsafe'], ['Shibuya', 'safety', 'rev_1388084310755601314', 'Upstairs sinks not working & toilet not secure and cannot shut the door for privacy.', 'wrong_aspect_or_not_about_area', 'bathroom privacy'], ['Shibuya', 'accommodation_value', 'rev_701746809937695410', 'Was not clean, hair was on the floor, the glasses were dirty.', 'keep', ''], ['Shibuya', 'accommodation_value', 'rev_1275114885506418132', "The reason, why i'm taking away one star are the beds.", 'keep', ''], ['Shibuya', 'crowd_level', 'rev_1121352933998795765', "All in all, I wouldn't book the apartment again, it's very noisy and", 'keep', ''], ['Shibuya', 'crowd_level', 'rev_1056864667211362060', 'The noise level in the room was horrible, clinking dishes and loud laughter from downstairs sounding like it was in the room with us.', 'keep', ''], ['Taito', 'food_dining', 'rev_866262336671096217', 'the sushi we had was not up to standard, tbh the worst we have in Japan.', 'keep', ''], ['Taito', 'food_dining', 'rev_782262394876646175', 'in 5 minutes, bars and restaurants abound.', 'not_negative', 'positive or neutral text misread as negative'], ['Taito', 'nightlife', 'rev_782262394876646175', 'in 5 minutes, bars and restaurants abound.', 'not_negative', 'positive or neutral text misread as negative'], ['Taito', 'traditional_cultural', 'rev_1254822498898903482', 'Wasted no time walking in the area and visiting the temple and places around it.', 'not_negative', 'positive or neutral text misread as negative'], ['Taito', 'traditional_cultural', 'rev_259885242', 'Just a stone’s throw away from Sensoji Temple (literally behind)!', 'not_negative', 'positive or neutral text misread as negative'], ['Taito', 'family_friendliness', 'rev_921367836437675626', 'One child had asthma, thus he had really bad reaction to the unsanitary environment.', 'wrong_aspect_or_not_about_area', 'cleanliness, child mentioned in passing'], ['Taito', 'walkability_transit', 'rev_1012013039936104487', 'No accessible entry to roll heavy suitcase in, only stairs.', 'wrong_aspect_or_not_about_area', 'building access (stairs), not transit'], ['Taito', 'walkability_transit', 'rev_891657453801265765', 'Near bus stop, a stone throw away.', 'not_negative', 'positive or neutral text misread as negative'], ['Taito', 'safety', 'rev_1508492675817698688', 'We could not open it as it was totally unsafe.', 'wrong_aspect_or_not_about_area', 'object, not area safety'], ['Taito', 'safety', 'rev_1209156397630751780', 'Very terrible experience, Our privacy and belongings are NOT SECURE.', 'wrong_aspect_or_not_about_area', 'belongings at the property'], ['Taito', 'accommodation_value', 'rev_884436596974496542', 'Too bad the shower and sink were completely dirty when we arrived!', 'keep', ''], ['Taito', 'accommodation_value', 'rev_1229447169636290621', 'Towels were replenished, beds were made, trash was taken out, and things were wiped down.', 'not_negative', 'positive or neutral text misread as negative'], ['Taito', 'crowd_level', 'rev_930794634133526093', 'We literally listen to the noise from next door and unable to sleep and was waken by the traffic at 7am.', 'keep', ''], ['Taito', 'crowd_level', 'rev_1413453302087971450', 'There was a horrible, rude girl who stayed in the bunk next to me who came home late and made a lot of noise for at least 2 hours.', 'wrong_aspect_or_not_about_area', 'another guest, not the area'], ['Chuo', 'food_dining', 'rev_1273668211042449942', 'left to get food and when we got back the code did not work.', 'wrong_aspect_or_not_about_area', 'door code'], ['Chuo', 'food_dining', 'rev_1168575080333365268', 'We were turned away even at opening and restaurant was empty at 6-owner says “full”.', 'keep', ''], ['Chuo', 'nature_parks', 'rev_1454055124169743394', 'And garbage is split between, flammable, non- flammable, and recycling that you have to bring out to the park by 8 am on different days of the week.', 'wrong_aspect_or_not_about_area', 'garbage rules'], ['Chuo', 'walkability_transit', 'rev_846731339305572524', 'We can go straight using only one train no transfer.', 'not_negative', 'positive or neutral text misread as negative'], ['Chuo', 'walkability_transit', 'rev_1422898853966854668', 'This was completely false there were no useful stations nearby for our JR rail passes so we had to walk 30 minutes in the rain.', 'keep', ''], ['Chuo', 'accommodation_value', 'rev_523492231946533175', '6) Unsettling Financial penalties : $1000 clean up fee for anybody honestly reporting having contracted covid simply did not make any sense.', 'keep', ''], ['Chuo', 'accommodation_value', 'rev_925027532928733462', "For a stay in the middle of one of the world's most expensive cities this place was an absolute steal!", 'not_negative', 'positive or neutral text misread as negative'], ['Chuo', 'crowd_level', 'rev_1076455207144824762', 'Very very noisy construction right across the street, starting at about 8am.', 'keep', ''], ['Chuo', 'crowd_level', 'rev_1496856105323612988', 'No kitchen, no utensils, horrible bathroom smell, clogged shower, nonstop noise, and all 4 of us got bitten by something (possibly bedbugs).', 'wrong_aspect_or_not_about_area', 'noise listed as a room defect'], ['Chiyoda', 'food_dining', 'rev_1422855839848591155', 'The moment I stepped in, I was hit with an unbearable stench — a mix of bad food and pigeons.', 'wrong_aspect_or_not_about_area', 'smell in the room'], ['Chiyoda', 'food_dining', 'rev_251052979', 'literally hundreds of restaurants and bars within a 10 to 15 minute walk, including ramen and sushi just steps away from the front door.', 'not_negative', 'positive or neutral text misread as negative'], ['Chiyoda', 'nightlife', 'rev_251052979', 'literally hundreds of restaurants and bars within a 10 to 15 minute walk, including ramen and sushi just steps away from the front door.', 'not_negative', 'positive or neutral text misread as negative'], ['Chiyoda', 'walkability_transit', 'rev_1338172293155742063', 'One stop on subway from Tokyo station.', 'not_negative', 'positive or neutral text misread as negative'], ['Chiyoda', 'walkability_transit', 'rev_1256190648265667558', 'Would not come back or recommend, price paid was not worth our stay, area is close to transport', 'not_negative', 'positive or neutral text misread as negative'], ['Chiyoda', 'safety', 'rev_761250666998437330', 'Dangerous sharp corner of wood next to bed at eye level.', 'wrong_aspect_or_not_about_area', 'furniture hazard'], ['Chiyoda', 'safety', 'rev_1356967932422030075', 'Potentially dangerous getting out of bunk and down ladder.', 'wrong_aspect_or_not_about_area', 'bunk ladder hazard'], ['Chiyoda', 'accommodation_value', 'rev_1120681436709045429', 'you have to buy plastic bags from the supermarket yourself, if you stay longer, you have to take out the trash yourself, toilet dirty', 'keep', ''], ['Chiyoda', 'accommodation_value', 'rev_817672696472993163', 'Bad smell and not clean bathroom at all.', 'keep', ''], ['Chiyoda', 'crowd_level', 'rev_1000378550298978804', 'The noise proof is terrible, you can hear all sorts of noises, even coughing sounds and whispering, throughout the day and night.', 'wrong_aspect_or_not_about_area', 'building soundproofing, not the area'], ['Chiyoda', 'crowd_level', 'rev_1136574137031429404', 'traffic noise at all hours was a nuisance when sleeping.', 'keep', ''], ['Minato', 'food_dining', 'rev_1304785489188151743', 'Vending machine literally outside the front door and 10 restaurants 20-50m away.', 'not_negative', 'positive or neutral text misread as negative'], ['Minato', 'food_dining', 'rev_1422224550509507843', 'Restaurants within a 5 minute walk abound!', 'not_negative', 'positive or neutral text misread as negative'], ['Minato', 'nightlife', 'rev_1011999240838104797', 'Waited at karaoke tired and unshowered into the morning with no staff ever showing up.', 'wrong_aspect_or_not_about_area', 'check-in problem'], ['Minato', 'family_friendliness', 'rev_1470801914382783812', 'Spotless and large enough for 3 adults and 2 children.', 'not_negative', 'positive or neutral text misread as negative'], ['Minato', 'family_friendliness', 'rev_427097500775334494', 'a had two children of 3 and younger with me Overall, horrible experience and no way to treat a guests with a family.', 'keep', ''], ['Minato', 'walkability_transit', 'rev_1372879231117427139', 'Yuki send check in info only one hour before check in time and we could not access it', 'not_negative', 'check-in info'], ['Minato', 'walkability_transit', 'rev_960502037677586258', 'Also one station ride away from Tokyo station.', 'not_negative', 'positive or neutral text misread as negative'], ['Minato', 'safety', 'rev_1317128197822107589', 'very dangerous for the waist line!', 'wrong_aspect_or_not_about_area', 'joke about food'], ['Minato', 'safety', 'rev_769885789668938877', 'very dangerous to use, and the side of the suitcase can get scratched.', 'wrong_aspect_or_not_about_area', 'object hazard'], ['Minato', 'accommodation_value', 'rev_566319427', 'The place was spotless and we were blown away by the neighborhood.', 'not_negative', 'positive or neutral text misread as negative'], ['Minato', 'accommodation_value', 'rev_1316427026440618198', 'No knife in the kitchen, pot, or mugs, and host refused to provide these items.', 'keep', ''], ['Minato', 'crowd_level', 'rev_1098216774119018015', 'Incredibly noisy once people start waking up.', 'keep', ''], ['Minato', 'crowd_level', 'rev_968432653869615765', 'A room with a horrible smell, bad and dirty air conditioning, lots of noise', 'wrong_aspect_or_not_about_area', 'noise listed as a room defect'], ['Sumida', 'food_dining', 'rev_1485982316157951449', 'we all cannot even fit at the dining table.', 'wrong_aspect_or_not_about_area', 'dining table size'], ['Sumida', 'food_dining', 'rev_1406969308948080750', 'When we arrived, the room was still not clean—dust everywhere, finger stains, dirty sofa armrests, and food crumbs in the sofa bed.', 'wrong_aspect_or_not_about_area', 'room cleanliness'], ['Sumida', 'nightlife', 'rev_1359861444155677299', "Hair everywhere, men's hair from shaving, broken razors, dirty soap bars.", 'wrong_aspect_or_not_about_area', 'soap bars'], ['Sumida', 'family_friendliness', 'rev_1264210560112187520', 'it was a very bad experience for me n my family.', 'wrong_aspect_or_not_about_area', 'family mentioned in passing'], ['Sumida', 'walkability_transit', 'rev_1066348220686007212', 'Near train line and noisy and no lift.', 'wrong_aspect_or_not_about_area', 'mixed: near train line is positive for transit'], ['Sumida', 'walkability_transit', 'rev_1019200484717043289', '17 minute walk to either of 2 trains and 5 minutes to the bus stop, 3 combini within 3 blocks walk, laundromat 2 blocks away to dry clothes.', 'not_negative', 'factual distances, mixed'], ['Sumida', 'safety', 'rev_724095342', 'This whole experience made me feel extremely unsafe and I was not able to sleep at night.', 'wrong_aspect_or_not_about_area', 'unclear what is unsafe'], ['Sumida', 'safety', 'rev_1017826401456649809', 'This makes you feel so unsafe , not private.', 'wrong_aspect_or_not_about_area', 'privacy at the property'], ['Sumida', 'accommodation_value', 'rev_1253296168208301739', 'What welcomed us was a mouldy smell house,mouldy furniture, unclean floors, super horrible bathroom.', 'keep', ''], ['Sumida', 'accommodation_value', 'rev_1210501929108847146', 'It’s actually 100yen/30 min so we wasted our money.', 'keep', ''], ['Sumida', 'crowd_level', 'rev_1299654044431996607', 'The mirror above the bathroom sink emits a horrible high pitched noise that you cannot turn off.', 'wrong_aspect_or_not_about_area', 'appliance noise'], ['Sumida', 'crowd_level', 'rev_1059777162642407556', 'no one else was complying so very noisy.', 'wrong_aspect_or_not_about_area', 'other guests, not the area'], ['Koto', 'food_dining', 'rev_1160522510247841231', 'I could not connect to the wifi at all and did not get any work done, I had to go to a nearby cafe (~15 min walk).', 'wrong_aspect_or_not_about_area', 'wifi'], ['Koto', 'food_dining', 'rev_1314973831220193571', "all the restaurants were either closed or told us that they can't accept us.", 'keep', ''], ['Koto', 'walkability_transit', 'rev_767004903724005379', 'Less than 10 minutes slow walk to two different rail stations.', 'not_negative', 'positive or neutral text misread as negative'], ['Koto', 'walkability_transit', 'rev_944538080880452686', 'We taxied from the airport and fell right asleep!', 'not_negative', 'positive or neutral text misread as negative'], ['Koto', 'safety', 'rev_862716142605484923', 'We felt unsafe & disgusted with our stay.', 'wrong_aspect_or_not_about_area', 'the stay, not the area'], ['Koto', 'accommodation_value', 'rev_1151093876633069229', 'Even with the help of a smartphone translator, the washing machine was trial and error.', 'not_negative', 'mild, not strongly worded'], ['Koto', 'accommodation_value', 'rev_1266424099447201476', 'Ants crawling everywhere and the bathroom was the worst part.', 'keep', ''], ['Koto', 'crowd_level', 'rev_1412228049321660596', "It's super noisy there I swear, and the landlord don't have any solution for this.", 'keep', ''], ['Koto', 'crowd_level', 'rev_1398755411846106560', 'The worst thing was the vent fan in the ONLY TOILET in the house makes a high-pitched buzzing noise 24/7, making it impossible to sleep in the first floor bedroom.', 'wrong_aspect_or_not_about_area', 'appliance noise'], ['Toshima', 'food_dining', 'rev_1308386377919467587', 'Tip: Grab baked goods from the station bakery after 9 PM for a 20% discount!', 'not_negative', 'positive or neutral text misread as negative'], ['Toshima', 'shopping', 'rev_259888001', 'Nothing to watch on TV, literally all about home shopping...etc.', 'wrong_aspect_or_not_about_area', 'TV shopping channel'], ['Toshima', 'family_friendliness', 'rev_1317148627581873937', 'I never once heard a baby cry!) not good', 'not_negative', 'positive or neutral text misread as negative'], ['Toshima', 'walkability_transit', 'rev_1427237989672128906', 'I accidentally left my backpack in the apartment after heading to the airport, and when I messaged the owner, she replied right away.', 'not_negative', 'positive or neutral text misread as negative'], ['Toshima', 'walkability_transit', 'rev_1338830460420668052', 'Literally 5 mins walk away from the train station.', 'not_negative', 'positive or neutral text misread as negative'], ['Toshima', 'safety', 'rev_1127988130869862161', 'With heavy luggage it was a difficult affair and also very dangerous!', 'wrong_aspect_or_not_about_area', 'luggage on stairs'], ['Toshima', 'safety', 'rev_481442155', 'It’s absolutely ridiculous and it really really shocked me that as a paying customer my safety was not valued in any way...', 'wrong_aspect_or_not_about_area', 'host conduct'], ['Toshima', 'accommodation_value', 'rev_785780269199585097', 'Overall a very bad stay, would not recommend if you place focus on cleanliness.', 'keep', ''], ['Toshima', 'accommodation_value', 'rev_1039552007945931542', 'Every light and window was covered in dust and there was mould EVERYWHERE in the bathroom.', 'keep', ''], ['Toshima', 'crowd_level', 'rev_1278720067535979128', 'The worst part is the loud train noise.', 'keep', ''], ['Toshima', 'crowd_level', 'rev_531633185', 'Train noise did not bother me at all.', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'food_dining', 'rev_971422810366072482', 'Junko’s place was close to the train station, grocery stores, and restaurants.', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'food_dining', 'rev_838802111968656135', 'literally steps away from tons of food and shopping.', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'shopping', 'rev_838802111968656135', 'literally steps away from tons of food and shopping.', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'shopping', 'rev_209769480', 'For those music junkies, there are even vinyl record shops in the area!', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'family_friendliness', 'rev_903216690399964048', 'Horrible experience in middle of night with family.', 'wrong_aspect_or_not_about_area', 'family mentioned in passing'], ['Setagaya', 'walkability_transit', 'rev_1398250627356969661', 'The location is also unbeatable, being one metro station away from the bustling Shibuya.', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'walkability_transit', 'rev_1396111290793066278', 'Literally less than two minutes from the train station.', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'safety', 'rev_216072849', "Unsafe and shouldn't promote this as an amenity.", 'wrong_aspect_or_not_about_area', 'an amenity'], ['Setagaya', 'accommodation_value', 'rev_447073336', 'It was spotless and only about a 5 minute walk from Yoga station.', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'accommodation_value', 'rev_1294684129233331579', 'This took him driving two round trips and he would not accept any money from us!', 'not_negative', 'positive or neutral text misread as negative'], ['Setagaya', 'crowd_level', 'rev_990987465701506191', 'It was so noisy we couldn’t sleep.', 'keep', ''], ['Setagaya', 'crowd_level', 'rev_950408271025078511', 'Faultless especially if you want to be away from the hustle and bustle of Shibuya', 'not_negative', 'positive or neutral text misread as negative'], ['Itabashi', 'food_dining', 'rev_795224311235054635', 'We tried to eat breakfast and found three spoons, one of which was so rusted and dirty we could not use.', 'wrong_aspect_or_not_about_area', 'tagged food: kitchen utensils, not dining'], ['Itabashi', 'food_dining', 'rev_1412801401736392856', 'Our refund request (due to cleanliness & not being able to cook, forcing extra taxi/food costs) was denied.', 'wrong_aspect_or_not_about_area', 'refund dispute'], ['Itabashi', 'family_friendliness', 'rev_1017097195622197649', 'I would not recommend this Airbnb to anyone, especially families with young children.', 'keep', ''], ['Itabashi', 'walkability_transit', 'rev_1014865640673953273', 'It is dangerous and in an unsafe neighborhood, not to mention far from Ikebukuro station.', 'keep', ''], ['Itabashi', 'safety', 'rev_1014865640673953273', 'It is dangerous and in an unsafe neighborhood, not to mention far from Ikebukuro station.', 'keep', ''], ['Itabashi', 'accommodation_value', 'rev_795224311235054635', 'We tried to eat breakfast and found three spoons, one of which was so rusted and dirty we could not use.', 'keep', ''], ['Itabashi', 'accommodation_value', 'rev_401268400', 'There was bags of trash, the beds were not made, and counters were not cleaned.', 'keep', '']]}
for fname, d in LABEL_DATA.items():
    pd.DataFrame(d['data'], columns=d['columns']).to_csv(ROOT/'data/labels'/fname, index=False)
print({k: len(v['data']) for k, v in LABEL_DATA.items()})

{'llm_labels.csv': 300, 'audit_completed_stage2.csv': 50, 'aspect_audit_sample.csv': 108, 'aspect_audit_llm.csv': 108, 'sentence_val_candidates.csv': 150, 'sentence_llm_labels.csv': 150, 'sentence_audit_completed.csv': 30, 'sentence_audit_key.csv': 30, 'labelset_design.csv': 300, 'negative_quote_check.csv': 120}


In [10]:
import numpy as np
print('python', sys.version.split()[0])
for p in ['numpy','pandas','pyarrow','scikit-learn','scipy','statsmodels','lingua-language-detector','datasketch',
          'vaderSentiment','umap-learn','bertopic','gensim','torch','transformers','sentence-transformers']:
    try: print(f'{p:26s}', md.version(p))
    except md.PackageNotFoundError: print(f'{p:26s} not installed (only needed for GPU steps)')

python 3.13.18


numpy                      2.4.6
pandas                     2.2.3


pyarrow                    23.0.1
scikit-learn               1.6.1
scipy                      1.16.3
statsmodels                0.15.0
lingua-language-detector   2.2.0
datasketch                 2.0.0
vaderSentiment             3.3.2
umap-learn                 0.5.12
bertopic                   0.17.4
gensim                     4.4.0
torch                      2.14.1


transformers               5.23.0
sentence-transformers      5.7.0


# Stage 0: Setup and schema check
Tokyo ward recommender (Inside Airbnb Tokyo, snapshot 2025-09-29). This stage loads both raw tables, prunes them to the needed columns, filters to the 11 in-scope wards, reports data quality, and saves parquet.

**Rationale**
- Listings and reviews stay as **two separate tables**. Reviews are filtered to in-scope listings by an ID lookup (`isin`), not a join, so nothing gets flattened.
- Columns are checked against `df.columns` **before** loading, to catch the British/American spelling trap.
- Language counts here come from a **preliminary seeded sample**. Full language detection runs in Stage 1.

In [11]:
import sys, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd

SEED = 42
RAW = RAW_DIR
OUT = ROOT/'data/interim'

print('python', sys.version.split()[0])
for p in ['pandas','numpy','pyarrow','lingua-language-detector']:
    try: print(f'{p:28s}', md.version(p))
    except md.PackageNotFoundError: print(f'{p:28s} NOT INSTALLED')

python 3.13.18
pandas                       2.2.3
numpy                        2.4.6
pyarrow                      23.0.1
lingua-language-detector     2.2.0


## Column contract
Spellings checked: `neighbourhood_cleansed` and `host_neighbourhood` are British; `neighborhood_overview` is American. The cell fails loudly if any column is missing.

In [12]:
LISTING_COLS = ['id','neighbourhood_cleansed','latitude','longitude','host_neighbourhood',
                'neighborhood_overview','price','review_scores_location','review_scores_rating',
                'number_of_reviews','description','last_scraped']
REVIEW_COLS  = ['id','listing_id','date','comments']

l_header = pd.read_csv(raw_paths('listings')[0], nrows=0).columns
r_header = pd.read_csv(raw_paths('reviews')[0], nrows=0).columns
missing = [c for c in LISTING_COLS if c not in l_header] + [c for c in REVIEW_COLS if c not in r_header]
assert not missing, f'Missing columns: {missing}'
print(f'listings file columns: {len(l_header)} | reviews file columns: {len(r_header)} | contract OK '
      f'(the repository copy keeps only these columns; the original files have 79 and 6)')

listings file columns: 12 | reviews file columns: 4 | contract OK (the repository copy keeps only these columns; the original files have 79 and 6)


## Load (pruned at read time)
`usecols` keeps memory low (about 7 GB RAM here). IDs are read as `int64` so they can't drift into floats.

In [13]:
listings_raw = read_raw('listings', usecols=LISTING_COLS, dtype={'id':'int64'})
reviews_raw  = read_raw('reviews',  usecols=REVIEW_COLS,  dtype={'id':'int64','listing_id':'int64'})
listings_raw = listings_raw.rename(columns={'id':'listing_id'})
reviews_raw  = reviews_raw.rename(columns={'id':'review_id'})
print('listings raw:', listings_raw.shape, '| reviews raw:', reviews_raw.shape)
assert listings_raw.listing_id.is_unique, 'duplicate listing_id'
assert reviews_raw.review_id.is_unique,  'duplicate review_id'
print('last_scraped values:', listings_raw.last_scraped.value_counts().to_dict())

listings raw: (27945, 12) | reviews raw: (1005052, 4)
last_scraped values: {'2025-09-29': 15841, '2025-09-30': 12104}


## Ward filter
Raw labels look like `"Shinjuku Ku"`. I map them to clean ward names and assert that all 11 wards are present.

In [14]:
WARDS = ['Shinjuku','Shibuya','Taito','Chuo','Chiyoda','Minato','Sumida','Koto','Toshima','Setagaya','Itabashi']
WARD_MAP = {f'{w} Ku': w for w in WARDS}
found = set(listings_raw.neighbourhood_cleansed.unique())
assert set(WARD_MAP) <= found, f'labels not found: {set(WARD_MAP) - found}'

listings = listings_raw[listings_raw.neighbourhood_cleansed.isin(WARD_MAP)].copy()
listings['ward'] = listings.neighbourhood_cleansed.map(WARD_MAP)
print(f'listings: {len(listings_raw):,} -> {len(listings):,} in scope '
      f'({len(listings)/len(listings_raw):.1%}); {listings_raw.neighbourhood_cleansed.nunique()} areas -> 11 wards')

# Reviews: filter by ID lookup, not a join. The two tables stay separate.
in_scope_ids = set(listings.listing_id)
orphans = ~reviews_raw.listing_id.isin(set(listings_raw.listing_id))
reviews = reviews_raw[reviews_raw.listing_id.isin(in_scope_ids)].copy()
print(f'reviews: {len(reviews_raw):,} -> {len(reviews):,} in scope ({len(reviews)/len(reviews_raw):.1%}); '
      f'orphan reviews (listing not in listings file): {orphans.sum():,}')

listings: 27,945 -> 20,370 in scope (72.9%); 49 areas -> 11 wards
reviews: 1,005,052 -> 738,305 in scope (73.5%); orphan reviews (listing not in listings file): 0


## Price parsing
Values carry a `$` sign, but they are **Japanese yen per night**: the magnitudes (median about ¥17,000) fit Tokyo prices, and the currency was confirmed as JPY by the project owner. The `$` and commas are stripped and the number is kept as JPY. A few listings carry placeholder prices (maximum 9,999,998), so only medians are reported.


In [15]:
listings['price_raw'] = listings.price
listings['price'] = pd.to_numeric(listings.price.str.replace(r'[$,]', '', regex=True), errors='coerce')
bad = listings.price_raw.notna() & listings.price.isna()
assert bad.sum() == 0, f'{bad.sum()} prices failed to parse'
listings = listings.drop(columns='price_raw')
print(listings.price.describe(percentiles=[.01,.5,.99]).round(0).to_string())

count      18451.0
mean       26558.0
std        86492.0
min         1700.0
1%          4600.0
50%        17372.0
99%       160539.0
max      9999998.0


## Shapes, null rates, ward counts

In [16]:
reviews['date'] = pd.to_datetime(reviews.date, errors='coerce')
reviews['comments_blank'] = reviews.comments.isna() | (reviews.comments.fillna('').str.strip() == '')
listings['overview_blank'] = listings.neighborhood_overview.isna() | (listings.neighborhood_overview.fillna('').str.strip() == '')

print('listings (in scope):', listings.shape, '| reviews (in scope):', reviews.shape)
print('review date range:', reviews.date.min().date(), '->', reviews.date.max().date(), '| unparsed dates:', reviews.date.isna().sum())
print('\nListing null rates:'); print(listings[LISTING_COLS[1:]].isna().mean().rename('null_rate').round(3).to_string())
print('\nReview null rates:');  print(reviews[['listing_id','date','comments']].isna().mean().rename('null_rate').round(4).to_string())
print(f"blank/whitespace comments: {reviews.comments_blank.sum():,}")

listings (in scope): (20370, 14) | reviews (in scope): (738305, 5)
review date range: 2011-09-21 -> 2025-09-30 | unparsed dates: 0

Listing null rates:
neighbourhood_cleansed    0.000
latitude                  0.000
longitude                 0.000
host_neighbourhood        0.733
neighborhood_overview     0.464
price                     0.094
review_scores_location    0.144
review_scores_rating      0.144
number_of_reviews         0.000
description               0.016
last_scraped              0.000

Review null rates:
listing_id    0.0000
date          0.0000
comments      0.0001
blank/whitespace comments: 110


In [17]:
rev_ward = reviews.listing_id.map(listings.set_index('listing_id').ward)   # lookup for counting only, not stored
ward_tbl = pd.DataFrame({
    'listings': listings.ward.value_counts(),
    'listings_with_overview': listings.loc[~listings.overview_blank, 'ward'].value_counts(),
    'reviews': rev_ward.value_counts(),
    'median_price': listings.groupby('ward').price.median(),
    'mean_review_scores_location': listings.groupby('ward').review_scores_location.mean().round(2),
}).reindex(WARDS)
ward_tbl['overview_rate'] = (ward_tbl.listings_with_overview / ward_tbl.listings).round(3)
ward_tbl.loc['TOTAL', ['listings','listings_with_overview','reviews']] = ward_tbl[['listings','listings_with_overview','reviews']].sum()
ward_tbl

,listings,listings_with_overview,reviews,median_price,mean_review_scores_location,overview_rate
Shinjuku,4991.0,2822.0,181983.0,17994.0,4.76,0.565
Shibuya,1902.0,1032.0,75648.0,21972.0,4.78,0.543
Taito,2950.0,1217.0,126842.0,19967.0,4.71,0.413
Chuo,382.0,234.0,22838.0,25429.0,4.76,0.613
Chiyoda,259.0,131.0,13554.0,22374.0,4.82,0.506
Minato,1190.0,764.0,40091.0,24293.0,4.81,0.642
Sumida,3985.0,2461.0,130038.0,15082.0,4.73,0.618
Koto,548.0,240.0,23481.0,14667.0,4.70,0.438
Toshima,2749.0,1306.0,78076.0,14852.5,4.69,0.475
Setagaya,912.0,538.0,34046.0,12910.0,4.76,0.590


## Preliminary language counts (seeded sample)
lingua-py runs on a seeded random sample of **2,000 reviews and 2,000 overviews**. These are estimates for scoping only. Full detection on every text, with confidence thresholds, happens in Stage 1.
The **nlptown-covered** share counts EN, NL, DE, FR, IT and ES.

In [18]:
import time
from lingua import LanguageDetectorBuilder
detector = LanguageDetectorBuilder.from_all_languages().build()
NLPTOWN = {'ENGLISH','DUTCH','GERMAN','FRENCH','ITALIAN','SPANISH'}

def lang_sample(texts, n=2000, seed=SEED):
    s = texts.dropna().loc[lambda x: x.str.strip() != ''].sample(n=min(n, texts.notna().sum()), random_state=seed)
    t0 = time.time()
    out = [ (r.name if (r := detector.detect_language_of(t[:1000])) else 'UNKNOWN') for t in s ]
    return pd.Series(out), time.time() - t0

rev_lang, t_r = lang_sample(reviews.comments)
ov_lang,  t_o = lang_sample(listings.neighborhood_overview)
lang_tbl = pd.concat({'reviews_share': rev_lang.value_counts(normalize=True),
                      'overviews_share': ov_lang.value_counts(normalize=True)}, axis=1).fillna(0).round(3)
print(f'detect time: reviews {t_r:.1f}s, overviews {t_o:.1f}s for 2,000 each (texts truncated to 1,000 chars)')
print(f"nlptown-covered share: reviews {rev_lang.isin(NLPTOWN).mean():.1%} | overviews {ov_lang.isin(NLPTOWN).mean():.1%}")
lang_tbl.head(12)

detect time: reviews 4.6s, overviews 8.4s for 2,000 each (texts truncated to 1,000 chars)
nlptown-covered share: reviews 63.8% | overviews 94.8%


,reviews_share,overviews_share
ENGLISH,0.574,0.946
CHINESE,0.142,0.003
JAPANESE,0.126,0.004
KOREAN,0.064,0.000
FRENCH,0.031,0.001
SPANISH,0.016,0.000
GERMAN,0.011,0.000
SOMALI,0.006,0.000
ITALIAN,0.005,0.000
THAI,0.004,0.000


## Save parquet
Pruned, in-scope tables only. Helper flag columns are dropped. Both tables stay separate.

In [19]:
listings_out = listings.drop(columns=['overview_blank'])
reviews_out  = reviews.drop(columns=['comments_blank'])
listings_out.to_parquet(OUT/'listings_pruned.parquet', index=False)
reviews_out.to_parquet(OUT/'reviews_pruned.parquet', index=False)
chk_l, chk_r = pd.read_parquet(OUT/'listings_pruned.parquet'), pd.read_parquet(OUT/'reviews_pruned.parquet')
assert len(chk_l) == len(listings_out) and len(chk_r) == len(reviews_out)
for f in ['listings_pruned.parquet','reviews_pruned.parquet']:
    print(f, f'{(OUT/f).stat().st_size/1e6:.1f} MB')
print('listings cols:', list(chk_l.columns)); print('reviews cols:', list(chk_r.columns))

listings_pruned.parquet 6.1 MB
reviews_pruned.parquet 113.2 MB
listings cols: ['listing_id', 'neighbourhood_cleansed', 'latitude', 'longitude', 'host_neighbourhood', 'neighborhood_overview', 'price', 'review_scores_location', 'review_scores_rating', 'number_of_reviews', 'description', 'last_scraped', 'ward']
reviews cols: ['review_id', 'listing_id', 'date', 'comments']


In [20]:
# Free memory from the previous stage (every stage reloads its inputs from disk)
import gc
for _n in ['listings_raw', 'reviews_raw', 'listings', 'reviews', 'rev_ward', 'detector', 'rev_lang', 'ov_lang', 'listings_out', 'reviews_out', 'chk_l', 'chk_r', 'orphans']:
    globals().pop(_n, None)
gc.collect()
print('freed memory before Stage 1')

freed memory before Stage 1


# Stage 1: Preprocess and unified corpus
Input: the Stage 0 parquet files. Output: `unified_corpus.parquet`, plus a dedup log and a run log.

**Decisions**
- **Cleaning** lives in `src/textclean.py`, which is unit-tested: HTML `<br/>` and tags removed, entities unescaped, URLs stripped, NFKC normalization (full-width to half-width), whitespace collapsed.
- **Reviews are not text-deduplicated.** Each `review_id` is a separate guest. Short texts like "Good" (about 1,600 times) or 「ありがとうございました」 repeat legitimately, and removing them would bias sentiment upward or downward. I drop only texts with no letters at all, such as `"."`.
- **Overviews:** one row per `listing_id` (asserted), then exact dedup, then MinHash near-dup dedup at Jaccard 0.7, **within each ward**. Property-manager templates inflate one ward's voice. Deduping within a ward removes that inflation while keeping a template that appears in two wards in both.
- **MinHash** uses character 5-gram shingles, which work for any script. LSH candidates are **verified with exact Jaccard ≥ 0.7**, so LSH false positives are never merged.
- **Language** uses lingua-py over all languages with `minimum_relative_distance=0.1`. On a 5,000-review test this cut false labels such as Somali on short English text, at a cost of about 2% marked unknown. Labels are ISO 639-1 codes; `UNK` means unknown.
- **Unified schema:** `text_id, source, ward, language, date, text`, plus helper columns: `listing_id` (needed for the Stage 2 per-listing external check), `nlptown_supported`, `approx_tokens`, `long_text`.

In [21]:
import sys, json, time, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd
from textclean import clean_text, has_content, approx_tokens, shingles, jaccard, NLPTOWN_LANGS

SEED = 42
LANG_MIN_DIST = 0.1
MH_THRESHOLD, MH_PERM, SHINGLE_K = 0.7, 128, 5
LONG_TOKEN_LIMIT = 512
TEST_N = 5000          # smoke-test size before the full run
DATA = ROOT/'data/interim'
LOG = {'seed': SEED, 'lang_min_relative_distance': LANG_MIN_DIST,
       'minhash': {'threshold': MH_THRESHOLD, 'num_perm': MH_PERM, 'shingle_k': SHINGLE_K}}
for p in ['pandas','pyarrow','lingua-language-detector','datasketch']:
    print(f'{p:26s}', md.version(p))

listings = pd.read_parquet(DATA/'listings_pruned.parquet')
reviews  = pd.read_parquet(DATA/'reviews_pruned.parquet')
print('loaded:', listings.shape, reviews.shape)

pandas                     2.2.3
pyarrow                    23.0.1
lingua-language-detector   2.2.0
datasketch                 2.0.0


loaded: (20370, 13) (738305, 4)


## 1. Reviews: clean and drop empty texts

In [22]:
ward_of = listings.set_index('listing_id').ward       # ID lookup; tables are not joined
r = reviews.copy()
r['text'] = r.comments.map(clean_text)
r['ward'] = r.listing_id.map(ward_of)
assert r.ward.notna().all(), 'review with no in-scope listing'

n0 = len(r)
n_blank = (r.text == '').sum()
r = r[r.text != '']
n_nocontent = (~r.text.map(has_content)).sum()
r = r[r.text.map(has_content)]
LOG['reviews'] = {'input': n0, 'dropped_blank': int(n_blank), 'dropped_no_letters': int(n_nocontent), 'kept': len(r)}
assert len(r) == n0 - n_blank - n_nocontent
print(LOG['reviews'], f"| drop rate {1 - len(r)/n0:.3%}")

{'input': 738305, 'dropped_blank': 111, 'dropped_no_letters': 2442, 'kept': 735752} | drop rate 0.346%


## 2. Overviews: one row per listing, exact dedup, then MinHash dedup (within ward)

In [23]:
o = listings[['listing_id','ward','neighborhood_overview','last_scraped']].copy()
assert o.listing_id.is_unique, 'overview table must be one row per listing_id'
o['text'] = o.neighborhood_overview.map(clean_text)
n_list = len(o)
o = o[(o.text != '') & o.text.map(has_content)].sort_values('listing_id').reset_index(drop=True)
n_nonblank = len(o)

# Exact dedup within ward. Canonical = lowest listing_id (deterministic).
o['norm'] = o.text.str.lower()
o['exact_canon'] = o.groupby(['ward','norm']).listing_id.transform('min')
exact_dups = o[o.listing_id != o.exact_canon]
o1 = o[o.listing_id == o.exact_canon].copy()

# Context: how many exact templates span more than one ward
xw = o.groupby('norm').ward.nunique()
LOG['overviews'] = {'listings': n_list, 'nonblank_overviews': n_nonblank,
                    'exact_dups_removed_within_ward': len(exact_dups),
                    'exact_dup_rate': round(len(exact_dups)/n_nonblank, 4),
                    'templates_spanning_multiple_wards': int((xw > 1).sum())}
print(LOG['overviews'])

{'listings': 20370, 'nonblank_overviews': 10916, 'exact_dups_removed_within_ward': 7063, 'exact_dup_rate': 0.647, 'templates_spanning_multiple_wards': 40}


In [24]:
from datasketch import MinHash, MinHashLSH

def minhash_of(sh):
    m = MinHash(num_perm=MH_PERM, seed=SEED)
    m.update_batch([s.encode('utf8') for s in sh])
    return m

def near_dup_canon(df):
    """Return {listing_id: canonical_listing_id} for one ward. Union-find over verified pairs."""
    ids = df.listing_id.tolist()
    sh = {i: shingles(t, SHINGLE_K) for i, t in zip(ids, df.text)}
    lsh = MinHashLSH(threshold=MH_THRESHOLD, num_perm=MH_PERM)
    mh = {}
    for i in ids:
        mh[i] = minhash_of(sh[i]); lsh.insert(i, mh[i])
    parent = {i: i for i in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    n_cand = n_ver = 0
    for i in ids:
        for j in lsh.query(mh[i]):
            if j <= i: continue
            n_cand += 1
            if jaccard(sh[i], sh[j]) >= MH_THRESHOLD:       # exact verification
                n_ver += 1
                a, b = find(i), find(j)
                parent[max(a, b)] = min(a, b)               # canonical = lowest id
    return {i: find(i) for i in ids}, n_cand, n_ver

# Smoke test on a tiny synthetic ward before the real run
_t = pd.DataFrame({'listing_id':[1,2,3], 'text':[
    'Quiet street near Asakusa station with many restaurants and a supermarket nearby.',
    'Quiet street near Asakusa station with many restaurants and a supermarket close by.',
    'Shibuya crossing is a five minute walk; lots of nightlife and shopping.']})
_c, _, _ = near_dup_canon(_t)
assert _c == {1:1, 2:1, 3:3}, _c
print('MinHash smoke test passed')

MinHash smoke test passed


In [25]:
t0 = time.time()
canon, per_ward = {}, []
for w, g in o1.groupby('ward'):
    cm, n_cand, n_ver = near_dup_canon(g)
    canon.update(cm)
    per_ward.append({'ward': w, 'after_exact': len(g), 'lsh_candidates': n_cand,
                     'verified_pairs': n_ver, 'kept': len(set(cm.values()))})
o1['near_canon'] = o1.listing_id.map(canon)
near_dups = o1[o1.listing_id != o1.near_canon]
ov = o1[o1.listing_id == o1.near_canon].copy()

# Dedup log: every removed overview -> the canonical it maps to
o['final_canon'] = o.exact_canon.map(lambda c: canon.get(c, c))
dedup_log = o.loc[o.listing_id != o.final_canon, ['listing_id','ward','final_canon']].rename(
    columns={'final_canon':'canonical_listing_id'})
dedup_log['reason'] = np.where(dedup_log.listing_id.isin(exact_dups.listing_id), 'exact', 'minhash_0.7')

assert len(ov) + len(dedup_log) == n_nonblank
LOG['overviews'].update({'near_dups_removed': len(near_dups), 'kept': len(ov),
                         'total_dedup_rate': round(1 - len(ov)/n_nonblank, 4),
                         'minhash_seconds': round(time.time() - t0, 1)})
pw = pd.DataFrame(per_ward).set_index('ward')
pw.insert(0, 'nonblank', o.groupby('ward').size())
pw['dedup_rate'] = (1 - pw.kept / pw.nonblank).round(3)
print(LOG['overviews']); pw

{'listings': 20370, 'nonblank_overviews': 10916, 'exact_dups_removed_within_ward': 7063, 'exact_dup_rate': 0.647, 'templates_spanning_multiple_wards': 40, 'near_dups_removed': 383, 'kept': 3470, 'total_dedup_rate': 0.6821, 'minhash_seconds': 2.3}


,nonblank,after_exact,lsh_candidates,verified_pairs,kept,dedup_rate
ward,,,,,,
Chiyoda,131,52,3,3,49,0.626
Chuo,234,66,9,9,58,0.752
Itabashi,175,114,8,6,108,0.383
Koto,240,133,19,18,117,0.512
Minato,764,222,37,31,204,0.733
Setagaya,538,234,30,26,214,0.602
Shibuya,1032,460,57,49,422,0.591
Shinjuku,2822,872,202,196,766,0.729
Sumida,2460,664,151,131,580,0.764


## 3. Language detection
Smoke test on a seeded sample first, then the full corpus. Texts are truncated to 1,000 characters for detection only; the stored text is not truncated.

In [26]:
from lingua import LanguageDetectorBuilder
detector = (LanguageDetectorBuilder.from_all_languages()
            .with_minimum_relative_distance(LANG_MIN_DIST)
            .with_preloaded_language_models().build())

def detect(texts):
    out = detector.detect_languages_in_parallel_of([t[:1000] for t in texts])
    return [x.iso_code_639_1.name if x else 'UNK' for x in out]

smp = r.sample(TEST_N, random_state=SEED)
t0 = time.time(); smp_lang = pd.Series(detect(smp.text.tolist()))
rate = TEST_N / (time.time() - t0)
print(f'smoke test: {rate:.0f} texts/s -> est. full review run {len(r)/rate/60:.1f} min')
print(smp_lang.value_counts(normalize=True).head(8).round(3).to_dict())

smoke test: 804 texts/s -> est. full review run 15.2 min
{'EN': 0.547, 'ZH': 0.14, 'JA': 0.131, 'KO': 0.067, 'FR': 0.031, 'UNK': 0.025, 'ES': 0.02, 'DE': 0.017}


In [27]:
# CORRECTED (v2): overviews re-detected with a restricted language set.
# Reason: all-language detection labelled ~7% of overviews as Tagalog/Latin/Shona etc.; samples were
# English lists of romanized Japanese place names. Reviews keep all-language detection.
from lingua import Language as L
OV_LANGS = [L.ENGLISH, L.JAPANESE, L.CHINESE, L.KOREAN, L.FRENCH, L.SPANISH,
            L.GERMAN, L.ITALIAN, L.DUTCH, L.PORTUGUESE, L.THAI, L.RUSSIAN]
ov_detector = (LanguageDetectorBuilder.from_languages(*OV_LANGS)
               .with_minimum_relative_distance(LANG_MIN_DIST).with_preloaded_language_models().build())

t0 = time.time()
r['language'] = detect(r.text.tolist())
ov['language_all'] = detect(ov.text.tolist())                    # kept only for the log below
out = ov_detector.detect_languages_in_parallel_of([t[:1000] for t in ov.text])
ov['language'] = [x.iso_code_639_1.name if x else 'UNK' for x in out]
LOG['language_seconds'] = round(time.time() - t0, 1)
LOG['overview_lang_restricted_set'] = [l.iso_code_639_1.name for l in OV_LANGS]
LOG['overview_lang_changed'] = int((ov.language != ov.language_all).sum())
print(f"full detection: {LOG['language_seconds']/60:.1f} min | overview labels changed by restriction: {LOG['overview_lang_changed']}")
print('overview nlptown coverage: all-language', round(ov.language_all.isin(NLPTOWN_LANGS).mean(), 3),
      '-> restricted', round(ov.language.isin(NLPTOWN_LANGS).mean(), 3))
ov = ov.drop(columns='language_all')

full detection: 12.9 min | overview labels changed by restriction: 331
overview nlptown coverage: all-language 0.889 -> restricted 0.981


## 4. Build the unified corpus

In [28]:
rev_u = pd.DataFrame({'text_id': 'rev_' + r.review_id.astype(str), 'source': 'airbnb_review',
                      'ward': r.ward, 'language': r.language, 'date': pd.to_datetime(r.date),
                      'text': r.text, 'listing_id': r.listing_id})
ov_u  = pd.DataFrame({'text_id': 'ov_' + ov.listing_id.astype(str), 'source': 'airbnb_overview',
                      'ward': ov.ward, 'language': ov.language, 'date': pd.to_datetime(ov.last_scraped),
                      'text': ov.text, 'listing_id': ov.listing_id})
corpus = pd.concat([rev_u, ov_u], ignore_index=True)
corpus['nlptown_supported'] = corpus.language.isin(NLPTOWN_LANGS)
corpus['approx_tokens'] = [approx_tokens(t, l) for t, l in zip(corpus.text, corpus.language)]
corpus['long_text'] = corpus.approx_tokens > LONG_TOKEN_LIMIT

# Row-count and integrity assertions
assert corpus.text_id.is_unique
assert len(corpus) == LOG['reviews']['kept'] + LOG['overviews']['kept']
assert (corpus.source == 'airbnb_review').sum() == LOG['reviews']['kept']
assert set(corpus.ward) == set(listings.ward) and corpus.ward.nunique() == 11
assert corpus[['text_id','source','ward','language','date','text']].notna().all().all()
assert (corpus.text.str.len() > 0).all()
print('corpus:', corpus.shape, '| assertions passed')

corpus: (739222, 10) | assertions passed


In [29]:
cov = corpus.groupby('source').agg(n=('text_id','size'), nlptown_coverage=('nlptown_supported','mean'),
                                   unknown_rate=('language', lambda s: (s=='UNK').mean()),
                                   long_text=('long_text','sum')).round(3)
LOG['coverage'] = cov.reset_index().to_dict('records')
print(cov.to_string(), '\n')
lang = (corpus.groupby(['source','language']).size().rename('n').reset_index()
        .assign(share=lambda d: d.n / d.groupby('source').n.transform('sum')))
print(lang.sort_values(['source','n'], ascending=[True,False]).groupby('source').head(8).round(3).to_string(index=False))

                      n  nlptown_coverage  unknown_rate  long_text
source                                                            
airbnb_overview    3470             0.981         0.007          7
airbnb_review    735752             0.621         0.021       1692 

         source language      n  share
airbnb_overview       EN   3339  0.962
airbnb_overview       ES     26  0.007
airbnb_overview       JA     25  0.007
airbnb_overview      UNK     24  0.007
airbnb_overview       FR     22  0.006
airbnb_overview       ZH     15  0.004
airbnb_overview       DE     14  0.004
airbnb_overview       IT      2  0.001
  airbnb_review       EN 406075  0.552
  airbnb_review       ZH 104402  0.142
  airbnb_review       JA 100664  0.137
  airbnb_review       KO  47854  0.065
  airbnb_review       FR  23830  0.032
  airbnb_review      UNK  15689  0.021
  airbnb_review       DE  11400  0.015
  airbnb_review       ES  11267  0.015


In [30]:
ward_cov = corpus.pivot_table(index='ward', columns='source', values='nlptown_supported',
                              aggfunc=['size','mean']).round(3)
ward_cov.columns = [f'{a}_{b}'.replace('size','n').replace('mean','nlptown_cov') for a, b in ward_cov.columns]
cjk = corpus[corpus.source=='airbnb_review'].groupby('ward').language.apply(lambda s: s.isin(['JA','ZH','KO']).mean()).round(3)
ward_cov['review_cjk_share'] = cjk
ward_cov

,n_airbnb_overview,n_airbnb_review,nlptown_cov_airbnb_overview,nlptown_cov_airbnb_review,review_cjk_share
ward,,,,,
Chiyoda,49,13490,0.980,0.604,0.354
Chuo,58,22740,0.983,0.629,0.336
Itabashi,108,11679,0.991,0.549,0.421
Koto,117,23391,0.966,0.588,0.376
Minato,204,39973,0.975,0.680,0.290
Setagaya,214,33973,0.986,0.638,0.338
Shibuya,422,75415,0.979,0.730,0.240
Shinjuku,766,181348,0.984,0.685,0.279
Sumida,580,129589,0.990,0.555,0.409


## 5. Save

In [31]:
corpus.to_parquet(DATA/'unified_corpus.parquet', index=False)
dedup_log.to_parquet(DATA/'overview_dedup_log.parquet', index=False)
(DATA/'stage1_log.json').write_text(json.dumps(LOG, indent=2, default=str))
chk = pd.read_parquet(DATA/'unified_corpus.parquet')
assert len(chk) == len(corpus)
print({f: f'{(DATA/f).stat().st_size/1e6:.1f} MB' for f in
       ['unified_corpus.parquet','overview_dedup_log.parquet','stage1_log.json']})
print(chk.dtypes.to_string())

{'unified_corpus.parquet': '119.4 MB', 'overview_dedup_log.parquet': '0.1 MB', 'stage1_log.json': '0.0 MB'}
text_id                      object
source                       object
ward                         object
language                     object
date                 datetime64[ns]
text                         object
listing_id                    int64
nlptown_supported              bool
approx_tokens                 int64
long_text                      bool


In [32]:
# Free memory from the previous stage (every stage reloads its inputs from disk)
import gc
for _n in ['listings', 'reviews', 'r', 'o', 'o1', 'ov', 'exact_dups', 'near_dups', 'dedup_log', 'rev_u', 'ov_u', 'corpus', 'chk', 'smp', 'smp_lang', 'detector', 'ov_detector', 'out', 'canon', 'xw']:
    globals().pop(_n, None)
gc.collect()
print('freed memory before Stage 2')

freed memory before Stage 2


# Stage 2: Sentiment bake-off
VADER runs here. nlptown and cardiffnlp run in **Stage 2b** below (GPU), or their precomputed outputs are loaded from `data/colab_out/`.

**Design decisions**
- **Ground truth:** a 300-review labeled set, **pre-labeled 1–5 by an LLM (Claude) and audited by a human on 50 reviews**. The audit is blind: the auditor doesn't see the LLM label. Inside Airbnb has no per-review stars, so this is the only per-review truth.
- **Label set composition:** 240 English plus 60 other nlptown languages (FR, ES, DE, IT, NL). VADER and cardiffnlp are English-only models, so the **three-way comparison uses the English 240**. The 60 non-English reviews measure nlptown's multilingual performance on its own. Japanese, Chinese and Korean are out of scope for nlptown and are reported as a coverage gap.
- **Oversampling:** the English set is drawn as 80 VADER-negative, 40 neutral and 120 positive, so 1–3 star cases exist. The set therefore does **not** reflect the natural class mix; accuracy here describes a harder set than the real data.
- **Strata:** round-robin across ward × length tertile, so every ward and length band is represented. Reviews have at least 3 words, so they can be labeled.
- **VADER → 1–5 mapping:** fixed equal-width cut points on the compound score (−0.6, −0.2, 0.2, 0.6), **chosen before seeing any labels**. The 3-class mapping uses VADER's standard ±0.05.
- **3-class collapse:** 1–2 = neg, 3 = neu, 4–5 = pos. cardiffnlp is natively 3-class, so it's compared only on the 3-class task.
- **Pre-specified winner rule:** highest **3-class weighted F1 on the English set**, with paired-bootstrap CIs and McNemar. Coverage (nlptown's 6 languages vs English only) is reported alongside, not folded into the score.
- **Sampling-weighted estimates (section 6d):** because the English set was drawn using VADER, the unweighted metrics favour VADER. Each English item is weighted by N/n of its sampling cell (VADER bucket × ward × length tertile), which estimates performance on the full English review population. CIs come from a bootstrap that resamples within each VADER bucket. Both versions are reported; the weighted ones are noisier (see effective sample size).
- **Colab subsample:** two-stage and stratified by ward. Up to 300 listings per ward are sampled from listings with ≥ 5 nlptown-language reviews and a non-null rating, then up to 10 such reviews per listing. This serves the listing-level Spearman check, which needs several scored reviews per listing. Seed 42.

In [33]:
import sys, json, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd
from labeling import round_robin, vader_bucket, length_bin
from metrics import (vader_to_5, vader_to_3, collapse3, eval_5, eval_3, confusion_df,
                     paired_bootstrap, mcnemar_test, kish_ess, weighted_boot_ci, weighted_paired_diff)
from textclean import NLPTOWN_LANGS
from sklearn.metrics import f1_score, cohen_kappa_score
from scipy.stats import spearmanr

SEED = 42
DATA, LAB, CIN, COUT = ROOT/'data/interim', ROOT/'data/labels', ROOT/'data/colab_in', ROOT/'data/colab_out'
for d in (LAB, CIN, COUT): d.mkdir(parents=True, exist_ok=True)
for p in ['vaderSentiment','scikit-learn','scipy','statsmodels','pandas']: print(f'{p:16s}', md.version(p))

corpus = pd.read_parquet(DATA/'unified_corpus.parquet')
listings = pd.read_parquet(DATA/'listings_pruned.parquet',
                           columns=['listing_id','ward','review_scores_rating','review_scores_location'])
rev = corpus[corpus.source == 'airbnb_review'].copy()
rev['n_words'] = rev.text.str.split().str.len()
print('reviews:', len(rev), '| nlptown-supported:', rev.nlptown_supported.sum(), '| English:', (rev.language=='EN').sum())

vaderSentiment   3.3.2
scikit-learn     1.6.1
scipy            1.16.3
statsmodels      0.15.0
pandas           2.2.3


reviews: 735752 | nlptown-supported: 456989 | English: 406075


## 1. VADER on all English reviews
VADER is a lexicon built for English, so it isn't run on other languages.

In [34]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()
en = rev[rev.language == 'EN']
vs = pd.DataFrame({'text_id': en.text_id.values,
                   'vader_compound': [vader.polarity_scores(t)['compound'] for t in en.text]})
vs['vader5'], vs['vader3'] = vader_to_5(vs.vader_compound), vader_to_3(vs.vader_compound)
vs.to_parquet(DATA/'vader_scores_en.parquet', index=False)
print('VADER scored:', len(vs))
print('3-class share (0=neg,1=neu,2=pos):', vs.vader3.value_counts(normalize=True).sort_index().round(3).to_dict())
print('5-class share:', vs.vader5.value_counts(normalize=True).sort_index().round(3).to_dict())

VADER scored: 406075
3-class share (0=neg,1=neu,2=pos): {0: 0.02, 1: 0.019, 2: 0.96}
5-class share: {1: 0.007, 2: 0.009, 3: 0.029, 4: 0.108, 5: 0.847}


## 2. Draw the 300-review label set

In [35]:
pool = rev[rev.nlptown_supported & (rev.n_words >= 3)].merge(vs[['text_id','vader_compound']], on='text_id', how='left')
en_pool = pool[pool.language == 'EN'].copy()
edges = en_pool.n_words.quantile([1/3, 2/3]).tolist()
pool['len_bin'] = length_bin(pool.n_words, edges).astype(str)
pool['vbucket'] = np.where(pool.language == 'EN', vader_bucket(pool.vader_compound.fillna(0)), 'n/a')
pool['stratum'] = pool.ward + '|' + pool.len_bin

EN_TARGET = {'neg': 80, 'neu': 40, 'pos': 120}
NONEN_TARGET = {'FR': 20, 'ES': 14, 'DE': 14, 'IT': 6, 'NL': 6}
parts = [round_robin(pool[(pool.language=='EN') & (pool.vbucket==b)], n, 'stratum', SEED).assign(stratum_group=f'EN_{b}')
         for b, n in EN_TARGET.items()]
parts += [round_robin(pool[pool.language==l], n, 'stratum', SEED).assign(stratum_group=l) for l, n in NONEN_TARGET.items()]
labelset = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
labelset.insert(0, 'item', np.arange(1, len(labelset)+1))
assert len(labelset) == 300 and labelset.text_id.is_unique

# The embedded labels belong to one specific draw. If a different library version (e.g. the language detector)
# changes the pool, rebuild the labeled set from the embedded membership instead of stopping.
design = pd.read_csv(LAB/'labelset_design.csv')
if set(labelset.text_id) != set(design.text_id):
    print(f'NOTE: this draw differs from the labeled set ({len(set(labelset.text_id) - set(design.text_id))} items); '
          'using the embedded label-set membership.')
    src_rows = rev.drop(columns='language').merge(vs[['text_id','vader_compound']], on='text_id', how='left')
    labelset = design.merge(src_rows, on='text_id', how='left', validate='1:1')
    assert labelset.text.notna().all(), 'labeled reviews missing from the corpus'
    labelset['len_bin'] = length_bin(labelset.n_words, edges).astype(str)
    labelset['vbucket'] = np.where(labelset.stratum_group.str.startswith('EN_'), labelset.stratum_group.str[3:], 'n/a')
    labelset['stratum'] = labelset.ward + '|' + labelset.len_bin
print('length tertile edges (words):', edges)
print(labelset.stratum_group.value_counts().to_string())
print(pd.crosstab(labelset.ward, labelset.len_bin).to_string())
cols = ['item','text_id','listing_id','ward','language','len_bin','stratum_group','vader_compound','text']
labelset[cols].to_parquet(LAB/'labelset_candidates.parquet', index=False)

length tertile edges (words): [21.0, 49.0]
stratum_group
EN_pos    120
EN_neg     80
EN_neu     40
FR         20
ES         14
DE         14
IT          6
NL          6
len_bin   long  medium  short
ward                         
Chiyoda      6       7      6
Chuo         6       8      7
Itabashi     7       6      7
Koto         8       7      9
Minato       8       9      8
Setagaya     8      10      6
Shibuya     10       9     10
Shinjuku    11      13     13
Sumida      12      11     13
Taito       11      10     12
Toshima      9      13     10


## 3. LLM pre-labels (rubric)
Labels are in `data/labels/llm_labels.csv` (`text_id, llm_label`). They were produced by Claude reading each full review against this rubric:

| Star | Meaning |
|---|---|
| 5 | Clearly positive and enthusiastic; no meaningful complaint |
| 4 | Positive overall, but plain or mild praise ("good", "nice place"), or a minor caveat |
| 3 | Mixed or neutral: real pros and real cons, or purely factual with no evaluation |
| 2 | Mostly negative, with some redeeming points |
| 1 | Clearly negative: serious problems, would not recommend |

The guest's overall verdict on the stay decides the label. Complaints about things outside the host's control count only if the guest's tone is negative.

In [36]:
lab_file = LAB/'llm_labels.csv'
HAVE_LABELS = lab_file.exists()
if HAVE_LABELS:
    llm = pd.read_csv(lab_file)
    labelset = labelset.merge(llm, on='text_id', how='left', validate='1:1')
    assert labelset.llm_label.notna().all() and labelset.llm_label.between(1, 5).all()
    print('LLM label distribution:', labelset.llm_label.value_counts().sort_index().to_dict())
    print(pd.crosstab(labelset.stratum_group, labelset.llm_label).to_string())
else:
    print('llm_labels.csv not found yet - label set drawn, waiting for labels.')

LLM label distribution: {1: 15, 2: 33, 3: 51, 4: 83, 5: 118}
llm_label       1   2   3   4   5
stratum_group                    
DE              1   0   0   5   8
EN_neg         13  29  23  10   5
EN_neu          1   2  17  19   1
EN_pos          0   1   6  28  85
ES              0   0   1   6   7
FR              0   0   1  12   7
IT              0   1   0   0   5
NL              0   0   3   3   0


## 4. Blind audit sheet (50 items) and kappa
The audit sample is stratified by LLM label (up to 10 per star value), so low-star labels get checked too. The sheet hides the LLM label. Fill in `rakim_label` (1–5) using the rubric, the completed audit is embedded above as `audit_completed_stage2.csv`.
**Threshold:** quadratic-weighted Cohen kappa ≥ 0.6. Below that, we revise the rubric and re-label the disagreements.

In [37]:
if HAVE_LABELS:
    audit = round_robin(labelset, 50, 'llm_label', SEED).sample(frac=1, random_state=SEED+1).reset_index(drop=True)
    audit.insert(0, 'audit_id', np.arange(1, 51))
    sheet = audit[['audit_id','text_id','language','text']].assign(rakim_label='', notes='')
    with pd.ExcelWriter(LAB/'audit_sheet_BLIND.xlsx', engine='openpyxl') as xw:
        sheet.to_excel(xw, sheet_name='audit', index=False)
        pd.DataFrame({'star':[5,4,3,2,1], 'meaning':[
            'Clearly positive and enthusiastic; no meaningful complaint',
            'Positive overall, but plain or mild praise, or a minor caveat',
            'Mixed or neutral: real pros and cons, or purely factual',
            'Mostly negative, with some redeeming points',
            'Clearly negative: serious problems, would not recommend']}).to_excel(xw, sheet_name='rubric', index=False)
        ws = xw.sheets['audit']; ws.column_dimensions['D'].width = 100
        from openpyxl.styles import Alignment
        for row in ws.iter_rows(min_row=2, min_col=4, max_col=4):
            for c in row: c.alignment = Alignment(wrap_text=True, vertical='top')
    audit[['audit_id','text_id','llm_label']].to_csv(LAB/'audit_key_DO_NOT_OPEN.csv', index=False)
    print('audit sheet written; LLM label mix in audit:', audit.llm_label.value_counts().sort_index().to_dict())

audit sheet written; LLM label mix in audit: {1: 10, 2: 10, 3: 10, 4: 10, 5: 10}


In [38]:
done = LAB/'audit_completed_stage2.csv'
if done.exists():
    a = pd.read_csv(done).merge(labelset[['text_id','text']], on='text_id', how='left').merge(pd.read_csv(LAB/'audit_key_DO_NOT_OPEN.csv'), on=['audit_id','text_id'])
    # The auditor reads English only: non-English rows were marked 3 with an 'unreadable' note or left blank,
    # so they are not ratings. Kappa uses English rows with a 1-5 label; the 60 non-English labels stay unaudited.
    n_all = len(a)
    a = a[(a.language == 'EN') & pd.to_numeric(a.rakim_label, errors='coerce').between(1,5)]
    print(f'audit rows used: {len(a)} of {n_all} (English only)')
    a['rakim_label'] = a.rakim_label.astype(int)
    qwk = cohen_kappa_score(a.rakim_label, a.llm_label, weights='quadratic')
    lwk = cohen_kappa_score(a.rakim_label, a.llm_label, weights='linear')
    rng = np.random.default_rng(SEED)
    bs = [cohen_kappa_score(a.rakim_label.values[i], a.llm_label.values[i], weights='quadratic')
          for i in (rng.integers(0, len(a), len(a)) for _ in range(2000))]
    print(f'QWK bootstrap 95% CI: {np.nanpercentile(bs, 2.5):.3f} to {np.nanpercentile(bs, 97.5):.3f}')
    print(pd.crosstab(a.rakim_label, a.llm_label, rownames=['rakim'], colnames=['claude']).to_string())
    print(f'n audited = {len(a)} | quadratic-weighted kappa = {qwk:.3f} | linear-weighted = {lwk:.3f} | exact agreement = {(a.rakim_label==a.llm_label).mean():.1%}')
    print('PASS (>= 0.6)' if qwk >= 0.6 else 'BELOW 0.6: revisit rubric and re-label disagreements')
    print(a.loc[a.rakim_label != a.llm_label, ['audit_id','rakim_label','llm_label','text']].to_string(max_colwidth=90))
else:
    print('audit_completed_stage2.csv not found - run the embedded-labels cell')

audit rows used: 38 of 50 (English only)


QWK bootstrap 95% CI: 0.948 to 1.000
claude  1  2  3  4  5
rakim                
1       8  0  0  0  0
2       1  9  1  0  0
3       0  0  6  0  0
4       0  0  0  7  1
5       0  0  0  0  5
n audited = 38 | quadratic-weighted kappa = 0.979 | linear-weighted = 0.949 | exact agreement = 92.1%
PASS (>= 0.6)
    audit_id  rakim_label  llm_label                                                                                       text
6          7            2          1  Near station and convenience store is the only advantage I found. The AC Dusty and can...
16        17            2          3  A very clean and convenient room that is approximately 5-minute walk from Morishita st...
25        26            4          5  chisaki was helpful, responsive and gave clear instructions for everything. noise come...


## 5. Export inputs for the Colab notebook

In [39]:
elig = pool.groupby('listing_id').size().rename('n_sup').reset_index()           # supported, >=3 words
elig = elig.merge(listings, on='listing_id')
elig = elig[(elig.n_sup >= 5) & elig.review_scores_rating.notna()]
def take_per_group(df, key, n, seed):
    # Up to n random rows per group (shuffle with a fixed seed, keep the first n per group)
    d = df.sample(frac=1, random_state=seed)
    return d[d.groupby(key).cumcount() < n]
pick_l = take_per_group(elig, 'ward', 300, SEED)
cand = pool[pool.listing_id.isin(pick_l.listing_id) & ~pool.text_id.isin(labelset.text_id)]
sample = take_per_group(cand, 'listing_id', 10, SEED)
sample = sample[['text_id','listing_id','ward','language','text']].reset_index(drop=True)
assert sample.text_id.is_unique and not sample.text_id.isin(labelset.text_id).any()
print(f'Colab subsample: {len(sample):,} reviews from {sample.listing_id.nunique():,} listings | seed {SEED}')
print(pd.DataFrame({'listings': sample.groupby('ward').listing_id.nunique(), 'reviews': sample.ward.value_counts()}).to_string())
print('language mix:', sample.language.value_counts().to_dict())
labelset[['text_id','language','text']].to_parquet(CIN/'labelset_texts.parquet', index=False)
sample.to_parquet(CIN/'colab_sample.parquet', index=False)
json.dump({'seed': SEED, 'n_sample': len(sample), 'n_listings': int(sample.listing_id.nunique()),
           'design': 'up to 300 listings/ward (>=5 nlptown-language reviews, rating not null), up to 10 reviews/listing'},
          open(CIN/'colab_sample_meta.json','w'), indent=2)

Colab subsample: 28,999 reviews from 3,088 listings | seed 42
          listings  reviews
ward                       
Chiyoda        183     1697
Chuo           242     2325
Itabashi       263     2433
Koto           300     2849
Minato         300     2822
Setagaya       300     2800
Shibuya        300     2854
Shinjuku       300     2858
Sumida         300     2779
Taito          300     2812
Toshima        300     2770
language mix: {'EN': 25498, 'FR': 1528, 'DE': 842, 'ES': 826, 'IT': 209, 'NL': 96}


## Stage 2b (GPU): nlptown and cardiffnlp scoring
Runs only if the four `sent_*.parquet` files are missing from `data/colab_out/`. Needs a GPU and Hugging Face access (e.g. Colab with a T4). It uses the tested scoring core in `src/colab_sentiment_core.py`: texts over 510 tokens are sentence-chunked, with token-weighted chunk averaging.

In [40]:
import re, time
from colab_sentiment_core import score_corpus
GPU2_FILES = ['sent_nlptown_labelset.parquet','sent_nlptown_sample.parquet','sent_cardiff_labelset.parquet','sent_cardiff_sample.parquet']
todo2 = [f for f in GPU2_FILES if not (COUT/f).exists()]
NLPTOWN_MODEL = os.environ.get('NLPTOWN_MODEL', 'nlptown/bert-base-multilingual-uncased-sentiment')
CARDIFF_MODEL = os.environ.get('CARDIFF_MODEL', 'cardiffnlp/twitter-roberta-base-sentiment-latest')
ALLOW_CPU = os.environ.get('CAPSTONE_ALLOW_CPU') == '1'   # CPU works but is very slow on the full inputs
def need_gpu(step):
    import torch
    if not (torch.cuda.is_available() or ALLOW_CPU):
        raise RuntimeError(f'{step} needs a GPU: on Colab choose Runtime > Change runtime type > T4 GPU, '
                           'or restore the precomputed files in data/colab_out/ (set CAPSTONE_ALLOW_CPU=1 to force CPU).')
def load_hf(name):
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSequenceClassification.from_pretrained(name).to(dev).eval()
    if dev == 'cuda': model = model.half()
    @torch.inference_mode()
    def pp(texts):
        enc = tok(texts, padding=True, truncation=True, max_length=512, return_tensors='pt').to(dev)
        return torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy()
    ct = lambda texts: [len(x) for x in tok(texts, add_special_tokens=False, truncation=False)['input_ids']]
    return model, pp, ct
if not todo2:
    print('precomputed Stage 2b outputs found - skipping GPU scoring')
else:
    import torch
    need_gpu('Stage 2b')
    ls_in, sm_in = pd.read_parquet(CIN/'labelset_texts.parquet'), pd.read_parquet(CIN/'colab_sample.parquet')
    P5 = ['p1','p2','p3','p4','p5']
    model, pp, ct = load_hf(NLPTOWN_MODEL)
    assert [int(re.match(r'\s*(\d)', model.config.id2label[i]).group(1)) for i in range(5)] == [1,2,3,4,5], model.config.id2label
    for name, df in [('labelset', ls_in), ('sample', sm_in)]:
        f = COUT/f'sent_nlptown_{name}.parquet'
        if f.exists(): continue
        o = score_corpus(df, pp, ct, P5); o['pred_star'] = o[P5].values.argmax(axis=1) + 1; o['exp_star'] = o[P5].values @ np.arange(1, 6)
        o.to_parquet(f, index=False); print('saved', f.name)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    model, pp, ct = load_hf(CARDIFF_MODEL)
    raw = [model.config.id2label[i].lower() for i in range(model.config.num_labels)]
    if all(l.startswith('label_') for l in raw): raw = ['negative','neutral','positive']
    NAME = {'negative':'p_neg','neutral':'p_neu','positive':'p_pos'}; assert set(raw) == set(NAME), model.config.id2label
    COLS = [NAME[l] for l in raw]
    for name, df in [('labelset', ls_in), ('sample', sm_in)]:
        f = COUT/f'sent_cardiff_{name}.parquet'
        if f.exists(): continue
        o = score_corpus(df[df.language == 'EN'], pp, ct, COLS)[['text_id','p_neg','p_neu','p_pos','n_chunks','n_tokens']]
        o['pred3'] = o[['p_neg','p_neu','p_pos']].values.argmax(axis=1); o['score'] = o.p_pos - o.p_neg
        o.to_parquet(f, index=False); print('saved', f.name)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()

precomputed Stage 2b outputs found - skipping GPU scoring


## 6. Evaluation
The functions below run on whatever predictions exist. Cell 6b tests them on **synthetic stub predictions**; those numbers are not results.

In [41]:
def load_preds():
    """Return the label-set table with every available model's predictions."""
    d = labelset[['text_id','language','llm_label']].merge(vs[['text_id','vader_compound','vader5','vader3']], on='text_id', how='left')
    f = COUT/'sent_nlptown_labelset.parquet'
    if f.exists():
        d = d.merge(pd.read_parquet(f)[['text_id','pred_star','exp_star']].rename(columns={'pred_star':'nlptown5','exp_star':'nlptown_exp'}), on='text_id', how='left')
    f = COUT/'sent_cardiff_labelset.parquet'
    if f.exists():
        d = d.merge(pd.read_parquet(f)[['text_id','pred3','score']].rename(columns={'pred3':'cardiff3','score':'cardiff_score'}), on='text_id', how='left')
    return d

def run_bakeoff(d):
    d = d.copy(); d['y3'] = collapse3(d.llm_label)
    if 'nlptown5' in d: d['nlptown3'] = collapse3(d.nlptown5)
    e = d[d.language == 'EN']
    rows5, rows3 = {}, {}
    rows5['VADER (EN)'] = eval_5(e.llm_label, e.vader5);  rows3['VADER (EN)'] = eval_3(e.y3, e.vader3)
    if 'nlptown5' in d:
        rows5['nlptown (EN)'] = eval_5(e.llm_label, e.nlptown5); rows3['nlptown (EN)'] = eval_3(e.y3, e.nlptown3)
        ne = d[d.language != 'EN']
        rows5['nlptown (non-EN)'] = eval_5(ne.llm_label, ne.nlptown5); rows3['nlptown (non-EN)'] = eval_3(ne.y3, ne.nlptown3)
        rows5['nlptown (all 300)'] = eval_5(d.llm_label, d.nlptown5); rows3['nlptown (all 300)'] = eval_3(d.y3, d.nlptown3)
    if 'cardiff3' in d:
        rows3['cardiff (EN)'] = eval_3(e.y3, e.cardiff3)
    t5, t3 = pd.DataFrame(rows5).T.round(3), pd.DataFrame(rows3).T.round(3)
    wf1 = lambda y, p: f1_score(y, p, average='weighted', zero_division=0)
    acc = lambda y, p: (y == p).mean()
    qwk5 = lambda y, p: cohen_kappa_score(y, p, weights='quadratic', labels=[1,2,3,4,5])
    tests = []
    models3 = [m for m in ['vader3','nlptown3','cardiff3'] if m in e]
    for i, a in enumerate(models3):
        for b in models3[i+1:]:
            bs = paired_bootstrap(e.y3, e[a], e[b], wf1, seed=SEED); mc = mcnemar_test(e.y3, e[a], e[b])
            tests.append({'task':'3-class EN', 'A':a, 'B':b, 'wF1 diff (A-B)':bs['diff'], 'CI low':bs['ci_low'], 'CI high':bs['ci_high'],
                          'boot p':bs['p_value'], 'McNemar p':mc['p_value'], 'A-only correct':mc['a_only_correct'], 'B-only correct':mc['b_only_correct']})
    if 'nlptown5' in e:
        bs = paired_bootstrap(e.llm_label, e.nlptown5, e.vader5, qwk5, seed=SEED); mc = mcnemar_test(e.llm_label, e.nlptown5, e.vader5)
        tests.append({'task':'5-class EN (QWK)', 'A':'nlptown5', 'B':'vader5', 'wF1 diff (A-B)':bs['diff'], 'CI low':bs['ci_low'], 'CI high':bs['ci_high'],
                      'boot p':bs['p_value'], 'McNemar p':mc['p_value'], 'A-only correct':mc['a_only_correct'], 'B-only correct':mc['b_only_correct']})
    tt = pd.DataFrame(tests).round(4)
    if len(tt): tt = tt.rename(columns={'wF1 diff (A-B)': 'diff (A-B)'})
    cms = {'VADER 3-class (EN)': confusion_df(e.y3, e.vader3, [0,1,2]), 'VADER 5-class (EN)': confusion_df(e.llm_label, e.vader5, [1,2,3,4,5])}
    if 'nlptown5' in d: cms['nlptown 5-class (all 300)'] = confusion_df(d.llm_label, d.nlptown5, [1,2,3,4,5])
    if 'cardiff3' in d: cms['cardiff 3-class (EN)'] = confusion_df(e.y3, e.cardiff3, [0,1,2])
    winner = t3.loc[[i for i in t3.index if i.endswith('(EN)')], 'weighted_f1'].idxmax()
    return t5, t3, tt, cms, winner

In [42]:
# 6b. STUB TEST of the evaluation path: synthetic predictions, NOT results
if HAVE_LABELS:
    _rng = np.random.default_rng(0)
    _d = load_preds().drop(columns=[c for c in ['nlptown5','nlptown_exp','cardiff3','cardiff_score'] if c in load_preds()])
    _d['nlptown5'] = np.clip(_d.llm_label + _rng.integers(-1, 2, len(_d)), 1, 5)
    _d['cardiff3'] = collapse3(np.clip(_d.llm_label + _rng.integers(-1, 2, len(_d)), 1, 5))
    _t5, _t3, _tt, _cms, _w = run_bakeoff(_d)
    assert {'VADER (EN)','nlptown (EN)','nlptown (non-EN)','cardiff (EN)'} <= set(_t3.index) and len(_tt) == 4
    print('stub evaluation path OK (synthetic numbers suppressed)')

stub evaluation path OK (synthetic numbers suppressed)


## 6c. Bake-off results

In [43]:
if HAVE_LABELS:
    preds = load_preds()
    have = [m for m in ['nlptown5','cardiff3'] if m in preds]
    print('models available:', ['vader'] + have)
    t5, t3, tt, cms, winner = run_bakeoff(preds)
    print('\n5-class (1-5 ordinal):'); print(t5.to_string())
    print('\n3-class collapse:');      print(t3.to_string())
    if len(tt): print('\nPaired tests:'); print(tt.to_string(index=False))
    for k, v in cms.items(): print(f'\n{k}'); print(v.to_string())
    print('\nWinner by pre-specified rule (3-class weighted F1, English set):', winner,
          '' if have else '(provisional: transformer outputs not uploaded yet)')

models available: ['vader', 'nlptown5', 'cardiff3']



5-class (1-5 ordinal):
                       n  accuracy  weighted_f1    mae    qwk
VADER (EN)         240.0     0.579        0.537  0.529  0.785
nlptown (EN)       240.0     0.667        0.669  0.375  0.844
nlptown (non-EN)    60.0     0.750        0.733  0.267  0.779
nlptown (all 300)  300.0     0.683        0.681  0.353  0.845

3-class collapse:
                       n  accuracy  weighted_f1  macro_f1    qwk
VADER (EN)         240.0     0.717        0.724     0.635  0.678
nlptown (EN)       240.0     0.858        0.854     0.805  0.850
nlptown (non-EN)    60.0     0.917        0.909     0.654  0.635
nlptown (all 300)  300.0     0.870        0.865     0.800  0.844
cardiff (EN)       240.0     0.858        0.852     0.792  0.853

Paired tests:
            task        A        B  diff (A-B)  CI low  CI high  boot p  McNemar p  A-only correct  B-only correct
      3-class EN   vader3 nlptown3     -0.1304 -0.1921  -0.0696   0.000     0.0000              14              48
      3-clas

## 6d. Sampling-weighted estimates (English set)
Weight = population count / sample count in the item's sampling cell (VADER bucket × ward × length tertile). If any populated cell had no sampled item, the code falls back to bucket-level weights and says so. **Kish effective sample size** shows how much information is left after weighting.

In [44]:
if HAVE_LABELS:
    preds = load_preds()
    en_ls = labelset[labelset.language == 'EN'][['text_id','vbucket','stratum']].merge(preds, on='text_id')
    pop = pool[pool.language == 'EN'].groupby(['vbucket','stratum']).size().rename('N')
    got = en_ls.groupby(['vbucket','stratum']).size().rename('n')
    cells = pd.concat([pop, got], axis=1)
    if cells.n.isna().any():
        print(f'{int(cells.n.isna().sum())} populated cells unsampled -> using bucket-level weights')
        pop = pool[pool.language == 'EN'].groupby('vbucket').size().rename('N'); got = en_ls.groupby('vbucket').size().rename('n')
        en_ls = en_ls.join(pd.concat([pop, got], axis=1), on='vbucket')
    else:
        en_ls = en_ls.join(cells, on=['vbucket','stratum'])
    en_ls['w'] = en_ls.N / en_ls.n
    assert abs(en_ls.w.sum() - (pool.language == 'EN').sum()) < 1e-6, 'weights must sum to the English population'
    print(f"English population represented: {en_ls.w.sum():,.0f} reviews | n = {len(en_ls)} | Kish ESS = {kish_ess(en_ls.w):.1f}")
    print('weight range by VADER bucket:'); print(en_ls.groupby('vbucket').w.agg(['min','median','max']).round(1).to_string())
    en_ls['y3'] = collapse3(en_ls.llm_label)
    pop_mix = en_ls.groupby('llm_label').w.sum() / en_ls.w.sum()
    print('\nEstimated population mix of LLM labels (weighted) vs label set (unweighted):')
    print(pd.DataFrame({'weighted_population': pop_mix, 'label_set': en_ls.llm_label.value_counts(normalize=True)}).sort_index().round(3).to_string())

English population represented: 397,542 reviews | n = 240 | Kish ESS = 82.5
weight range by VADER bucket:
           min  median     max
vbucket                       
neg        6.0    74.0   466.3
neu        5.0   124.5   789.0
pos      521.7  2080.2  9070.5

Estimated population mix of LLM labels (weighted) vs label set (unweighted):
           weighted_population  label_set
llm_label                                
1                        0.003      0.058
2                        0.011      0.133
3                        0.069      0.192
4                        0.257      0.238
5                        0.659      0.379


In [45]:
if HAVE_LABELS:
    S = en_ls.vbucket.values
    if 'nlptown5' in en_ls: en_ls['nlptown3'] = collapse3(en_ls.nlptown5)
    specs = [('VADER', '5', 'llm_label', 'vader5'), ('VADER', '3', 'y3', 'vader3')]
    if 'nlptown5' in en_ls: specs += [('nlptown', '5', 'llm_label', 'nlptown5'), ('nlptown', '3', 'y3', 'nlptown3')]
    if 'cardiff3' in en_ls: specs += [('cardiff', '3', 'y3', 'cardiff3')]
    rows = []
    for model, kind, ycol, pcol in specs:
        unw = (eval_5 if kind == '5' else eval_3)(en_ls[ycol], en_ls[pcol])
        ci = weighted_boot_ci(en_ls[ycol], en_ls[pcol], en_ls.w, S, kind, seed=SEED)
        for m, (pt, lo, hi) in ci.items():
            rows.append({'model': model, 'task': f'{kind}-class', 'metric': m, 'unweighted': unw[m],
                         'weighted': pt, 'w_ci_low': lo, 'w_ci_high': hi})
    wtbl = pd.DataFrame(rows).round(3)
    print(wtbl.to_string(index=False))
    m3 = [c for c in ['vader3','nlptown3','cardiff3'] if c in en_ls]
    wtests = []
    for i, a in enumerate(m3):
        for b in m3[i+1:]:
            r = weighted_paired_diff(en_ls.y3, en_ls[a], en_ls[b], en_ls.w, S, '3', 'weighted_f1', seed=SEED)
            wtests.append({'A': a, 'B': b, **r})
    if wtests:
        print('\nWeighted paired bootstrap, 3-class weighted F1 (A - B):'); print(pd.DataFrame(wtests).round(4).to_string(index=False))
    else:
        print('\n(weighted model-vs-model tests run once transformer outputs are uploaded)')

  model    task      metric  unweighted  weighted  w_ci_low  w_ci_high
  VADER 5-class    accuracy       0.579     0.709     0.604      0.801
  VADER 5-class weighted_f1       0.537     0.658     0.526      0.771
  VADER 5-class         qwk       0.785     0.538     0.381      0.692
  VADER 5-class         mae       0.529     0.332     0.217      0.464
  VADER 3-class    accuracy       0.717     0.921     0.860      0.966
  VADER 3-class weighted_f1       0.724     0.902     0.816      0.965
  VADER 3-class         qwk       0.678     0.569     0.408      0.749
  VADER 3-class    macro_f1       0.635     0.601     0.536      0.686
nlptown 5-class    accuracy       0.667     0.754     0.663      0.841
nlptown 5-class weighted_f1       0.669     0.758     0.666      0.843
nlptown 5-class         qwk       0.844     0.698     0.543      0.814
nlptown 5-class         mae       0.375     0.261     0.167      0.361
nlptown 3-class    accuracy       0.858     0.952     0.907      0.986
nlptow


Weighted paired bootstrap, 3-class weighted F1 (A - B):
       A        B    diff  ci_low  ci_high  p_value
  vader3 nlptown3 -0.0484 -0.1273   0.0074    0.122
  vader3 cardiff3 -0.0117 -0.0190  -0.0046    0.002
nlptown3 cardiff3  0.0367 -0.0181   0.1151    0.319


## 7. External check: listing-level Spearman
Per-listing mean predicted sentiment vs `review_scores_rating` and `review_scores_location`. Listings need ≥ 5 scored reviews. This is an **indirect** check: listing scores are on a different scale, are cumulative, and are dominated by 4.7–5.0 ratings, so expect modest correlations. Location scores should correlate more weakly than overall ratings, because review sentiment mixes many aspects.

In [46]:
def listing_spearman(scores, value_col, label, min_reviews=5):
    s = scores.merge(corpus[['text_id','listing_id']], on='text_id')
    g = s.groupby('listing_id')[value_col].agg(['mean','size']).query('size >= @min_reviews').join(listings.set_index('listing_id'))
    out = {'signal': label, 'n_listings': len(g)}
    for col in ['review_scores_rating','review_scores_location']:
        gg = g.dropna(subset=[col]); rho, p = spearmanr(gg['mean'], gg[col])
        out[f'rho_{col.replace("review_scores_","")}'], out[f'p_{col.replace("review_scores_","")}'] = rho, p
    return out

ext = [listing_spearman(vs, 'vader_compound', 'VADER compound, all English reviews')]
smp = pd.read_parquet(CIN/'colab_sample.parquet')
ext.append(listing_spearman(vs[vs.text_id.isin(smp.text_id)], 'vader_compound', 'VADER compound, Colab subsample (EN only)'))
for f, col, lab in [('sent_nlptown_sample.parquet','exp_star','nlptown expected star, Colab subsample'),
                    ('sent_cardiff_sample.parquet','score','cardiff P(pos)-P(neg), Colab subsample (EN only)')]:
    if (COUT/f).exists(): ext.append(listing_spearman(pd.read_parquet(COUT/f), col, lab))
    else: print(f'waiting for Colab output: {f}')
ext_tbl = pd.DataFrame(ext).set_index('signal').round(4); ext_tbl

,n_listings,rho_rating,p_rating,rho_location,p_location
signal,,,,,
"VADER compound, all English reviews",11803,0.6233,0.0,0.3789,0.0
"VADER compound, Colab subsample (EN only)",2979,0.5130,0.0,0.3031,0.0
"nlptown expected star, Colab subsample",3086,0.6680,0.0,0.4222,0.0
"cardiff P(pos)-P(neg), Colab subsample (EN only)",2979,0.5792,0.0,0.3652,0.0


In [47]:
summary = {'seed': SEED, 'labelset_n': 300, 'labelset_design': {'EN': EN_TARGET, 'non_EN': NONEN_TARGET},
           'vader5_edges': [-0.6,-0.2,0.2,0.6], 'external_check': ext_tbl.reset_index().to_dict('records')}
if HAVE_LABELS:
    summary.update({'llm_label_dist': labelset.llm_label.value_counts().sort_index().to_dict(),
                    'results_3class': t3.reset_index().to_dict('records'), 'results_5class': t5.reset_index().to_dict('records'),
                    'winner_rule': '3-class weighted F1, English set', 'winner_current': winner,
                    'weighted': {'kish_ess': kish_ess(en_ls.w), 'table': wtbl.to_dict('records')}})
(DATA/'stage2_summary.json').write_text(json.dumps(summary, indent=2, default=str)); print('saved stage2_summary.json')

saved stage2_summary.json


In [48]:
# Free memory from the previous stage (every stage reloads its inputs from disk)
import gc
for _n in ['corpus', 'listings', 'rev', 'en', 'vs', 'pool', 'en_pool', 'labelset', 'parts', 'elig', 'pick_l', 'cand', 'sample', 'preds', 'smp', 'vader', 'en_ls', '_d']:
    globals().pop(_n, None)
gc.collect()
print('freed memory before Stage 3')

freed memory before Stage 3


# Stage 3: Topic modeling (BERTopic)
**Embeddings:** `paraphrase-multilingual-mpnet-base-v2`, computed in **Stage 3b** below (GPU) or loaded from `data/colab_out/`. Everything else runs on CPU: UMAP, HDBSCAN, c-TF-IDF, tuning and metrics. The identical pipeline also runs on a **TF-IDF + SVD fallback embedding**, which validated the code before the GPU embeddings existed; the fallback topics are not used.

**Design decisions**
- **Unit = sentence, not whole review.** A typical review packs "location, clean, host" into one text, so whole-review topics collapse into a generic junk-drawer topic. Sentences give aspect-level topics, which Stage 4 also needs. Units need ≥ 5 words, and host bullet lists (・ ■ - |) are split too.
- **English only.** c-TF-IDF needs word tokens, and the default tokenizer can't segment Japanese or Chinese (no spaces). This leaves the same Japanese/Chinese/Korean gap as Stage 2 (and drops the ~2% French/German/Spanish/Italian/Dutch), stated as a limitation. A Japanese tokenizer (e.g. janome) is a possible extension.
- **Sample:** 2,500 review sentences per ward (equal weight per ward, seed 42) plus **all** English overview sentences from the deduplicated overviews. Topic shares are reported **by source**, since guest and host voices differ.
- **Domain stopwords** apply to topic *words* only, not embeddings: ward names, tokyo/japan, generic accommodation words (stay, place, apartment, room, location…), and generic praise (great, nice, recommend…). Sentiment belongs to Stage 4. Aspect-bearing words such as *station*, *host*, *clean* and *quiet* are kept. c-TF-IDF also uses `reduce_frequent_words=True`.
- **Tuning grid (final):** `n_neighbors` ∈ {10, 15, 30} × `min_topic_size` (HDBSCAN `min_cluster_size`) ∈ {150, 200, 250, 300}, `min_samples` = 3; UMAP with 5 dimensions, cosine metric, `min_dist` 0, seed 42, fitted once per `n_neighbors` and reused. Configs that can't be fitted are recorded as invalid.
- **Selection rule (v2):** a config is *valid* if its largest topic holds < 20% of units, outliers are ≤ 30%, and it has 15–80 topics. Among valid configs, pick the **highest c_npmi**, breaking ties on diversity.
- **How the grid and rule got here (full history, in order):**
  1. First grid `min_topic_size` ∈ {50, 100, 200} on the fallback embedding gave 85–237 topics, none under the 80-topic cap, so sizes were raised to {100, 200, 400, 800}.
  2. The first rule (largest topic < 40%, outliers ≤ 40%, 8–80 topics) then picked, on the fallback embedding, a degenerate model: 8 topics, one holding 38%, with 40% outliers. c_npmi rewards a few coarse topics, so the rule was tightened to the v2 rule above, and the grid moved to {200, 300, 400, 600} with `min_samples` = 10. This happened **before** the mpnet embeddings were seen.
  3. On mpnet, sizes ≥ 300 merged all "shops / restaurants / convenience stores nearby" sentences into one cluster holding 26–51% of units, and at size 200 the outlier rate sat just above the 30% cap. **After seeing this**, `min_samples` was lowered from 10 to 3 and the sizes were set to {150, 200, 250, 300}. The rule itself did not change, but this grid change was made with knowledge of mpnet results, and the selected model's outlier rate (29.8%) sits just under the cap.
- **Outliers:** the raw −1 rate is the reported metric. For downstream use, outliers are then reassigned with c-TF-IDF similarity (threshold 0.1), and the post-reassignment rate is reported too.

In [49]:
import sys, json, time, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd
from topics import (split_units, make_vectorizer, tfidf_svd_embed, run_grid, select_config,
                    fit_bertopic, topic_words, topic_diversity, coherence, STOPWORDS)
from gensim.corpora import Dictionary
SEED = 42
DATA, CIN, COUT = ROOT/'data/interim', ROOT/'data/colab_in', ROOT/'data/colab_out'
NN_GRID, MCS_GRID, MIN_SAMPLES = [10, 15, 30], [150, 200, 250, 300], 3
RULE = dict(max_largest=0.20, max_outlier=0.30, k_range=(15, 80))   # amended v2, fixed BEFORE mpnet results
REVIEW_UNITS_PER_WARD = 2500
for p in ['bertopic','umap-learn','scikit-learn','gensim','numpy']: print(f'{p:14s}', md.version(p))

bertopic       0.17.4
umap-learn     0.5.12
scikit-learn   1.6.1
gensim         4.4.0
numpy          2.4.6


## 1. Build sentence units

In [50]:
corpus = pd.read_parquet(DATA/'unified_corpus.parquet', columns=['text_id','source','ward','language','text'])
en = corpus[corpus.language == 'EN']
rows = [(tid, src, w, i, u) for tid, src, w, t in zip(en.text_id, en.source, en.ward, en.text)
        for i, u in enumerate(split_units(t))]
units_all = pd.DataFrame(rows, columns=['text_id','source','ward','unit_no','unit_text'])
units_all['unit_id'] = units_all.text_id + '_u' + units_all.unit_no.astype(str)

rv = units_all[units_all.source == 'airbnb_review'].sample(frac=1, random_state=SEED)
rv = rv[rv.groupby('ward').cumcount() < REVIEW_UNITS_PER_WARD]
ov = units_all[units_all.source == 'airbnb_overview']
units = pd.concat([rv, ov]).sort_values('unit_id').reset_index(drop=True)
assert units.unit_id.is_unique
print(f'English texts: {len(en):,} -> units available: {len(units_all):,} -> modeled: {len(units):,}')
print(pd.crosstab(units.ward, units.source, margins=True).to_string())
units[['unit_id','text_id','source','ward','unit_text']].to_parquet(CIN/'topic_units_input.parquet', index=False)
docs = units.unit_text.tolist()

# tokens and dictionary for coherence, built with the same analyzer as the topic representation
analyzer = make_vectorizer().build_analyzer()
tokens = [analyzer(d) for d in docs]
dictionary = Dictionary(tokens)
print('coherence vocabulary:', len(dictionary))

English texts: 409,414 -> units available: 1,346,904 -> modeled: 42,147
source    airbnb_overview  airbnb_review    All
ward                                           
Chiyoda               192           2500   2692
Chuo                  279           2500   2779
Itabashi              503           2500   3003
Koto                  577           2500   3077
Minato                896           2500   3396
Setagaya              911           2500   3411
Shibuya              1953           2500   4453
Shinjuku             3183           2500   5683
Sumida               2476           2500   4976
Taito                1806           2500   4306
Toshima              1871           2500   4371
All                 14647          27500  42147


coherence vocabulary: 13399


## 2. Smoke test (3,000 units, fallback embedding, one config)

In [51]:
t0 = time.time()
_i = np.random.default_rng(SEED).choice(len(docs), 3000, replace=False)
_d = [docs[i] for i in _i]; _tok = [tokens[i] for i in _i]; _dict = Dictionary(_tok)
_g, _ = run_grid(_d, tfidf_svd_embed(_d), [15], [30], MIN_SAMPLES, _tok, _dict, SEED, log=lambda *a: None)
assert len(_g) == 1 and _g.n_topics.iloc[0] > 0
print(f'smoke test OK in {time.time()-t0:.0f}s:', _g.round(3).to_dict('records')[0])

smoke test OK in 21s: {'n_neighbors': 15, 'min_topic_size': 30, 'n_topics': 53, 'outlier_rate': 0.11, 'largest_topic_share': 0.036, 'c_v': 0.441, 'c_npmi': -0.253, 'diversity': 0.73, 'seconds': 5.2}


## Stage 3b (GPU): mpnet sentence embeddings
Runs only if `topic_embeddings_mpnet.npy` is missing from `data/colab_out/`. Uses `sentence-transformers/paraphrase-multilingual-mpnet-base-v2` with normalized embeddings, stored as float16.

In [52]:
if emb_available() and (COUT/'topic_embeddings_ids.parquet').exists():
    print('precomputed mpnet embeddings found - skipping GPU embedding')
else:
    import torch
    need_gpu('Stage 3b')
    from sentence_transformers import SentenceTransformer
    st_model = SentenceTransformer(os.environ.get('MPNET_MODEL', 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'),
                                   device='cuda' if torch.cuda.is_available() else 'cpu')
    _u = pd.read_parquet(CIN/'topic_units_input.parquet')
    _e = st_model.encode(_u.unit_text.tolist(), batch_size=128, show_progress_bar=False, normalize_embeddings=True, convert_to_numpy=True)
    save_embeddings(_e.astype(np.float16)); _u[['unit_id']].to_parquet(COUT/'topic_embeddings_ids.parquet', index=False)
    del st_model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('saved mpnet embeddings', _e.shape)


precomputed mpnet embeddings found - skipping GPU embedding


## 3. Embeddings and tuning grid
The fallback grid always runs, as pipeline validation. The mpnet grid runs on the Stage 3b embeddings, and the final model uses mpnet.

In [53]:
emb_sets = {}
t0 = time.time(); emb_sets['tfidf_svd'] = tfidf_svd_embed(docs); print(f'fallback embedding: {emb_sets["tfidf_svd"].shape} in {time.time()-t0:.0f}s')
f_ids = COUT/'topic_embeddings_ids.parquet'
if emb_available() and f_ids.exists():
    ids = pd.read_parquet(f_ids).unit_id.tolist()
    assert ids == units.unit_id.tolist(), 'embedding rows do not match unit order'
    emb_sets['mpnet'] = load_embeddings().astype(np.float32)
    print('mpnet embeddings:', emb_sets['mpnet'].shape)
else:
    print('mpnet embeddings missing -> final model is PROVISIONAL (fallback)')

grids, umap_cache = {}, {}
for name, E in emb_sets.items():
    print(f'\n=== grid: {name} ===')
    g, cache = run_grid(docs, E, NN_GRID, MCS_GRID, MIN_SAMPLES, tokens, dictionary, SEED)
    grids[name], umap_cache[name] = select_config(g, **RULE)[0].assign(embedding=name), cache
grid_all = pd.concat(grids.values(), ignore_index=True)
grid_all.to_csv(DATA/'stage3_tuning_grid.csv', index=False)
grid_all.round(3)

fallback embedding: (42147, 100) in 2s
mpnet embeddings: (42147, 768)

=== grid: tfidf_svd ===


UMAP n_neighbors=10: 41s


{'n_neighbors': 10, 'min_topic_size': 150, 'n_topics': 97, 'outlier_rate': 0.157, 'largest_topic_share': 0.027, 'c_v': 0.473, 'c_npmi': -0.081, 'diversity': 0.797, 'seconds': 10.2}


{'n_neighbors': 10, 'min_topic_size': 200, 'n_topics': 86, 'outlier_rate': 0.178, 'largest_topic_share': 0.027, 'c_v': 0.485, 'c_npmi': -0.061, 'diversity': 0.801, 'seconds': 10.3}


{'n_neighbors': 10, 'min_topic_size': 250, 'n_topics': 66, 'outlier_rate': 0.205, 'largest_topic_share': 0.027, 'c_v': 0.505, 'c_npmi': -0.023, 'diversity': 0.8, 'seconds': 11.1}


{'n_neighbors': 10, 'min_topic_size': 300, 'n_topics': 59, 'outlier_rate': 0.243, 'largest_topic_share': 0.027, 'c_v': 0.507, 'c_npmi': -0.021, 'diversity': 0.803, 'seconds': 11.3}


... (22 outputs left out)

{'n_neighbors': 30, 'min_topic_size': 200, 'n_topics': 52, 'outlier_rate': 0.32, 'largest_topic_share': 0.066, 'c_v': 0.546, 'c_npmi': 0.052, 'diversity': 0.813, 'seconds': 10.5}


{'n_neighbors': 30, 'min_topic_size': 250, 'n_topics': 24, 'outlier_rate': 0.167, 'largest_topic_share': 0.503, 'c_v': 0.567, 'c_npmi': 0.086, 'diversity': 0.879, 'seconds': 9.0}


{'n_neighbors': 30, 'min_topic_size': 300, 'n_topics': 17, 'outlier_rate': 0.139, 'largest_topic_share': 0.503, 'c_v': 0.559, 'c_npmi': 0.064, 'diversity': 0.935, 'seconds': 9.5}


,n_neighbors,min_topic_size,n_topics,outlier_rate,largest_topic_share,c_v,c_npmi,diversity,seconds,valid,embedding
0,10,150,97,0.157,0.027,0.473,-0.081,0.797,10.2,False,tfidf_svd
1,10,200,86,0.178,0.027,0.485,-0.061,0.801,10.3,False,tfidf_svd
2,10,250,66,0.205,0.027,0.505,-0.023,0.800,11.1,True,tfidf_svd
3,10,300,59,0.243,0.027,0.507,-0.021,0.803,11.3,True,tfidf_svd
4,15,150,108,0.127,0.024,0.459,-0.112,0.796,10.8,False,tfidf_svd
5,15,200,83,0.146,0.029,0.482,-0.081,0.810,11.6,False,tfidf_svd
6,15,250,70,0.190,0.029,0.502,-0.050,0.813,11.2,True,tfidf_svd
7,15,300,59,0.229,0.029,0.514,-0.028,0.808,11.3,True,tfidf_svd
8,30,150,112,0.098,0.024,0.469,-0.108,0.783,10.9,False,tfidf_svd
9,30,200,93,0.124,0.028,0.488,-0.081,0.794,10.1,False,tfidf_svd


## 4. Select and fit the final model

In [54]:
FINAL_EMB = 'mpnet' if 'mpnet' in emb_sets else 'tfidf_svd'
_, best = select_config(grids[FINAL_EMB].drop(columns=['valid','embedding']), **RULE)
if best is None:
    raise RuntimeError('No config passes the validity rule - widen the grid before continuing.')
NN, MCS = int(best.n_neighbors), int(best.min_topic_size)
print(f'final embedding: {FINAL_EMB} | n_neighbors={NN} | min_topic_size={MCS}')
tm, topics_raw = fit_bertopic(docs, umap_cache[FINAL_EMB][NN], MCS, MIN_SAMPLES, SEED)
words = topic_words(tm)
raw_outlier = float((topics_raw == -1).mean())
if (topics_raw == -1).any():
    topics = np.asarray(tm.reduce_outliers(docs, topics_raw.tolist(), strategy='c-tf-idf', threshold=0.1))
else:
    topics = topics_raw.copy()
final_metrics = {'embedding': FINAL_EMB, 'n_neighbors': NN, 'min_topic_size': MCS, 'n_topics': len(words),
                 'outlier_rate_raw': raw_outlier, 'outlier_rate_after_reassign': float((topics == -1).mean()),
                 'largest_topic_share': float(pd.Series(topics_raw[topics_raw != -1]).value_counts().iloc[0] / len(topics_raw)),
                 'c_v': coherence(words, tokens, dictionary, 'c_v'), 'c_npmi': coherence(words, tokens, dictionary, 'c_npmi'),
                 'diversity': topic_diversity(words)}
assert final_metrics['largest_topic_share'] < 0.40, 'junk-drawer topic'
print({k: round(float(v), 3) if isinstance(v, float) else v for k, v in final_metrics.items()})

final embedding: mpnet | n_neighbors=15 | min_topic_size=250


{'embedding': 'mpnet', 'n_neighbors': 15, 'min_topic_size': 250, 'n_topics': 41, 'outlier_rate_raw': 0.298, 'outlier_rate_after_reassign': 0.111, 'largest_topic_share': 0.063, 'c_v': 0.634, 'c_npmi': 0.121, 'diversity': 0.846}


In [55]:
units['topic_raw'], units['topic'] = topics_raw, topics
info = pd.DataFrame({'topic': list(words), 'top_words': [', '.join(w) for w in words.values()]})
info['label'] = [' / '.join(w[:3]) for w in words.values()]
info['n_units'] = info.topic.map(units.topic.value_counts())
src_share = pd.crosstab(units.topic, units.source, normalize='columns')
info['share_of_reviews'] = info.topic.map(src_share.get('airbnb_review', pd.Series(dtype=float))).fillna(0)
info['share_of_overviews'] = info.topic.map(src_share.get('airbnb_overview', pd.Series(dtype=float))).fillna(0)
info['review_vs_overview'] = np.where(info.share_of_reviews > 2*info.share_of_overviews, 'guest voice',
                             np.where(info.share_of_overviews > 2*info.share_of_reviews, 'host voice', 'both'))
# 2 example sentences per topic: random (seeded) sentences of 40-160 characters, after outlier reassignment
ex = units[units.topic != -1].sample(frac=1, random_state=SEED)
ex = ex[ex.unit_text.str.len().between(40, 160)].groupby('topic').unit_text.apply(lambda s: list(s.head(2)))
info['examples'] = info.topic.map(ex)
info = info.sort_values('n_units', ascending=False).reset_index(drop=True)
pd.set_option('display.max_colwidth', 120)
info[['topic','n_units','review_vs_overview','share_of_reviews','share_of_overviews','top_words']].round(3)

,topic,n_units,review_vs_overview,share_of_reviews,share_of_overviews,top_words
0,0,2787,host voice,0.037,0.122,"convenience, supermarkets, stores, supermarket, grocery, restaurants, cafes, store, nearby, shops"
1,1,2407,guest voice,0.072,0.029,"visiting, visit, come, return, standards, explore, base, im, days, looking"
2,3,2356,host voice,0.018,0.127,"line, toei, sta, oedo, jr, harajuku, yamanote, fukutoshin, nearest, metro"
3,2,2043,guest voice,0.063,0.021,"subway, stations, train, close, far, metro, trains, bus, near, walkable"
4,4,1636,guest voice,0.055,0.008,"bathroom, shower, toilet, bath, smell, hot, towels, bathrooms, cold, sink"
5,6,1500,guest voice,0.054,0.002,"adults, family, space, group, people, size, nights, smaller, small, fit"
6,5,1435,guest voice,0.051,0.002,"enjoyed, felt, pleasant, loved, happy, week, letting, enjoyable, weeks, absolutely"
7,7,1433,guest voice,0.052,0.001,"responsive, questions, host, helpful, hosts, quickly, respond, friendly, answered, responded"
8,9,1223,both,0.028,0.031,"quiet, safe, peaceful, environment, residential, calm, neighborhood, neighbourhood, night, bustle"
9,12,1097,both,0.030,0.019,"restaurants, stores, stations, near, convenience, close, subway, grocery, options, shops"


In [56]:
print(info[['topic','label','examples']].head(15).to_string(index=False))

 topic                               label                                                                                                                                                                                                                                          examples
     0 convenience / supermarkets / stores                                                                                                     [Many restaurants and convenience stores nearby., There are restaurants, convenience stores, supermarkets, discount shops, and more around!!]
     1             visiting / visit / come                                                                                          [Thank you and we would love to stay at your place next time we visit Japan ❤️, This is a one-room apartment located between Tokyo Skytree and Asakusa.]
     3                   line / toei / sta                                                                                                       

## 5. Topic-by-ward
**Share** = the topic's share of a ward's units, computed separately for reviews and overviews and then averaged (a topic absent from one source counts as 0 there), so wards with many host overviews aren't dominated by host voice. **Lift** = ward share ÷ average share across all wards; above 1 means the ward over-indexes on that topic.

In [57]:
t = units[units.topic != -1]
# FIXED (audit, 24 Nov): unstack with zeros BEFORE averaging. The earlier version averaged only the sources where the
# topic appeared, which doubled the share of topics missing from one source in a ward (116 of 450 cells).
share_src = t.groupby(['source','ward']).topic.value_counts(normalize=True).unstack(fill_value=0)
share = share_src.groupby(level='ward').mean().sort_index(axis=1)
assert np.allclose(share.sum(axis=1), 1)
lift = share.div(share.mean(axis=0), axis=1)
label = info.set_index('topic').label
top_cols = info.topic.head(15).tolist()
print('Share of units (top 15 topics):')
display(share[top_cols].rename(columns=label).T.round(3))
print('Top 3 over-indexed topics per ward (lift, min share 1%):')
top3 = {w: ', '.join(f"{label[c]} ({lift.loc[w,c]:.1f}x)" for c in lift.loc[w][share.loc[w] >= 0.01].sort_values(ascending=False).index[:3])
        for w in share.index}
pd.Series(top3, name='top over-indexed topics').to_frame()

Share of units (top 15 topics):


ward,Chiyoda,Chuo,Itabashi,Koto,Minato,Setagaya,Shibuya,Shinjuku,Sumida,Taito,Toshima
topic,,,,,,,,,,,
convenience / supermarkets / stores,0.079,0.080,0.087,0.081,0.095,0.117,0.106,0.094,0.066,0.078,0.081
visiting / visit / come,0.047,0.068,0.063,0.067,0.058,0.072,0.053,0.058,0.053,0.052,0.051
line / toei / sta,0.046,0.072,0.080,0.064,0.090,0.098,0.122,0.079,0.060,0.047,0.103
subway / stations / train,0.044,0.052,0.057,0.051,0.039,0.049,0.037,0.050,0.045,0.043,0.053
bathroom / shower / toilet,0.050,0.034,0.047,0.031,0.034,0.025,0.033,0.036,0.035,0.039,0.041
adults / family / space,0.038,0.036,0.025,0.028,0.027,0.025,0.032,0.036,0.030,0.031,0.035
enjoyed / felt / pleasant,0.032,0.027,0.033,0.035,0.029,0.034,0.030,0.032,0.025,0.034,0.023
responsive / questions / host,0.029,0.026,0.034,0.031,0.025,0.024,0.030,0.028,0.033,0.030,0.035
quiet / safe / peaceful,0.017,0.017,0.038,0.034,0.034,0.048,0.037,0.027,0.038,0.022,0.040


Top 3 over-indexed topics per ward (lift, min share 1%):


,top over-indexed topics
Chiyoda,"anime / ikebukuro / manga (2.8x), tree / sky / sensoji (1.7x), noise / noisy / loud (1.5x)"
Chuo,"foot / store / seveneleven (1.8x), solo / budget / price (1.7x), early / luggage / check (1.5x)"
Itabashi,"anime / ikebukuro / manga (1.8x), playground / parks / sports (1.7x), ramen / sushi / japanese (1.6x)"
Koto,"museum / sumo / km (1.8x), tree / sky / sensoji (1.4x), washing / machine / dryer (1.4x)"
Minato,"observation / deck / view (2.2x), garden / gyoen / park (1.8x), airport / narita / haneda (1.6x)"
Setagaya,"crossing / scramble / youth (3.7x), responsive / san / helpful (1.8x), quiet / safe / peaceful (1.5x)"
Shibuya,"crossing / scramble / youth (4.8x), garden / gyoen / park (2.0x), line / toei / sta (1.6x)"
Shinjuku,"shinokubo / okubo / korean (7.6x), kabukicho / kagurazaka / golden (5.6x), noise / noisy / loud (1.3x)"
Sumida,"observation / deck / view (3.0x), tree / sky / sensoji (2.2x), temple / asakusa / gate (2.0x)"
Taito,"temple / asakusa / gate (3.7x), museum / sumo / km (2.3x), tree / sky / sensoji (1.5x)"


## 6. Save

In [58]:
units.to_parquet(DATA/'topic_units.parquet', index=False)
info.to_parquet(DATA/'topic_info.parquet', index=False)
share.to_csv(DATA/'topic_by_ward_share.csv'); lift.to_csv(DATA/'topic_by_ward_lift.csv')
summary = {'seed': SEED, 'n_units': len(units), 'units_by_source': units.source.value_counts().to_dict(),
           'grid': grid_all.round(4).to_dict('records'), 'final': final_metrics,
           'provisional': FINAL_EMB != 'mpnet', 'stopwords_domain_count': len(STOPWORDS)}
(DATA/'stage3_summary.json').write_text(json.dumps(summary, indent=2, default=str))
print('saved:', [f for f in ['topic_units.parquet','topic_info.parquet','topic_by_ward_share.csv','topic_by_ward_lift.csv','stage3_summary.json']])

saved: ['topic_units.parquet', 'topic_info.parquet', 'topic_by_ward_share.csv', 'topic_by_ward_lift.csv', 'stage3_summary.json']


In [59]:
# Free memory from the previous stage (every stage reloads its inputs from disk)
import gc
for _n in ['corpus', 'en', 'rows', 'units_all', 'rv', 'ov', 'units', 'docs', 'tokens', 'dictionary', 'analyzer', 'emb_sets', 'grids', 'umap_cache', 'grid_all', 'tm', 'topics_raw', 'topics', 'ex', 't', 'share_src', 'share', 'lift', '_tok', '_dict', '_d', '_g']:
    globals().pop(_n, None)
gc.collect()
print('freed memory before Stage 4')

freed memory before Stage 4


# Stage 4: Aspect-based sentiment
Units are the 42,147 English sentences from Stage 3, so every unit has keywords, an mpnet embedding and a topic. Aspect assignment runs here; sentiment scoring with **nlptown** (the Stage 2 winner) runs in **Stage 4b** below (GPU) or is loaded from `data/colab_out/`.

**Hybrid assignment (one row per unit × aspect)**, checked in this order:
1. **keyword:** a seed keyword for the aspect occurs in the sentence (English regex; "FamilyMart" and "Sky Tree" are excluded from false matches).
2. **topic+embedding:** the unit's Stage 3 topic maps to the aspect **and** its centred cosine similarity to the aspect centroid is ≥ 0.40.
3. **embedding:** similarity is ≥ 0.55 with no keyword or topic needed. This path is **switched off for safety and family-friendliness**; inspection showed high-similarity non-keyword hits there were mostly wrong ("felt comfortable" for safety, "we were 5 adults" for family).

**v2 after the precision audit (section 4):** a non-keyword path is kept for an aspect only where its audit precision was ≥ 0.75. **topic+embedding** is kept for accommodation, walkability, nature and traditional/cultural; **embedding-only** for accommodation, walkability and crowd level. Keyword fixes: Ginza/Akihabara (place names, not shopping), "lines to" (queues) and "amusement park" (not nature) were removed.

**Keyword-path extension (coverage):** 2,500 review sentences per ward left 31 of 110 ward × aspect cells under 30 mentions. So the keyword path also runs on English review sentences **outside** the Stage 3 sample: up to 30,000 keyword-matched sentences per ward, seed 42. These rows are marked `sample = 'extension'`; the keyword path is the one with the highest audited precision.

**v3 prune (after the sentence-level validation, Stage 4c):** word-sense exclusions are stripped before matching: mini-bar, bar stools, electrical "outlet", "snacks and drinks", microwave-safe, and "park the car". **v4 strict place names (decision by Rakim, 23 Nov):** Asakusa, Meiji, Roppongi, Kabukicho and Yoyogi no longer count on their own, because sentences that only name the area mostly describe the stay, not the aspect. Named attractions (Senso-ji, Kaminarimon, Golden Gai, Gyoen) and activity words (temple, bar, park) still count. The extension sample is frozen from v2, and any clause that changed falls back to its full-sentence score, so no new Colab run is needed.

**Aspect centroids:** the mean mpnet embedding of units whose keywords hit exactly **one** aspect, after centring all embeddings on the corpus mean (otherwise every sentence looks alike). No extra labels or embeddings are needed.

**Sentiment:** nlptown expected star, mapped to **[−1, 1]** as (star − 3) / 2. A sentence that mentions several aspects gets one score for all of them. To reduce that bleed, sentences with a contrastive connective (*but, however, although, …*; 5.9% of units) are split into clauses, and an aspect takes the score **of the clause holding its keyword**.

**Thresholds** (0.40 / 0.55) were set by inspecting similarity distributions and examples, not tuned on an outcome. A precision audit (section 4) checks them per aspect and path.

In [60]:
import sys, json, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd
from absa import (ASPECTS, KEYWORDS, TOPIC_MAP, TOPIC_ANCHOR, TAU_TOPIC, TAU_EMB, TOPIC_OK, EMB_OK,
                  keyword_hits, split_clauses, centroids, assign, pick_clause, to_score, resolve_topic_map)
from topics import split_units
EXT_CAP_PER_WARD = 30000
SEED = 42
DATA, LAB, CIN, COUT = ROOT/'data/interim', ROOT/'data/labels', ROOT/'data/colab_in', ROOT/'data/colab_out'
units = pd.read_parquet(DATA/'topic_units.parquet')
E = load_embeddings().astype(np.float32)
assert pd.read_parquet(COUT/'topic_embeddings_ids.parquet').unit_id.tolist() == units.unit_id.tolist()
info = pd.read_parquet(DATA/'topic_info.parquet').set_index('topic')
TOPIC_MAP_USED = resolve_topic_map(info)   # hand-written map if topic ids match, else rule-based fallback
print('units:', len(units), '| embeddings:', E.shape, '| topic map covers', len(TOPIC_MAP), 'of', len(info), 'topics')

topic ids match the Stage 3 run the map was written for -> using hand-written TOPIC_MAP
units: 42147 | embeddings: (42147, 768) | topic map covers 31 of 41 topics


## 1. Signals: keywords, centroid similarity, topic

In [61]:
kw = [keyword_hits(t) for t in units.unit_text]
H = pd.DataFrame([{a: bool(v[a]) for a in ASPECTS} for v in kw])
Ec, Cent = centroids(E, H)
S = Ec @ Cent.T
print('pure-keyword units used for each centroid:', {a: int((H[a] & (H.sum(axis=1)==1)).sum()) for a in ASPECTS})
print('centroid-to-centroid cosine (lower = more distinct):')
pd.DataFrame(Cent @ Cent.T, index=ASPECTS, columns=[a[:8] for a in ASPECTS]).round(2)

pure-keyword units used for each centroid: {'food_dining': 1479, 'nightlife': 128, 'shopping': 795, 'traditional_cultural': 925, 'family_friendliness': 664, 'nature_parks': 652, 'walkability_transit': 6229, 'safety': 98, 'accommodation_value': 4578, 'crowd_level': 1286}
centroid-to-centroid cosine (lower = more distinct):


,food_din,nightlif,shopping,traditio,family_f,nature_p,walkabil,safety,accommod,crowd_le
food_dining,1.00,0.42,0.58,0.02,-0.18,0.04,-0.08,-0.24,-0.33,-0.03
nightlife,0.42,1.00,0.37,0.34,-0.11,0.24,-0.05,-0.04,-0.36,0.16
shopping,0.58,0.37,1.00,0.23,-0.28,0.18,0.13,-0.24,-0.43,0.03
traditional_cultural,0.02,0.34,0.23,1.00,-0.22,0.51,0.24,-0.26,-0.45,0.03
family_friendliness,-0.18,-0.11,-0.28,-0.22,1.00,-0.19,-0.51,0.20,0.26,-0.06
nature_parks,0.04,0.24,0.18,0.51,-0.19,1.00,0.25,-0.20,-0.43,0.19
walkability_transit,-0.08,-0.05,0.13,0.24,-0.51,0.25,1.00,-0.36,-0.53,-0.13
safety,-0.24,-0.04,-0.24,-0.26,0.20,-0.20,-0.36,1.00,0.32,0.42
accommodation_value,-0.33,-0.36,-0.43,-0.45,0.26,-0.43,-0.53,0.32,1.00,0.11
crowd_level,-0.03,0.16,0.03,0.03,-0.06,0.19,-0.13,0.42,0.11,1.00


## 2. Assign aspects

In [62]:
A = assign(units, kw, S, topic_map=TOPIC_MAP_USED)
A = A.merge(units[['unit_id','text_id','source','ward','unit_text']], on='unit_id').assign(sample='core')
print(f'core assignments: {len(A):,} | units with >= 1 aspect: {A.unit_id.nunique():,} of {len(units):,} ({A.unit_id.nunique()/len(units):.1%})')
display(pd.crosstab(A.aspect, A.path, margins=True).reindex(ASPECTS + ['All']))

# Extension: keyword path on English review sentences outside the Stage 3 sample
corpus = pd.read_parquet(DATA/'unified_corpus.parquet', columns=['text_id','source','ward','language','text'])
r = corpus[(corpus.source == 'airbnb_review') & (corpus.language == 'EN')]
X = pd.DataFrame([(tid, w, i, u) for tid, w, t in zip(r.text_id, r.ward, r.text) for i, u in enumerate(split_units(t))],
                 columns=['text_id','ward','unit_no','unit_text'])
X['unit_id'] = X.text_id + '_u' + X.unit_no.astype(str)
X = X[~X.unit_id.isin(set(units.unit_id))]
from absa import KW_RE, EXCLUDE      # vectorized keyword screen (same rule as keyword_hits, much less memory)
_has = np.zeros(len(X), bool)
_obj = X.unit_text.astype(object)       # object dtype -> Python re (pyarrow's regex treats \b as ASCII-only, e.g. misses 'café')
for _a, _rx in KW_RE.items():
    _txt = _obj.str.replace(EXCLUDE[_a], ' ', regex=True) if _a in EXCLUDE else _obj
    _has |= _txt.str.contains(_rx).values
X['has_kw'] = _has
n_cand = int(X.has_kw.sum())
frozen = COUT/'absa_extension_units_v2.parquet'
if frozen.exists():     # reuse the scored v2 sample so later keyword changes don't pull in unscored sentences
    keep = X[X.unit_id.isin(set(pd.read_parquet(frozen).unit_id)) & X.has_kw]
    print(f'using frozen v2 extension sample: {len(keep):,} units still have a keyword after pruning')
else:
    keep = X[X.has_kw].sample(frac=1, random_state=SEED)
    keep = keep[keep.groupby('ward').cumcount() < EXT_CAP_PER_WARD]
    keep[['unit_id']].to_parquet(frozen, index=False)    # freeze it so later reruns score the same sentences
kwmap = {uid: keyword_hits(t) for uid, t in zip(keep.unit_id, keep.unit_text)}
ext = pd.DataFrame([(uid, a, 'keyword', np.nan, ', '.join(dict.fromkeys(k.lower() for k in kwmap[uid][a])))
                    for uid in keep.unit_id for a in ASPECTS if kwmap[uid][a]],
                   columns=['unit_id','aspect','path','sim','keywords'])
ext = ext.merge(keep[['unit_id','text_id','ward','unit_text']], on='unit_id').assign(source='airbnb_review', sample='extension')
print(f'extension: {len(X):,} candidate sentences, {n_cand:,} with a keyword, kept {len(keep):,} (cap {EXT_CAP_PER_WARD:,}/ward) -> {len(ext):,} assignments')
A = pd.concat([A, ext[A.columns]], ignore_index=True)
assert not A.duplicated(['unit_id','aspect']).any()
print('total assignments:', f'{len(A):,}')
del X, kwmap, ext, r, corpus, _has, _obj; import gc; _ = gc.collect()

core assignments: 33,702 | units with >= 1 aspect: 24,611 of 42,147 (58.4%)


path,embedding,keyword,topic+embedding,All
aspect,,,,
food_dining,0,4463,0,4463
nightlife,0,675,0,675
shopping,0,3216,0,3216
traditional_cultural,0,2346,207,2553
family_friendliness,0,1091,0,1091
nature_parks,0,1681,63,1744
walkability_transit,53,9543,197,9793
safety,0,334,0,334
accommodation_value,27,5789,1020,6836


using frozen v2 extension sample: 256,823 units still have a keyword after pruning


extension: 1,304,757 candidate sentences, 587,212 with a keyword, kept 256,823 (cap 30,000/ward) -> 326,369 assignments
total assignments: 360,071


In [63]:
# Clause selection for contrastive sentences
A['clauses'] = A.unit_text.map(split_clauses)
A['clause_idx'] = [pick_clause(c, a) if c else None for c, a in zip(A.clauses, A.aspect)]
A['score_id'] = np.where(A.clause_idx.notna(), 'c:' + A.unit_id + ':' + A.clause_idx.fillna(-1).astype(int).astype(str), 'u:' + A.unit_id)
A['supporting_span'] = [c[int(k)] if pd.notna(k) else t for c, k, t in zip(A.clauses, A.clause_idx, A.unit_text)]
print(f"aspect rows scored on a clause: {(A.clause_idx.notna()).sum():,} ({A.clause_idx.notna().mean():.1%})")
ex = A[A.clause_idx.notna()].sample(3, random_state=SEED)
for r in ex.itertuples(): print(f'  [{r.aspect}] {r.unit_text[:100]}\n      -> span: {r.supporting_span[:90]}')

aspect rows scored on a clause: 38,441 (10.7%)
  [accommodation_value] It is a spacious yet cozy-feeling building, has some highly comfortable and stylish furnishings and 
      -> span: It is a spacious
  [food_dining] Location wise, it is in the business district in Akasaka, near many restaurants and bars but minus t
      -> span: Location wise, it is in the business district in Akasaka, near many restaurants and bars
  [safety] The area is very safe and quiet, outside of the city hustle and bustle, although close to bus stops 
      -> span: The area is very safe and quiet, outside of the city hustle and bustle


## 3. Export the scoring input for Colab
All units plus every needed clause, so a single Colab run covers everything.

In [64]:
score_units = A[['unit_id','unit_text']].drop_duplicates('unit_id')
score_in = pd.concat([
    pd.DataFrame({'text_id': 'u:' + score_units.unit_id, 'text': score_units.unit_text}),
    A.loc[A.clause_idx.notna(), ['score_id','supporting_span']].drop_duplicates('score_id')
     .rename(columns={'score_id':'text_id','supporting_span':'text'})], ignore_index=True)
assert score_in.text_id.is_unique and set(A.score_id) <= set(score_in.text_id)
score_in['language'] = 'EN'
score_in.to_parquet(CIN/'absa_score_input.parquet', index=False)
print('scoring input rows:', len(score_in), '(units + clauses)')

scoring input rows: 315067 (units + clauses)


## Stage 4b (GPU): nlptown scoring of aspect sentences and clauses
Runs only if `absa_scores_nlptown.parquet` is missing from `data/colab_out/`.

In [65]:
if (COUT/'absa_scores_nlptown.parquet').exists():
    print('precomputed aspect scores found - skipping GPU scoring')
else:
    import torch
    need_gpu('Stage 4b')
    from colab_sentiment_core import score_corpus
    model, pp, ct = load_hf(NLPTOWN_MODEL)
    P5 = ['p1','p2','p3','p4','p5']
    o = score_corpus(pd.read_parquet(CIN/'absa_score_input.parquet'), pp, ct, P5)
    o['pred_star'] = o[P5].values.argmax(axis=1) + 1; o['exp_star'] = o[P5].values @ np.arange(1, 6)
    o.to_parquet(COUT/'absa_scores_nlptown.parquet', index=False)
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('saved absa_scores_nlptown.parquet', o.shape)

precomputed aspect scores found - skipping GPU scoring


## 4. Precision audit of aspect assignment
The sample was drawn from the **v1 assignment** (before the path restrictions) and is **frozen** in `aspect_audit_sample.csv`, so the judgments stay aligned. Up to 4 per aspect × path, seeded, judged by Claude: does the sentence really talk about this aspect? (LLM-judged, like the Stage 2 labels.) This measures **precision only**; recall isn't measured.

In [66]:
aud = pd.read_csv(LAB/'aspect_audit_sample.csv')          # frozen v1 sample
print('audit sample (frozen, v1):', len(aud))
f = LAB/'aspect_audit_llm.csv'
if f.exists():
    j = aud.merge(pd.read_csv(f), on='audit_id', validate='1:1')
    prec = j.pivot_table(index='aspect', columns='path', values='correct', aggfunc='mean').reindex(ASPECTS)
    prec['all_paths'] = j.groupby('aspect').correct.mean()
    print('precision (share judged correct); n per cell <= 4, so read as indicative:')
    display(prec.round(2))
    print('v1 overall precision by path:', j.groupby('path').correct.mean().round(3).to_dict(), '| overall:', round(j.correct.mean(), 3))
    kept = j[(j.path == 'keyword') | ((j.path == 'topic+embedding') & j.aspect.isin(TOPIC_OK)) | ((j.path == 'embedding') & j.aspect.isin(EMB_OK))]
    print(f'precision of the cells retained in v2: {kept.correct.mean():.3f} (n={len(kept)}; optimistic, because the same audit chose the cells)')
else:
    print('aspect_audit_llm.csv not found yet')

audit sample (frozen, v1): 108
precision (share judged correct); n per cell <= 4, so read as indicative:


path,embedding,keyword,topic+embedding,all_paths
aspect,,,,
food_dining,NaN,1.00,0.25,0.62
nightlife,0.50,0.75,0.25,0.50
shopping,0.25,0.50,0.00,0.25
traditional_cultural,0.00,1.00,0.75,0.58
family_friendliness,NaN,1.00,0.25,0.62
nature_parks,0.25,0.75,0.75,0.58
walkability_transit,1.00,0.75,1.00,0.92
safety,NaN,1.00,0.00,0.50
accommodation_value,1.00,1.00,1.00,1.00


v1 overall precision by path: {'embedding': 0.536, 'keyword': 0.85, 'topic+embedding': 0.475} | overall: 0.63
precision of the cells retained in v2: 0.868 (n=68; optimistic, because the same audit chose the cells)


## 5. Score and build the output table

In [67]:
f_scores = COUT/'absa_scores_nlptown.parquet'
HAVE_SCORES = f_scores.exists()
if HAVE_SCORES:
    sc = pd.read_parquet(f_scores)[['text_id','pred_star','exp_star']].rename(columns={'text_id':'score_id'})
    print('loaded real nlptown scores:', len(sc))
else:
    # STUB: synthetic scores so the output path can be tested. NOT results; nothing is saved from this branch.
    rng = np.random.default_rng(0)
    sc = pd.DataFrame({'score_id': score_in.text_id, 'exp_star': rng.uniform(1, 5, len(score_in))})
    sc['pred_star'] = sc.exp_star.round().clip(1, 5).astype(int)
    print('STUB scores (Colab output not uploaded yet)')
A['fallback'] = ~A.score_id.isin(set(sc.score_id))       # clause changed after pruning -> use the full-sentence score
A.loc[A.fallback, 'score_id'] = 'u:' + A.loc[A.fallback, 'unit_id']
A.loc[A.fallback, 'supporting_span'] = A.loc[A.fallback, 'unit_text']
A.loc[A.fallback, 'clause_idx'] = np.nan
print(f'rows falling back to the sentence score: {int(A.fallback.sum()):,}')
unscored = ~A.score_id.isin(set(sc.score_id))   # units that gained an aspect only after pruning were never sent to Colab
print(f'rows dropped because the sentence was never scored: {int(unscored.sum()):,}')
A = A[~unscored]
out = A.merge(sc, on='score_id', how='left', validate='m:1')
assert out.exp_star.notna().all(), 'missing scores'
out['sentiment_score'] = to_score(out.exp_star)
absa = out[['text_id','ward','source','aspect','sentiment_score','supporting_span',
            'unit_id','path','sim','keywords','exp_star','pred_star','sample']].copy()
absa['scored_on'] = np.where(out.clause_idx.notna(), 'clause', 'sentence')
assert absa.sentiment_score.between(-1, 1).all()
print(absa.shape); absa.head(3)

loaded real nlptown scores: 317670


rows falling back to the sentence score: 35


rows dropped because the sentence was never scored: 27


(360044, 14)


,text_id,ward,source,aspect,sentiment_score,supporting_span,unit_id,path,sim,keywords,exp_star,pred_star,sample,scored_on
0,ov_1000288508337168466,Setagaya,airbnb_overview,walkability_transit,0.469918,Sangenjaya Station is served by the Tokyu Denentoshi Line and Tokyu Setagaya Line.,ov_1000288508337168466_u0,keyword,0.465795,station,3.939837,5,core,sentence
1,ov_1000288508337168466,Setagaya,airbnb_overview,walkability_transit,-0.031543,Shibuya is approximately 5 minutes away by train.,ov_1000288508337168466_u1,keyword,0.602874,train,2.936914,3,core,sentence
2,ov_1000288508337168466,Setagaya,airbnb_overview,walkability_transit,0.594168,It is known as a convenient location with easy access to central Tokyo.,ov_1000288508337168466_u2,keyword,0.500234,access,4.188337,4,core,sentence


In [68]:
if HAVE_SCORES:
    absa.to_parquet(DATA/'absa_aspects.parquet', index=False)
    print('saved absa_aspects.parquet')
    rv = absa[absa.source == 'airbnb_review']; ov = absa[absa.source == 'airbnb_overview']
    tbl = pd.DataFrame({'review_mentions': rv.aspect.value_counts(), 'review_mean_sent': rv.groupby('aspect').sentiment_score.mean(),
                        'review_share_negative': rv.groupby('aspect').sentiment_score.apply(lambda s: (s < -0.25).mean()),
                        'overview_mentions': ov.aspect.value_counts(), 'overview_mean_sent': ov.groupby('aspect').sentiment_score.mean()}).reindex(ASPECTS)
    print('Aspect summary (negative = score < -0.25, i.e. expected star below 2.5):'); display(tbl.round(3))

saved absa_aspects.parquet
Aspect summary (negative = score < -0.25, i.e. expected star below 2.5):


,review_mentions,review_mean_sent,review_share_negative,overview_mentions,overview_mean_sent
aspect,,,,,
food_dining,37530,0.559,0.027,2772,0.490
nightlife,4733,0.518,0.048,494,0.540
shopping,14821,0.558,0.025,2531,0.525
traditional_cultural,6238,0.549,0.031,2291,0.508
family_friendliness,18727,0.570,0.045,272,0.555
nature_parks,4374,0.585,0.014,1534,0.505
walkability_transit,95645,0.495,0.046,5317,0.388
safety,4671,0.563,0.066,141,0.582
accommodation_value,117233,0.521,0.074,383,0.438


## 6. Coverage by ward
Review-mention counts per ward × aspect. Cells with **fewer than 30 review mentions** are flagged as thin; Stage 5 reports them but marks them low-confidence.

In [69]:
core_counts = A[(A.source == 'airbnb_review') & (A['sample'] == 'core')].pivot_table(index='ward', columns='aspect', values='unit_id', aggfunc='count', fill_value=0)[ASPECTS]
print('thin cells using the core sample only:', int((core_counts < 30).values.sum()), 'of', core_counts.size)
rv_counts = A[A.source == 'airbnb_review'].pivot_table(index='ward', columns='aspect', values='unit_id', aggfunc='count', fill_value=0)[ASPECTS]
thin = (rv_counts < 30)
print('thin cells (< 30 review mentions):', int(thin.values.sum()), 'of', thin.size)
print(rv_counts.where(~thin, rv_counts.astype(str) + '*').to_string())
summary = {'n_units': len(units), 'n_assignments': len(A), 'units_with_aspect': int(A.unit_id.nunique()),
           'by_path': A.path.value_counts().to_dict(), 'tau_topic': TAU_TOPIC, 'tau_emb': TAU_EMB,
           'topic_ok': sorted(TOPIC_OK), 'emb_ok': sorted(EMB_OK), 'ext_cap_per_ward': EXT_CAP_PER_WARD,
           'by_sample': A['sample'].value_counts().to_dict(), 'thin_cells': int(thin.values.sum()), 'scores_real': HAVE_SCORES}
(DATA/'stage4_summary.json').write_text(json.dumps(summary, indent=2, default=str)); print('saved stage4_summary.json')

thin cells using the core sample only: 39 of 110


thin cells (< 30 review mentions): 0 of 110
aspect    food_dining  nightlife  shopping  traditional_cultural  family_friendliness  nature_parks  walkability_transit  safety  accommodation_value  crowd_level
ward                                                                                                                                                              
Chiyoda          1173         89       350                   122                  557            94                 3356     127                 4517         1140
Chuo             2005        120      1538                   225                  731           178                 5612     289                 8304         1969
Itabashi         1369         86       570                   169                  451           134                 3191     167                 3767         1480
Koto             2493        174       896                   461                 1331           402                 6665     296             

saved stage4_summary.json


In [70]:
# Free memory from the previous stage (every stage reloads its inputs from disk)
import gc
for _n in ['units', 'E', 'kw', 'H', 'Ec', 'Cent', 'S', 'A', 'corpus', 'r', 'X', 'keep', 'kwmap', 'ext', 'score_units', 'score_in', 'aud', 'sc', 'out', 'absa', 'rv_counts', 'core_counts', 'info']:
    globals().pop(_n, None)
gc.collect()
print('freed memory before Stage 4c')

freed memory before Stage 4c


# Stage 4c: Sentence-level validation of nlptown and aspect tags
nlptown was validated on **whole reviews** in Stage 2. Stage 4 applies it to **single sentences and clauses**, so this notebook checks it at that level.

**Sample:** 150 unique review spans from `absa_aspects.parquet`, 15 per aspect, with 5 from each nlptown score band (neg < −0.25, mid, pos > 0.5), seed 42. Negatives are oversampled, so metrics are reported **unweighted and population-weighted** (weight = unique review spans in the aspect × band cell ÷ 5, where a span tagged with two aspects counts in both; bootstrap within cells). The 150 spans were drawn before the v3/v4 keyword prune; section 4 shows the effect of the prune.

**Labels:** Claude (LLM-assisted) gives each span a 1–5 star for the sentiment it expresses (3 = neutral or purely factual, e.g. "10 min walk to the station"), plus `aspect_ok` (0/1: does the span really talk about the tagged aspect?). Rakim audited 30 blind (embedded above); the threshold is quadratic-weighted kappa ≥ 0.6, as in Stage 2.

In [71]:
import numpy as np, pandas as pd
from metrics import weighted_boot_ci, kish_ess, eval_5, eval_3, collapse3, confusion_df
from sklearn.metrics import cohen_kappa_score
SEED = 42; DATA, LAB = ROOT/'data/interim', ROOT/'data/labels'
d = pd.read_csv(LAB/'sentence_val_candidates.csv').merge(pd.read_csv(LAB/'sentence_llm_labels.csv')[['item','llm_star','aspect_ok']], on='item', validate='1:1')
a = pd.read_parquet(DATA/'absa_aspects.parquet')
# FIXED (audit, 24 Nov): deduplicate by span AND aspect. Deduplicating by span alone counted a span with two aspects
# under only one of them (20% of spans), under-weighting nightlife and shopping by 55-60%.
r = a[a.source == 'airbnb_review'].drop_duplicates(['supporting_span', 'aspect']).copy()
r['band'] = np.select([r.sentiment_score < -0.25, r.sentiment_score > 0.5], ['neg','pos'], 'mid')
N = r.groupby(['aspect','band']).size().rename('N'); n = d.groupby(['aspect','band']).size().rename('n')
d = d.join(pd.concat([N, n], axis=1), on=['aspect','band']); d['w'] = d.N / d.n
S = (d.aspect + '|' + d.band).values
print(f'labeled spans: {len(d)} | population of unique review span x aspect pairs: {int(N.sum()):,} | Kish ESS: {kish_ess(d.w):.1f}')
print('population band mix:', r.band.value_counts(normalize=True).round(3).to_dict())

labeled spans: 150 | population of unique review span x aspect pairs: 337,014 | Kish ESS: 42.6
population band mix: {'pos': 0.669, 'mid': 0.278, 'neg': 0.053}


## 1. Sentiment agreement (nlptown vs labels)

In [72]:
rows = []
for kind, y, p in [('5', d.llm_star, d.pred_star), ('3', collapse3(d.llm_star), collapse3(d.pred_star))]:
    unw = (eval_5 if kind == '5' else eval_3)(y, p)
    for m, (pt, lo, hi) in weighted_boot_ci(y, p, d.w, S, kind, seed=SEED).items():
        rows.append({'task': f'{kind}-class', 'metric': m, 'unweighted': unw[m], 'weighted': pt, 'w_ci_low': lo, 'w_ci_high': hi})
display(pd.DataFrame(rows).round(3))
wm = lambda x, m=None: np.average(x if m is None else x[m], weights=d.w if m is None else d.w[m])
print(f"weighted mean star - nlptown expected {wm(d.exp_star):.2f} vs label {wm(d.llm_star):.2f} (bias {wm(d.exp_star - d.llm_star):+.2f})")
neg = (d.pred_star <= 2).values
print(f"precision of nlptown 'negative' (pred 1-2 that the label also rates 1-2): {wm((d.llm_star <= 2).values.astype(float), neg):.3f}")
print('bias by band (pred - label, unweighted):', (d.pred_star - d.llm_star).groupby(d.band).mean().round(2).to_dict())
confusion_df(d.llm_star, d.pred_star, [1,2,3,4,5])

,task,metric,unweighted,weighted,w_ci_low,w_ci_high
0,5-class,accuracy,0.493,0.612,0.500,0.734
1,5-class,weighted_f1,0.505,0.615,0.495,0.741
2,5-class,qwk,0.504,0.583,0.436,0.707
3,5-class,mae,0.800,0.481,0.341,0.608
4,3-class,accuracy,0.673,0.832,0.775,0.885
5,3-class,weighted_f1,0.678,0.833,0.766,0.894
6,3-class,qwk,0.544,0.629,0.475,0.731
7,3-class,macro_f1,0.547,0.481,0.410,0.575


weighted mean star - nlptown expected 4.07 vs label 4.19 (bias -0.11)
precision of nlptown 'negative' (pred 1-2 that the label also rates 1-2): 0.516
bias by band (pred - label, unweighted): {'mid': 0.22, 'neg': -1.06, 'pos': 0.24}


,pred_1,pred_2,pred_3,pred_4,pred_5
true_1,1,2,1,0,0
true_2,4,16,6,0,2
true_3,1,6,5,2,8
true_4,6,6,6,24,11
true_5,3,1,1,10,28


## 2. Aspect-tag precision at sentence level

In [73]:
print(f'aspect precision: unweighted {d.aspect_ok.mean():.3f} | weighted {wm(d.aspect_ok.astype(float)):.3f}')
display(d.groupby('aspect').aspect_ok.mean().round(2).to_frame('precision (n=15 each)'))
print('Wrong tags (examples):')
print(d.loc[d.aspect_ok == 0, ['aspect','supporting_span']].head(12).to_string(index=False, max_colwidth=100))

aspect precision: unweighted 0.740 | weighted 0.905


,precision (n=15 each)
aspect,
accommodation_value,1.00
crowd_level,0.80
family_friendliness,0.93
food_dining,0.93
nature_parks,0.60
nightlife,0.53
safety,0.67
shopping,0.60
traditional_cultural,0.47


Wrong tags (examples):
              aspect                                                                                      supporting_span
        nature_parks When we arrived, he greeted us warmly, helped direct traffic so we could park our car on a narrow...
           nightlife                       The refrigerator was like a “Mini-bar” and unfriendly for cooking by ourselves
        nature_parks                            Overall, I enjoyed the ambiance and convenience of staying at Sakura Inn!
            shopping Under the TV the electrical outlet usage felt dangerous (surge protector plugged into another pro...
         crowd_level A few even 24hrs which is great if you are not quiet on the right time zone and wake up hunger at...
        nature_parks                                                                    Only minutes from Yoyogi station.
traditional_cultural Transportation was also very easy with all the bus stops close by and not being far from Asakusa ...
t

## 3. Human audit (30 blind)

In [74]:
f = LAB/'sentence_audit_completed.csv'
if f.exists():
    h = pd.read_csv(f).merge(pd.read_csv(LAB/'sentence_audit_key.csv'), on=['audit_id','item'])
    h = h[pd.to_numeric(h.rakim_star, errors='coerce').between(1, 5)]
    h['rakim_star'] = h.rakim_star.astype(int)
    print(f'n = {len(h)} | star QWK = {cohen_kappa_score(h.rakim_star, h.llm_star, weights="quadratic"):.3f} | exact = {(h.rakim_star == h.llm_star).mean():.1%}')
    ok = h[pd.to_numeric(h.rakim_aspect_ok, errors='coerce').isin([0, 1])]
    print(f'aspect_ok agreement = {(ok.rakim_aspect_ok.astype(int) == ok.aspect_ok).mean():.1%} (n={len(ok)})')
else:
    print('Audit not completed yet: upload data/labels/sentence_audit_completed.xlsx')

n = 30 | star QWK = 0.816 | exact = 56.7%
aspect_ok agreement = 70.0% (n=30)


## 4. Effect of the v3 + v4 keyword prune on this sample
Which labeled spans lose their aspect tag under the v3 word-sense exclusions and the v4 strict place-name rule, and what precision remains on the rest? This is **optimistic**, because the exclusions were written after seeing these errors.

In [75]:
a3 = pd.read_parquet(DATA/'absa_aspects.parquet')[['unit_id','aspect']].drop_duplicates()
d['kept_v3'] = d.set_index(['unit_id','aspect']).index.isin(a3.set_index(['unit_id','aspect']).index)
print(f"spans dropped by v3+v4: {(~d.kept_v3).sum()} | of which Claude had marked wrong: {int(((~d.kept_v3) & (d.aspect_ok == 0)).sum())}")
k = d[d.kept_v3]
print(f'aspect precision on retained spans: unweighted {k.aspect_ok.mean():.3f} | weighted {np.average(k.aspect_ok, weights=k.w):.3f}')
display(k.groupby('aspect').aspect_ok.mean().round(2).to_frame('precision after prune (Claude definition)'))

spans dropped by v3+v4: 20 | of which Claude had marked wrong: 19
aspect precision on retained spans: unweighted 0.846 | weighted 0.922


,precision after prune (Claude definition)
aspect,
accommodation_value,1.00
crowd_level,0.80
family_friendliness,0.93
food_dining,0.93
nature_parks,0.82
nightlife,1.00
safety,0.71
shopping,0.64
traditional_cultural,0.78


In [76]:
# Free memory from the previous stage (every stage reloads its inputs from disk)
import gc
for _n in ['d', 'a', 'r', 'a3', 'k', 'N', 'n']:
    globals().pop(_n, None)
gc.collect()
print('freed memory before Stage 5')

freed memory before Stage 5


# Stage 5: Ward profiles
One JSON per ward (`data/ward_profiles/<ward>.json`), plus a combined file and a sanity-check table. **This is the final stage of the notebook pipeline.**

**Rules (agreed with Rakim)**
- **Aspect sentiment = review sentences only**, as a mean on the −1…+1 scale with a **95% cluster-bootstrap CI that resamples listings** (sentences from one listing aren't independent), 2,000 replicates, seed 42. Host overviews are promotional, so they count toward mention share only.
- **No "share negative":** sentence-level validation (Stage 4c) found only about 51% of nlptown's negative flags were truly negative.
- **Comparison to Tokyo:** each ward × aspect mean is compared with the 11-ward pooled mean. `vs_tokyo` is "above" / "below" only when the ward's CI excludes the pooled mean, otherwise "similar". Ward differences are small (0.06–0.18 points), so most cells are expected to be "similar".
- **Quotes:** review sentences only, keyword path, with the aspect keyword inside the quoted span, an expected star ≥ 4.5 (positive) or ≤ 1.5 (negative), and 6–30 words. Up to 3 positive and 2 negative per aspect, from different listings, seeded. These filters are proxies, not a check of correctness, and Stage 4c showed nlptown's negative flags are unreliable. So **every negative quote was read** (embedded `negative_quote_check.csv`): a negative quote is published only if it is really negative and really about the tagged aspect as a traveler would read it (safety = neighborhood or personal safety, crowd level = noise or crowds from outside the unit). Negative quotes not in that check are not published. Positive quotes were not read one by one; a spot check of about 30 found 1–2 off-aspect.
- **Structured fields** come from the listings table: listing count, median price (JPY per night; the source marks it with `$`), mean `review_scores_location`, review count, and language coverage.

In [77]:
import sys, json, re, importlib.metadata as md
from pathlib import Path
import numpy as np, pandas as pd
from absa import ASPECTS, keyword_hits
SEED, N_BOOT = 42, 2000
DATA, OUTD = ROOT/'data/interim', ROOT/'data/ward_profiles'; OUTD.mkdir(parents=True, exist_ok=True)
WARDS = ['Shinjuku','Shibuya','Taito','Chuo','Chiyoda','Minato','Sumida','Koto','Toshima','Setagaya','Itabashi']
absa = pd.read_parquet(DATA/'absa_aspects.parquet')
corpus = pd.read_parquet(DATA/'unified_corpus.parquet', columns=['text_id','listing_id','source','ward','language','nlptown_supported'])
listings = pd.read_parquet(DATA/'listings_pruned.parquet')
info = pd.read_parquet(DATA/'topic_info.parquet').set_index('topic')
t_share = pd.read_csv(DATA/'topic_by_ward_share.csv', index_col=0); t_lift = pd.read_csv(DATA/'topic_by_ward_lift.csv', index_col=0)
s1 = json.loads((DATA/'stage1_log.json').read_text()); s2 = json.loads((DATA/'stage2_summary.json').read_text())
s3 = json.loads((DATA/'stage3_summary.json').read_text()); s4 = json.loads((DATA/'stage4_summary.json').read_text())
LAB = ROOT/'data/labels'
absa = absa.merge(corpus[['text_id','listing_id']], on='text_id', how='left', validate='m:1')
assert absa.listing_id.notna().all()
rv = absa[absa.source == 'airbnb_review']
print('aspect rows:', len(absa), '| review rows:', len(rv), '| wards:', absa.ward.nunique())

aspect rows: 360044 | review rows: 343104 | wards: 11


## 1. Aspect sentiment with a listing-level cluster bootstrap

In [78]:
rng = np.random.default_rng(SEED)
def cluster_boot(df, n_boot=N_BOOT):
    g = df.groupby('listing_id').sentiment_score.agg(['sum','count'])
    s, c, k = g['sum'].values, g['count'].values, len(g)
    w = rng.multinomial(k, np.full(k, 1/k), size=n_boot)        # listing resampling weights
    reps = (w @ s) / (w @ c)
    return s.sum()/c.sum(), np.percentile(reps, 2.5), np.percentile(reps, 97.5), k

pooled = rv.groupby('aspect').sentiment_score.mean()
rows = []
for (w, a), g in rv.groupby(['ward','aspect']):
    m, lo, hi, k = cluster_boot(g)
    rows.append({'ward': w, 'aspect': a, 'mean': m, 'ci_low': lo, 'ci_high': hi, 'n_sentences': len(g), 'n_listings': k,
                 'tokyo_mean': pooled[a], 'vs_tokyo': 'above' if lo > pooled[a] else ('below' if hi < pooled[a] else 'similar')})
sent = pd.DataFrame(rows)
print('vs_tokyo counts:', sent.vs_tokyo.value_counts().to_dict())
sent.pivot(index='ward', columns='aspect', values='vs_tokyo').reindex(WARDS)[ASPECTS]

vs_tokyo counts: {'similar': 59, 'below': 27, 'above': 24}


aspect,food_dining,nightlife,shopping,traditional_cultural,family_friendliness,nature_parks,walkability_transit,safety,accommodation_value,crowd_level
ward,,,,,,,,,,
Shinjuku,similar,above,similar,below,below,similar,above,below,below,below
Shibuya,above,similar,above,similar,similar,similar,similar,similar,similar,similar
Taito,similar,similar,similar,similar,similar,above,similar,similar,above,above
Chuo,similar,similar,similar,similar,similar,similar,above,above,similar,below
Chiyoda,similar,similar,similar,below,similar,similar,above,similar,below,below
Minato,similar,similar,similar,similar,below,similar,above,above,similar,similar
Sumida,below,below,below,similar,similar,similar,below,similar,similar,above
Koto,below,below,below,similar,similar,similar,similar,similar,above,above
Toshima,below,below,below,below,similar,below,below,below,below,similar


## 2. Mentions and source shares

In [79]:
cnt = absa.groupby(['ward','aspect','source']).size().unstack(fill_value=0).rename(columns={'airbnb_review':'review_mentions','airbnb_overview':'overview_mentions'})
cnt['overview_share'] = cnt.overview_mentions / (cnt.review_mentions + cnt.overview_mentions)
cnt['share_of_ward_review_mentions'] = cnt.review_mentions / cnt.groupby(level='ward').review_mentions.transform('sum')
cnt['share_of_ward_overview_mentions'] = cnt.overview_mentions / cnt.groupby(level='ward').overview_mentions.transform('sum')
cnt = cnt.reset_index()
print('review-mention share by aspect (rows = wards):')
cnt.pivot(index='ward', columns='aspect', values='share_of_ward_review_mentions').reindex(WARDS)[ASPECTS].round(3)

review-mention share by aspect (rows = wards):


aspect,food_dining,nightlife,shopping,traditional_cultural,family_friendliness,nature_parks,walkability_transit,safety,accommodation_value,crowd_level
ward,,,,,,,,,,
Shinjuku,0.116,0.020,0.034,0.008,0.049,0.009,0.289,0.014,0.351,0.110
Shibuya,0.127,0.026,0.048,0.009,0.054,0.015,0.254,0.014,0.324,0.129
Taito,0.090,0.007,0.037,0.059,0.060,0.019,0.263,0.013,0.348,0.105
Chuo,0.096,0.006,0.073,0.011,0.035,0.008,0.268,0.014,0.396,0.094
Chiyoda,0.102,0.008,0.030,0.011,0.048,0.008,0.291,0.011,0.392,0.099
Minato,0.124,0.016,0.031,0.010,0.057,0.010,0.275,0.016,0.350,0.110
Sumida,0.079,0.005,0.039,0.022,0.061,0.013,0.303,0.014,0.348,0.115
Koto,0.107,0.007,0.039,0.020,0.057,0.017,0.287,0.013,0.339,0.114
Toshima,0.099,0.007,0.045,0.010,0.054,0.008,0.307,0.013,0.345,0.113


## 3. Quotes

In [80]:
q = rv[(rv.path == 'keyword') & ((rv.exp_star >= 4.5) | (rv.exp_star <= 1.5))].copy()
q['n_words'] = q.supporting_span.str.split().str.len()
q = q[q.n_words.between(6, 30)]
q = q[[bool(keyword_hits(s)[a]) for s, a in zip(q.supporting_span, q.aspect)]]   # keyword must sit in the quoted span
q['polarity'] = np.where(q.exp_star >= 4.5, 'positive', 'negative')
q['norm'] = q.supporting_span.str.lower().str.replace(r'[^a-z ]', '', regex=True).str.strip()
q = q.sample(frac=1, random_state=SEED).drop_duplicates(['ward','aspect','norm']).drop_duplicates(['ward','aspect','polarity','listing_id'])
quotes = (pd.concat([q[q.polarity=='positive'].groupby(['ward','aspect']).head(3), q[q.polarity=='negative'].groupby(['ward','aspect']).head(2)])
          .sort_values(['ward','aspect','polarity','exp_star'], ascending=[True, True, False, False]))
# Negative quotes: publish only those confirmed in the one-by-one check (see markdown above)
qchk = pd.read_csv(LAB/'negative_quote_check.csv')
ok_keys = set(zip(qchk.loc[qchk.verdict == 'keep', 'ward'], qchk.loc[qchk.verdict == 'keep', 'aspect'], qchk.loc[qchk.verdict == 'keep', 'text_id'], qchk.loc[qchk.verdict == 'keep', 'text']))
neg = quotes.polarity == 'negative'
in_check = pd.Series([k in set(zip(qchk.ward, qchk.aspect, qchk.text_id, qchk.text)) for k in zip(quotes.ward, quotes.aspect, quotes.text_id, quotes.supporting_span)], index=quotes.index)
approved = pd.Series([k in ok_keys for k in zip(quotes.ward, quotes.aspect, quotes.text_id, quotes.supporting_span)], index=quotes.index)
print(f'negative quotes selected: {int(neg.sum())} | in the check: {int((neg & in_check).sum())} | confirmed and kept: {int((neg & approved).sum())}')
print('verdicts in the check:', qchk.verdict.value_counts().to_dict())
quotes = quotes[~neg | approved]
qc = quotes.groupby(['ward','aspect']).size().unstack(fill_value=0).reindex(index=WARDS, columns=ASPECTS, fill_value=0)
print('eligible quote pool:', len(q), '| published:', len(quotes), f"({int((quotes.polarity == 'positive').sum())} positive, {int((quotes.polarity == 'negative').sum())} negative)",
      '| ward x aspect cells with < 3 quotes:', int((qc < 3).values.sum()))
qc

negative quotes selected: 120 | in the check: 120 | confirmed and kept: 36
verdicts in the check: {'not_negative': 44, 'wrong_aspect_or_not_about_area': 40, 'keep': 36}
eligible quote pool: 33337 | published: 366 (330 positive, 36 negative) | ward x aspect cells with < 3 quotes: 0


aspect,food_dining,nightlife,shopping,traditional_cultural,family_friendliness,nature_parks,walkability_transit,safety,accommodation_value,crowd_level
ward,,,,,,,,,,
Shinjuku,4,3,3,3,3,3,4,3,4,3
Shibuya,3,4,3,3,4,3,3,3,5,5
Taito,4,3,3,3,3,3,3,3,4,4
Chuo,4,3,3,3,3,3,4,3,4,4
Chiyoda,3,3,3,3,3,3,3,3,5,4
Minato,3,3,3,3,4,3,3,3,4,4
Sumida,3,3,3,3,3,3,3,3,5,3
Koto,4,3,3,3,3,3,3,3,4,4
Toshima,3,3,3,3,3,3,3,3,5,4


## 4. Structured fields and top topics

In [81]:
rev_c = corpus[corpus.source == 'airbnb_review']
struct = pd.DataFrame({
    'listing_count': listings.groupby('ward').size(),
    'listings_with_price': listings.groupby('ward').price.count(),
    'median_price': listings.groupby('ward').price.median(),
    'mean_review_scores_location': listings.groupby('ward').review_scores_location.mean(),
    'mean_review_scores_rating': listings.groupby('ward').review_scores_rating.mean(),
    'review_count': rev_c.groupby('ward').size(),
    'review_nlptown_coverage': rev_c.groupby('ward').nlptown_supported.mean(),
    'review_cjk_share': rev_c.groupby('ward').language.apply(lambda s: s.isin(['JA','ZH','KO']).mean()),
}).reindex(WARDS)
def top_topics(w, by, k=5):
    s = (t_lift.loc[w] if by == 'lift' else t_share.loc[w]).copy(); s.index = s.index.astype(int)
    sh = t_share.loc[w].copy(); sh.index = sh.index.astype(int)
    s = s[sh >= 0.01].sort_values(ascending=False).head(k)
    return [{'topic_id': int(t), 'label': info.loc[t, 'label'], 'top_words': info.loc[t, 'top_words'].split(', ')[:6],
             'share': round(float(sh[t]), 4), 'lift': round(float(t_lift.loc[w, str(t)]), 2)} for t in s.index]
struct.round(3)

,listing_count,listings_with_price,median_price,mean_review_scores_location,mean_review_scores_rating,review_count,review_nlptown_coverage,review_cjk_share
ward,,,,,,,,
Shinjuku,4991,4559,17994.0,4.757,4.708,181348,0.685,0.279
Shibuya,1902,1756,21972.0,4.777,4.737,75415,0.730,0.240
Taito,2950,2613,19967.0,4.705,4.738,126372,0.582,0.377
Chuo,382,304,25429.0,4.756,4.712,22740,0.629,0.336
Chiyoda,259,187,22374.0,4.820,4.757,13490,0.604,0.354
Minato,1190,1094,24293.0,4.810,4.749,39973,0.680,0.290
Sumida,3985,3610,15082.0,4.726,4.739,129589,0.555,0.409
Koto,548,525,14667.0,4.702,4.755,23391,0.588,0.376
Toshima,2749,2458,14852.5,4.693,4.722,77782,0.524,0.441


## 5. Write one JSON per ward

In [82]:
def r(x, n=4): return None if pd.isna(x) else round(float(x), n)
meta = {'source': 'Inside Airbnb Tokyo, snapshot 2025-09-29 (scraped 29-30 Sep 2025)', 'scope': '11 wards; Airbnb listings and reviews only',
        'sentiment_model': 'nlptown/bert-base-multilingual-uncased-sentiment (winner of Stage 2 bake-off)',
        'sentiment_scale': '(expected star - 3) / 2, range -1..+1; review sentences only',
        'ci_method': f'95% cluster bootstrap over listings, {N_BOOT} replicates, seed {SEED}',
        'languages': 'English sentences only for aspects and topics; Japanese, Chinese, Korean reviews not covered',
        'caveats': ['Ward differences in aspect sentiment are small; prefer vs_tokyo over raw ranks.',
                    'Sentence-level nlptown accuracy is lower than review-level (3-class weighted accuracy about 0.83); about half of its negative flags are wrong.',
                    'Negative quotes were each read and kept only if truly negative and on-aspect; positive quotes were not individually checked.',
                    'Aspect tags are keyword-led; validated precision about 0.92 (weighted), weakest for shopping and safety.',
                    'vs_tokyo flags come from 110 separate 95% intervals with no multiple-comparison adjustment; a few may be chance.',
                    'Host overviews are promotional and used for mention share only.',
                    'Median price is JPY per night (the source marks it with $).']}
profiles = {}
for w in WARDS:
    aspects = {}
    for a in ASPECTS:
        s_ = sent[(sent.ward == w) & (sent.aspect == a)].iloc[0]; c_ = cnt[(cnt.ward == w) & (cnt.aspect == a)].iloc[0]
        qq = quotes[(quotes.ward == w) & (quotes.aspect == a)]
        aspects[a] = {'mean_sentiment': r(s_['mean']), 'ci_95': [r(s_.ci_low), r(s_.ci_high)], 'tokyo_mean': r(s_.tokyo_mean),
                      'vs_tokyo': s_.vs_tokyo, 'review_mentions': int(c_.review_mentions), 'overview_mentions': int(c_.overview_mentions),
                      'listings_mentioning': int(s_.n_listings), 'share_of_ward_review_mentions': r(c_.share_of_ward_review_mentions),
                      'share_by_source': {'review': r(1 - c_.overview_share), 'overview': r(c_.overview_share)},
                      'quotes': [{'text': t, 'polarity': p, 'expected_star': r(e, 2), 'text_id': i}
                                 for t, p, e, i in zip(qq.supporting_span, qq.polarity, qq.exp_star, qq.text_id)]}
    st = struct.loc[w]
    profiles[w] = {'ward': w, 'meta': meta,
                   'structured': {'listing_count': int(st.listing_count), 'median_price': r(st.median_price, 0), 'listings_with_price': int(st.listings_with_price),
                                  'mean_review_scores_location': r(st.mean_review_scores_location, 3), 'mean_review_scores_rating': r(st.mean_review_scores_rating, 3),
                                  'review_count': int(st.review_count), 'review_nlptown_coverage': r(st.review_nlptown_coverage, 3),
                                  'review_cjk_share': r(st.review_cjk_share, 3)},
                   'top_topics_by_lift': top_topics(w, 'lift'), 'top_topics_by_share': top_topics(w, 'share'),
                   'aspects': aspects}
    (OUTD/f'{w.lower()}.json').write_text(json.dumps(profiles[w], indent=2, ensure_ascii=False))
(OUTD/'ward_profiles_all.json').write_text(json.dumps(profiles, indent=2, ensure_ascii=False))
print('written:', sorted(p.name for p in OUTD.glob('*.json')))

written: ['chiyoda.json', 'chuo.json', 'itabashi.json', 'koto.json', 'minato.json', 'setagaya.json', 'shibuya.json', 'shinjuku.json', 'sumida.json', 'taito.json', 'toshima.json', 'ward_profiles_all.json']


In [83]:
# Schema and integrity checks
for w, p in profiles.items():
    assert set(p['aspects']) == set(ASPECTS)
    for a, v in p['aspects'].items():
        assert -1 <= v['mean_sentiment'] <= 1 and v['ci_95'][0] <= v['mean_sentiment'] <= v['ci_95'][1], (w, a)
        assert len(v['quotes']) <= 5 and v['review_mentions'] > 0
        assert all(qq['polarity'] == 'positive' or qq['text_id'] in set(qchk.text_id) for qq in v['quotes'])
        assert all(6 <= len(qq['text'].split()) <= 30 for qq in v['quotes'])
    assert len(p['top_topics_by_lift']) > 0 and p['structured']['listing_count'] > 0
reloaded = json.loads((OUTD/'ward_profiles_all.json').read_text()); assert set(reloaded) == set(WARDS)
print('schema checks passed for', len(profiles), 'wards')
print(json.dumps({k: profiles['Taito'][k] for k in ['structured','top_topics_by_lift']}, indent=1, ensure_ascii=False)[:1500])

schema checks passed for 11 wards
{
 "structured": {
  "listing_count": 2950,
  "median_price": 19967.0,
  "listings_with_price": 2613,
  "mean_review_scores_location": 4.705,
  "mean_review_scores_rating": 4.738,
  "review_count": 126372,
  "review_nlptown_coverage": 0.582,
  "review_cjk_share": 0.377
 },
 "top_topics_by_lift": [
  {
   "topic_id": 19,
   "label": "temple / asakusa / gate",
   "top_words": [
    "temple",
    "asakusa",
    "gate",
    "sensoji",
    "kaminarimon",
    "temples"
   ],
   "share": 0.096,
   "lift": 3.66
  },
  {
   "topic_id": 36,
   "label": "museum / sumo / km",
   "top_words": [
    "museum",
    "sumo",
    "km",
    "art",
    "national",
    "zoo"
   ],
   "share": 0.0428,
   "lift": 2.34
  },
  {
   "topic_id": 22,
   "label": "tree / sky / sensoji",
   "top_words": [
    "tree",
    "sky",
    "sensoji",
    "temple",
    "skytree",
    "meiji"
   ],
   "share": 0.0507,
   "lift": 1.51
  },
  {
   "topic_id": 17,
   "label": "early / luggage / 

## 6. Sanity-check table
One row per ward: structured fields, aspects above or below Tokyo, the strongest topic, and quote coverage. Then **face-validity checks**: expectations from general knowledge of Tokyo, each marked pass/fail. They were written **after** intermediate outputs had been seen, so they are confirmatory, not blind. They test whether the pipeline recovers well-known ward characters; they don't prove it is correct.

In [84]:
lab = lambda w: profiles[w]['top_topics_by_lift'][0]['label']
san = struct[['listing_count','median_price','mean_review_scores_location','review_cjk_share']].copy()
san['aspects_above_tokyo'] = [', '.join(sent[(sent.ward==w)&(sent.vs_tokyo=='above')].aspect) or '-' for w in WARDS]
san['aspects_below_tokyo'] = [', '.join(sent[(sent.ward==w)&(sent.vs_tokyo=='below')].aspect) or '-' for w in WARDS]
san['top_topic_by_lift'] = [lab(w) for w in WARDS]
san['quotes_total'] = qc.sum(axis=1)
san['aspects_with_<3_quotes'] = (qc < 3).sum(axis=1)
san.to_csv(OUTD/'sanity_check.csv')
pd.set_option('display.max_colwidth', 80); san.round(3)

,listing_count,median_price,mean_review_scores_location,review_cjk_share,aspects_above_tokyo,aspects_below_tokyo,top_topic_by_lift,quotes_total,aspects_with_<3_quotes
ward,,,,,,,,,
Shinjuku,4991,17994.0,4.757,0.279,"nightlife, walkability_transit","accommodation_value, crowd_level, family_friendliness, safety, traditional_c...",shinokubo / okubo / korean,33,0
Shibuya,1902,21972.0,4.777,0.240,"food_dining, shopping",-,crossing / scramble / youth,36,0
Taito,2950,19967.0,4.705,0.377,"accommodation_value, crowd_level, nature_parks",-,temple / asakusa / gate,33,0
Chuo,382,25429.0,4.756,0.336,"safety, walkability_transit",crowd_level,foot / store / seveneleven,34,0
Chiyoda,259,22374.0,4.820,0.354,walkability_transit,"accommodation_value, crowd_level, traditional_cultural",anime / ikebukuro / manga,33,0
Minato,1190,24293.0,4.810,0.290,"safety, walkability_transit",family_friendliness,observation / deck / view,33,0
Sumida,3985,15082.0,4.726,0.409,crowd_level,"food_dining, nightlife, shopping, walkability_transit",observation / deck / view,32,0
Koto,548,14667.0,4.702,0.376,"accommodation_value, crowd_level","food_dining, nightlife, shopping",museum / sumo / km,33,0
Toshima,2749,14852.5,4.693,0.441,-,"accommodation_value, food_dining, nature_parks, nightlife, safety, shopping,...",anime / ikebukuro / manga,33,0


In [85]:
share = cnt.pivot(index='ward', columns='aspect', values='share_of_ward_review_mentions')
mean_ = sent.pivot(index='ward', columns='aspect', values='mean')
words = lambda w: ' '.join(t['label'] + ' ' + ' '.join(t['top_words']) for t in profiles[w]['top_topics_by_lift'][:3]).lower()
checks = [
    ('Taito has the highest traditional/cultural mention share', share.traditional_cultural.idxmax() == 'Taito'),
    ('Shinjuku, Shibuya or Minato in the top 3 for nightlife mention share', bool(set(share.nightlife.nlargest(3).index) & {'Shinjuku','Shibuya','Minato'})),
    ('Taito or Sumida top-3 lift topics mention temple / sensoji / skytree', any(k in words('Taito') + words('Sumida') for k in ['temple','sensoji','skytree','sky'])),
    ('Shibuya top-3 lift topics mention crossing / scramble', any(k in words('Shibuya') for k in ['crossing','scramble'])),
    ('Toshima top-3 lift topics mention ikebukuro / anime', any(k in words('Toshima') for k in ['ikebukuro','anime'])),
    ('Itabashi is in the bottom 3 for walkability sentiment (most outlying ward)', 'Itabashi' in mean_.walkability_transit.nsmallest(3).index),
    ('Chuo, Chiyoda or Minato has the highest median price', struct.median_price.idxmax() in {'Chuo','Chiyoda','Minato'}),
    ('Itabashi has the lowest median price', struct.median_price.idxmin() == 'Itabashi'),
]
from scipy.stats import spearmanr
rho, p = spearmanr(struct.mean_review_scores_location, mean_.walkability_transit.reindex(WARDS))
chk = pd.DataFrame(checks, columns=['expectation','pass'])
print(chk.to_string(index=False))
print(f'\nSpearman across 11 wards, mean review_scores_location vs walkability sentiment: rho = {rho:.2f} (p = {p:.3f}; n = 11, low power)')
summary = {'n_wards': len(profiles), 'vs_tokyo_counts': sent.vs_tokyo.value_counts().to_dict(), 'quotes_selected': int(len(quotes)),
           'face_validity': chk.assign(**{'pass': chk['pass'].astype(bool)}).to_dict('records'), 'rho_location_vs_walkability': round(float(rho), 3)}
(DATA/'stage5_summary.json').write_text(json.dumps(summary, indent=2, default=str)); print('saved stage5_summary.json')

                                                               expectation  pass
                  Taito has the highest traditional/cultural mention share  True
      Shinjuku, Shibuya or Minato in the top 3 for nightlife mention share  True
      Taito or Sumida top-3 lift topics mention temple / sensoji / skytree  True
                     Shibuya top-3 lift topics mention crossing / scramble  True
                       Toshima top-3 lift topics mention ikebukuro / anime  True
Itabashi is in the bottom 3 for walkability sentiment (most outlying ward)  True
                      Chuo, Chiyoda or Minato has the highest median price  True
                                      Itabashi has the lowest median price  True

Spearman across 11 wards, mean review_scores_location vs walkability sentiment: rho = 0.87 (p = 0.000; n = 11, low power)
saved stage5_summary.json


## Limitations (for the write-up)
- **Language coverage:** aspects and topics use English sentences only. nlptown covers 6 European languages; about 34% of reviews (Japanese, Chinese, Korean) are not scored, unevenly by ward (24% Shibuya to 44% Toshima).
- **Labels:** per-review truth is LLM-assisted with human audits (Stage 2: QWK 0.979 on 38 English items; Stage 4c: star QWK 0.816 on 30). The 60 non-English labels were not human-audited.
- **Evaluation design:** the English label set oversampled VADER-negative reviews; weighted estimates are reported alongside.
- **Topic-selection rule:** amended once, on the fallback embedding and before the mpnet results were seen. The grid changed three times; the last change (`min_samples` 10 → 3) was made after seeing mpnet results, and the selected model's 29.8% outlier rate sits just under the 30% cap (Stage 3).
- **Aspects:** keyword-led; non-keyword paths were narrowed after a precision audit; strict place-name rule (Stage 4). Retained-cell precision figures are optimistic, because the same audits chose the rules.
- **Sentence-level sentiment:** weaker than review-level (weighted 3-class accuracy about 0.83, QWK about 0.6); about half of negative flags are wrong, so "share negative" is not reported, and negative quotes were checked one by one (36 of 120 kept).
- **Ward vs Tokyo flags:** 110 separate 95% intervals with no multiple-comparison adjustment (about 5 could be flagged by chance), and each ward is compared with a pooled Tokyo mean that includes that ward.
- **Host overviews** are promotional, with heavy template reuse (68% deduplicated); they are used for mention share only.
- **Single snapshot;** minpaku rules shape listing supply, so listing counts are not tourism demand. Price is JPY per night (marked `$` in the source).
- **Face-validity checks** were written after seeing intermediate outputs, so they are confirmatory, not blind.